# Road-traffic mortality in Spain · DGT crash microdata, 2022–2025
## Exposure-adjusted risk, geography and hierarchical Bayesian modelling

This notebook analyses the annual `TABLA_ACCIDENTES_YY` files (crashes with casualties) published by the Spanish traffic authority (DGT) and takes them from descriptive statistics to a **publication-grade analysis**: year-on-year comparisons that separate signal from noise, a move from *case fatality* to *risk* through explicit exposure denominators, geocoding by road and kilometre point, and spatially smoothed hierarchical Bayesian models.

| Block | Content | Question it answers |
|---|---|---|
| 1 | Loading, cross-year schema validation, reconciliation with official totals, dictionary decoding | Are the annual files comparable and internally consistent? |
| 2 | Overview: KPIs, seasonality, hourly profile, 24-hour vs 30-day deaths | What happens, and when? |
| 3 | **Comparing years**: exact rate ratios, calendar standardisation, Kitagawa composition/rate decomposition, quasi-Poisson trend | Has anything really changed, or is it noise and mix? |
| 4 | Territory: empirical-Bayes shrinkage, funnel plots, between-year stability | Are provincial differences real or sampling variation? |
| 5 | Roads and segments: location quality, hot-spot persistence, regression to the mean | Does the past predict where fatal crashes will occur? |
| **6** | **Exposure** (INE population): deaths per million inhabitant-years; frequency × severity decomposition | Case fatality or risk? Why is a province dangerous? |
| **7** | **Geocoding** road + km point → coordinates; Moran's I, Getis-Ord Gi\*, spatio-temporal persistence of hot spots | Where, and does it recur? |
| **8** | **Hierarchical Bayesian models with BYM2 smoothing** (severity, frequency, risk) in PyMC; posterior-predictive checks, LOO, unseen-province validation | How much of the risk is spatially structured, and which provinces differ credibly from the mean? |
| **9** | **Robustness**: alternative priors and outcome definition | Do conclusions survive reasonable analytic choices? |
| 10 | Predictive model (gradient boosting) with forward-in-time and spatial validation | What separates a fatal crash from a non-fatal one? |
| 11 | Typologies of fatal crashes and their drift over time | Is the *kind* of fatal crash changing? |
| **12** | **Reviewer-proofing analyses**: EB score for segments, baselines for cells, spatial test for F and R, point-in-province check, alternative denominators, hypotheses that the data can test | Do the claims survive the checks a referee will ask for? |
| 13–14 | Export with reproducibility manifest · manuscript plan | |

**Assumptions and limits (read before interpreting any number):**
- All annual files are assumed to share the **2024 structure**. The validation cell warns if they do not.
- The microdata contain **no coordinates**. Geography is province, municipality (≥ 5,000 inhabitants only), island and — on interurban roads — road identifier plus kilometre point. Section 7 *estimates* coordinates, and only if a kilometre-point layer is supplied; urban crashes cannot be placed below municipality level.
- There is no age, sex, alcohol, speed or exact date (only `MES`). The files cover *crashes with casualties*: without exposure, any "rate" is **case fatality** (fatal ÷ casualty crashes) and depends on how many minor crashes are recorded.
- The only exposure used is **INE resident population**, which ignores tourism and through traffic. Distance-based exposure (vehicle-kilometres) is not used because no province-level source is available (Section 6).
- Always use **30-day deaths** (`*_30DF`). The 24-hour → 30-day uplift differs sharply by area and road-user type (Section 2).
- Every province-level estimate is **ecological**: it must not be attributed to individuals.

**Run time:** about 12 minutes with four years on a two-core laptop (≈ 1 min reading Excel the first time, ≈ 4 min for Sections 1–7, ≈ 4–6 min for the Bayesian models with `nutpie`, ≈ 2.5 min for Section 10). Without `nutpie` PyMC's own sampler is 5–10× slower: set `RUN_KFOLD = False`. The `RUN_*` switches in the configuration cell turn the heavy blocks on and off.

**Setup:** the first code cell installs any missing package (including `pymc`, `arviz` and `nutpie`) and Section 7 downloads the kilometre-point layer by itself the first time, so no manual preparation is needed beyond placing the DGT files, the dictionary and the INE population table in `DATA_DIR`.

In [ ]:
# ---- Environment: install any missing package (set INSTALL_MISSING = False to skip) -------------------
# pymc / arviz / nutpie are needed for the Bayesian sections (8-9); pyproj for geocoding (7). Nothing is reinstalled if already present.
INSTALL_MISSING = True
import importlib.util, subprocess, sys
REQUIRED = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "sklearn": "scikit-learn>=1.4", "statsmodels": "statsmodels",
            "openpyxl": "openpyxl", "pyproj": "pyproj", "pymc": "pymc", "arviz": "arviz", "nutpie": "nutpie", "h5netcdf": "h5netcdf", "pyarrow": "pyarrow"}
_missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing and INSTALL_MISSING:
    print("Installing:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
    importlib.invalidate_caches()
elif _missing:
    print("Missing packages (INSTALL_MISSING = False):", ", ".join(_missing))
else:
    print("All required packages are already installed.")

# ---- Health check: PyTensor (PyMC's backend) imports setuptools -> packaging. A half-upgraded `packaging` (common in Anaconda after
# installing pymc) breaks it with "cannot import name '_TrimmedRelease'". Detect it in a throw-away process and repair it.
if INSTALL_MISSING and importlib.util.find_spec("pymc") is not None:
    _probe = subprocess.run([sys.executable, "-c", "import packaging.utils, setuptools, setuptools.errors"], capture_output=True, text=True)
    if _probe.returncode != 0:
        print("Broken `packaging`/`setuptools` detected -> repairing:\n  ", _probe.stderr.strip().splitlines()[-1])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "packaging>=24.2", "setuptools"])
        _probe = subprocess.run([sys.executable, "-c", "import packaging.utils, setuptools, setuptools.errors"], capture_output=True, text=True)
        if _probe.returncode == 0:
            raise SystemExit("Repaired `packaging`/`setuptools`. RESTART THE KERNEL (Kernel > Restart) and run the notebook again from the top.")
        print("Could not repair automatically. Run in a terminal: pip install --force-reinstall packaging setuptools")

In [ ]:
import os, re, calendar, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80, "display.width", 220, "display.max_colwidth", 60,
              "display.float_format", "{:,.3f}".format)

# =============================== CONFIGURATION (edit here) ===============================
# --- Input / output ---
DATA_DIR  = Path(os.environ.get("DGT_DATA_DIR", "../data/raw"))      # folder with TABLA_ACCIDENTES_22..25 and the dictionary workbook
OUT_DIR   = Path(os.environ.get("DGT_OUT_DIR", "../outputs"))  # figures, tables, caches and the reproducibility manifest are written here
YEARS     = [2022, 2023, 2024, 2025]
SEGMENT_KM = 2        # length (km) of the road segment used to aggregate interurban crashes; Section 5 tests other lengths (MAUP)
SEED      = 42        # one seed drives every stochastic step (permutation tests, MCMC, resampling)
# --- Optional geographic layers (leave as None if unavailable) ---
PROV_GEO      = None  # path/URL to province polygons (geojson/shp): enables choropleth maps and the point-in-polygon check of the geocoder
PROV_CODE_COL = None  # column of that file holding the INE province code (1..52)
# --- Exposure (Section 6) ---
POPULATION_XLSX       = DATA_DIR / "67988.xlsx"   # INE population by province: table 67988 (annual census 2022-2025) or 2852 (official figures). Several years as columns are fine. None = no per-capita rates
POPULATION_FIXED_YEAR = None   # e.g. 2021: use that single year's population for *every* analysis year (only if INE gives you one year). None = use each year's figures
POPULATION_CSV        = None   # (legacy) long CSV with COD_PROVINCIA, ANYO, POBLACION; prefer POPULATION_XLSX
# --- Geocoding and spatial analysis (Section 7) ---
PK_CSV        = "auto"  # kilometre-point layer: "auto" = use DATA_DIR/puntosKilometricosSpain.csv, downloading it from PK_URL if absent; a path = your own layer; None = skip geocoding
PK_URL        = "https://raw.githubusercontent.com/jabiologo/roads/main/puntosKilometricosSpain.csv"   # public layer compiled in 2022 (DGT source field); check its licence and replace it with the current IGN/CNIG layer for publication
PK_COLS       = dict(x="X", y="Y", road="Nombre", km="numero")   # column names of that layer (lon/lat, WGS84, by default)
PK_CRS        = "EPSG:4326"                 # CRS of X, Y if they are not lon/lat
PK_MAX_DIST_PROV_KM = 175                   # candidates farther than this from the robust centre of their province/island are discarded (homonymous roads in other regions)
GRID_KM       = 10                          # side of the analysis cell
RMIN_KM       = 5                           # minimum km of road network for a cell to enter the spatial analysis
HOSPITALS_CSV = None                        # optional: CSV with lon, lat columns of hospitals
# --- Computation switches and MCMC settings (Sections 8-10) ---
RUN_ML        = True     # Section 10 (gradient boosting): ~2.5 min
RUN_BAYES     = True     # Sections 8-9 (PyMC)
RUN_SENS      = True     # robustness refits (4 extra fits)
RUN_KFOLD     = True     # unseen-province cross-validation (K x 2 fits; ~3 min with nutpie, hours without it -> switches itself off)
KFOLD_K       = 6
BAYES_DRAWS, BAYES_TUNE, BAYES_CHAINS, BAYES_TARGET_ACCEPT = 2000, 1500, 4, 0.95
BAYES_SAMPLER = "auto"   # "auto" uses nutpie when installed (pip install nutpie; 5-10x faster) and PyMC's NUTS otherwise
# --- Official DGT totals (crashes with casualties, 30-day deaths) used to verify that the files reconcile ---
# 2022-2024 come from the DGT publication "Las principales cifras de la siniestralidad vial". Fill in 2025 when it is published.
OFFICIAL = {2022: (97916, 1746), 2023: (101306, 1806), 2024: (101996, 1785), 2025: None}
# ===========================================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT_DIR / "cache"; CACHE_DIR.mkdir(exist_ok=True)
FIG_DIR   = OUT_DIR / "figs";  FIG_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
TABLES = {}      # every DataFrame stored here is exported to Excel in Section 12
MONTHS   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
WEEKDAYS = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

In [ ]:
# ---- Graphic style -------------------------------------------------------------------------------
# One validated categorical palette (colour-vision-deficiency safe), one sequential ramp and one diverging
# convention (red = above reference, blue = below, grey = statistically indistinguishable) are used throughout,
# so that colour always carries the same meaning from figure to figure.
SURFACE, INK, INK2, GRID, NEUTRAL = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df", "#a8a7a1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ    = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
UP, DOWN = "#e34948", "#2a78d6"
SEQ_CMAP = mpl.colors.LinearSegmentedColormap.from_list("seq", SEQ)
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110, "lines.linewidth": 2, "legend.frameon": False,
    "axes.prop_cycle": mpl.cycler(color=SERIES),
})
def show(fig, name):
    """Save a figure at 160 dpi under OUT_DIR/figs and display it."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
def wilson(x, n, z=1.96):
    """Wilson score interval for a binomial proportion. Preferred over the Wald interval because it keeps
    nominal coverage for the small counts and extreme proportions (case fatality ≈ 1-3 %) that dominate here."""
    x, n = np.asarray(x, float), np.asarray(n, float)
    n = np.where(n == 0, np.nan, n); p = x / n; den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return c - h, c + h

## 1 · Data loading and validation

**Purpose.** Before any estimate is produced we establish that the four annual files are (i) structurally comparable, (ii) internally consistent, and (iii) consistent with the totals the DGT publishes. Failing any of these would make year-on-year differences uninterpretable.

- Each Excel workbook is read once and cached (`outputs/cache`, keyed by file size and modification time); the first read of each file is slow.
- The file name determines the year; the `ANYO` column must agree with it.
- `ID_ACCIDENTE` restarts every year, so the analysis key is `UID = year-ID`.
- Counters absent from one year's file (`TOT_*`, `TOTAL_*`) are filled with 0 and reported, because an absent victim-type column means "none recorded" rather than "missing".

In [ ]:
def find_file(year):
    """Locate TABLA_ACCIDENTES_<yy> (xlsx or csv) anywhere under DATA_DIR; the negative look-ahead avoids matching e.g. _224 for 2022."""
    yy = f"{year % 100:02d}"
    pat = re.compile(rf"TABLA_ACCIDENTES_{yy}(?!\d)", re.I)
    c = [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in (".xlsx", ".csv") and pat.search(p.name)]
    return sorted(c)[0] if c else None

def read_year(year):
    p = find_file(year)
    if p is None:
        print(f"⚠️  {year}: TABLA_ACCIDENTES_{year % 100:02d} not found in {DATA_DIR.resolve()}")
        return None
    cache = CACHE_DIR / f"acc_{year}_{p.stat().st_size}_{int(p.stat().st_mtime)}.pkl"
    if cache.exists():
        return pd.read_pickle(cache)
    if p.suffix.lower() == ".csv":
        df = pd.read_csv(p, sep=None, engine="python")
    else:
        xl = pd.ExcelFile(p)
        sheet = next((s for s in xl.sheet_names if re.match(r"ACCIDENTES", s, re.I)), xl.sheet_names[0])
        df = xl.parse(sheet)
    df.columns = [str(c).strip().upper() for c in df.columns]
    df.to_pickle(cache)
    print(f"read {p.name}: {df.shape}")
    return df

raw = {y: read_year(y) for y in YEARS}
raw = {y: d for y, d in raw.items() if d is not None}
assert raw, f"No year could be loaded. Check DATA_DIR = {DATA_DIR.resolve()}"
REF_YEAR = 2024 if 2024 in raw else max(raw)      # 2024 is the reference schema (the structure the dictionary describes)
ref_cols = list(raw[REF_YEAR].columns)

# Schema audit: compare each year's columns with the reference and check year coding, ID uniqueness and month coverage.
rows = []
for y, d in raw.items():
    rows.append(dict(year=y, rows=len(d), columns=d.shape[1],
                     missing_vs_ref=sorted(set(ref_cols) - set(d.columns)),
                     extra_vs_ref=sorted(set(d.columns) - set(ref_cols)),
                     ANYO_in_file=sorted(pd.Series(d["ANYO"]).dropna().unique().tolist()) if "ANYO" in d else None,
                     unique_id=bool(d["ID_ACCIDENTE"].is_unique) if "ID_ACCIDENTE" in d else None,
                     months=int(d["MES"].nunique()) if "MES" in d else None))
schema = pd.DataFrame(rows).set_index("year")
display(schema)
bad_year = [y for y in raw if raw[y]["ANYO"].nunique() != 1 or int(raw[y]["ANYO"].iloc[0]) != y]
if bad_year: print("⚠️  The ANYO column does not match the file year in:", bad_year, "(the year in the file name is used)")
if (schema["months"] < 12).any(): print("⚠️  Some years cover fewer than 12 months: annual totals are NOT comparable:", schema.index[schema["months"] < 12].tolist())
if schema["missing_vs_ref"].map(len).sum() + schema["extra_vs_ref"].map(len).sum():
    print("⚠️  Schemas differ. Counters missing from a year are filled with 0 (TOT_*/TOTAL_*).")

In [ ]:
# ---- Harmonisation and stacking -------------------------------------------------------------------
# All files are reindexed to the union of columns; text columns are normalised (trim + upper case) and the rest coerced to numeric.
TEXT_COLS = ["CARRETERA", "CARRETERA_CRUCE"]
all_cols = list(dict.fromkeys([c for d in raw.values() for c in d.columns]))
frames = []
for y, d in raw.items():
    d = d.reindex(columns=all_cols).copy()
    for c in all_cols:
        d[c] = d[c].astype("string").str.strip().str.upper() if c in TEXT_COLS else pd.to_numeric(d[c], errors="coerce")
    d["ANYO"] = y
    frames.append(d)
acc = pd.concat(frames, ignore_index=True)
count_cols = [c for c in acc.columns if c.startswith(("TOT_", "TOTAL_"))]
absent = [c for c in count_cols if acc[c].isna().any()]
if absent: print("Counters absent from at least one year (filled with 0):", absent)
acc[count_cols] = acc[count_cols].fillna(0).astype(int)
acc["UID"] = acc["ANYO"].astype(str) + "-" + acc["ID_ACCIDENTE"].astype(str)   # ID_ACCIDENTE restarts every year
YL = [int(v) for v in sorted(acc.ANYO.unique())]; MULTI = len(YL) > 1
print(f"{len(acc):,} crashes · years {YL} · {acc.shape[1]} columns")

In [ ]:
# ---- Internal integrity: every record must satisfy the identities imposed by the register itself -----
# These are logical constraints (not statistical ones): a violation is a data-entry or export error, so each should be ~0.
user30 = [c for c in acc.columns if c.startswith("TOT_") and c.endswith("_MU30DF")]
bad = pd.DataFrame({
    "casualties30 != dead+serious+minor":     acc.TOTAL_VICTIMAS_30DF != acc.TOTAL_MU30DF + acc.TOTAL_HG30DF + acc.TOTAL_HL30DF,
    "casualties24h != dead+serious+minor":    acc.TOTAL_VICTIMAS_24H  != acc.TOTAL_MU24H + acc.TOTAL_HG24H + acc.TOTAL_HL24H,
    "sum of deaths by user type != total":    acc[user30].sum(axis=1) != acc.TOTAL_MU30DF,
    "30-day deaths < 24-hour deaths":         acc.TOTAL_MU30DF < acc.TOTAL_MU24H,
    "municipality inconsistent with province": (acc.COD_MUNICIPIO > 0) & (acc.COD_MUNICIPIO // 1000 != acc.COD_PROVINCIA),
    "crash without any casualty at 30 days":  acc.TOTAL_VICTIMAS_30DF == 0,
})
bad["ANYO"] = acc.ANYO
chk = bad.groupby("ANYO").sum()
display(chk)
TABLES["integrity"] = chk

# ---- Reconciliation with the official DGT totals ----------------------------------------------------
# Exact agreement of crash and death counts is the strongest evidence that the extract is complete and that the 30-day definition is the official one.
rec = acc.groupby("ANYO").agg(crashes=("UID", "size"), deaths_30d=("TOTAL_MU30DF", "sum"), deaths_24h=("TOTAL_MU24H", "sum"))
rec["official_crashes"] = [OFFICIAL.get(y)[0] if OFFICIAL.get(y) else np.nan for y in rec.index]
rec["official_deaths"] = [OFFICIAL.get(y)[1] if OFFICIAL.get(y) else np.nan for y in rec.index]
rec["diff_crashes"] = rec.crashes - rec.official_crashes
rec["diff_deaths"] = rec.deaths_30d - rec.official_deaths
display(rec)
TABLES["official_reconciliation"] = rec

In [ ]:
# ---- Dictionary decoding: Value -> Label for each categorical variable -------------------------------
# The DGT dictionary workbook stores one sheet per variable with a "Valor | Etiqueta" block. Category labels are
# reproduced *verbatim* (in Spanish) so that every table can be traced back to the official codebook.
def find_dict():
    c = sorted(p for p in DATA_DIR.rglob("*.xlsx") if re.search(r"diccionario", p.name, re.I))
    return c[0] if c else None

DEC = {}
DICT_PATH = find_dict()
if DICT_PATH is None:
    print("⚠️  Dictionary workbook not found in", DATA_DIR.resolve(), "· labels will remain numeric codes.")
else:
    for name, t in pd.read_excel(DICT_PATH, sheet_name=None, header=None).items():
        if t.shape[1] < 2: continue
        h = t.index[(t[0].astype(str).str.strip() == "Valor") & (t[1].astype(str).str.strip() == "Etiqueta")]
        if len(h) == 0: continue
        b = t.loc[h[0] + 1:, [0, 1]].copy(); b.columns = ["v", "l"]
        b["k"] = pd.to_numeric(b["v"], errors="coerce"); b = b.dropna(subset=["l"])
        if name == "ISLA": b.loc[b["k"].isna(), "k"] = 0          # 'No aplica' (not applicable)
        b = b.dropna(subset=["k"])
        DEC[name] = {int(k): str(l).strip() for k, l in zip(b["k"], b["l"])}
    # The 2024 dictionary omits three night-time lighting codes and the wind code; they are added by hand (labels kept in Spanish for consistency).
    DEC.setdefault("CONDICION_ILUMINACION", {}).update({4: "Noche, vía iluminada", 5: "Noche, alumbrado apagado", 6: "Noche, sin alumbrado"})
    DEC["CONDICION_VIENTO"] = {1: "Viento fuerte"}
    print(f"Dictionary: {DICT_PATH.name} · {len(DEC)} decodable variables")

def decode(col):
    d = DEC.get(col, {})
    return acc[col].map(lambda v: d.get(int(v), f"code {int(v)}") if pd.notna(v) else "No data")

for new, col in {"province": "COD_PROVINCIA", "weekday": "DIA_SEMANA", "zone": "ZONA",
                 "ownership": "TITULARIDAD_VIA", "road_type": "TIPO_VIA", "crash_type": "TIPO_ACCIDENTE",
                 "lighting": "CONDICION_ILUMINACION", "weather": "CONDICION_METEO", "surface": "CONDICION_FIRME",
                 "alignment": "TRAZADO_PLANTA", "visibility": "VISIB_RESTRINGIDA_POR"}.items():
    acc[new] = decode(col)
acc["area"] = acc["ZONA_AGRUPADA"].map({1: "Interurban", 2: "Urban"}).fillna("Unknown")   # 1 = interurban roads, 2 = urban streets

In [ ]:
# ---- Derived variables -------------------------------------------------------------------------------
# Autonomous community (INE province code -> region). Used for CCAA-level kilometre figures and for grouping.
CCAA = {**{p: "Andalucía" for p in (4, 11, 14, 18, 21, 23, 29, 41)}, **{p: "Aragón" for p in (22, 44, 50)}, 33: "Asturias",
        7: "Illes Balears", **{p: "Canarias" for p in (35, 38)}, 39: "Cantabria",
        **{p: "Castilla y León" for p in (5, 9, 24, 34, 37, 40, 42, 47, 49)},
        **{p: "Castilla-La Mancha" for p in (2, 13, 16, 19, 45)}, **{p: "Cataluña" for p in (8, 17, 25, 43)},
        **{p: "C. Valenciana" for p in (3, 12, 46)}, **{p: "Extremadura" for p in (6, 10)},
        **{p: "Galicia" for p in (15, 27, 32, 36)}, 28: "Madrid", 30: "Murcia", 31: "Navarra",
        **{p: "País Vasco" for p in (1, 20, 48)}, 26: "La Rioja", 51: "Ceuta", 52: "Melilla"}
# Road class groups from TIPO_VIA codes (1-14 in the dictionary).
ROAD_GROUP = {1: "Motorway/dual carriageway", 2: "Motorway/dual carriageway", 3: "Motorway/dual carriageway", 4: "Expressway",
              5: "Conventional road", 6: "Conventional road", 7: "Service road/ramp", 8: "Service road/ramp", 9: "Urban street",
              10: "Track/other", 11: "Track/other", 12: "Cycle path", 13: "Cycle path", 14: "Track/other"}
# Outcome definitions. A crash is "fatal" if at least one person died within 30 days (official definition) or within 24 hours (legacy definition).
acc["fatal30"]  = (acc.TOTAL_MU30DF > 0).astype(int)
acc["fatal24"]  = (acc.TOTAL_MU24H > 0).astype(int)
acc["late_deaths"] = acc.TOTAL_MU30DF - acc.TOTAL_MU24H          # deaths occurring between 24 hours and 30 days
acc["region"]     = acc.COD_PROVINCIA.map(CCAA).fillna("Other")
acc["road_group"] = acc.TIPO_VIA.map(ROAD_GROUP).fillna("Other")
acc["weekend"]    = acc.DIA_SEMANA.isin([6, 7])
acc["time_band"]  = pd.cut(acc.HORA, [-1, 5, 11, 17, 23], labels=["Night 0-5", "Morning 6-11", "Afternoon 12-17", "Evening 18-23"]).astype(str)
acc["month_start"] = pd.to_datetime(dict(year=acc.ANYO, month=acc.MES, day=1))
# "Vulnerable road users" = pedestrians, cyclists, moped riders, motorcyclists and e-scooter (VMP) users.
vul = [c for c in ("TOT_PEAT_MU30DF", "TOT_BICI_MU30DF", "TOT_CICLO_MU30DF", "TOT_MOTO_MU30DF", "TOT_VMP_MU30DF") if c in acc]
acc["vulnerable_deaths"] = acc[vul].sum(axis=1)

def make_segment(df, w=SEGMENT_KM):
    """Linear referencing: a segment is a road identifier plus a w-km window. Only interurban records with a valid road and km are assigned;
    roads flagged 'INVENTARIADA' (not in the national inventory) have unreliable kilometre origins and are excluded."""
    ok = (df.ZONA_AGRUPADA == 1) & df.KM.notna() & df.CARRETERA.notna() & ~df.CARRETERA.astype(str).str.contains("INVENTARIADA")
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    k = (np.floor(df.loc[ok, "KM"] / w) * w).astype(int).astype(str)
    out.loc[ok] = df.loc[ok, "CARRETERA"].astype(str) + "|" + k
    return out
acc["segment"] = make_segment(acc, SEGMENT_KM)
print("Fatal crashes (30 d) per year:", acc.groupby("ANYO").fatal30.sum().to_dict())
print("Locatable interurban fatal crashes (road + km):",
      acc[(acc.fatal30 == 1) & (acc.ZONA_AGRUPADA == 1)].groupby("ANYO").segment.apply(lambda s: s.notna().mean()).round(3).to_dict())

## 2 · Overview

**Purpose.** Establish the descriptive baseline against which every later inferential result is read: annual headline indicators by area type, the seasonal and hourly structure of fatal crashes, and — importantly — how much the count of deaths grows between the 24-hour and the 30-day definitions.

The 24 h → 30 d uplift is not a cosmetic detail. It differs by road-user type (it is larger for pedestrians, cyclists and motorcyclists, whose injuries more often prove fatal after hospital admission) and by area (urban deaths are registered later). Comparing a 24-hour series with a 30-day series, or pooling jurisdictions with different follow-up practices, would therefore introduce a spurious difference. **All results below use 30-day deaths** unless stated otherwise.

In [ ]:
kpi = acc.groupby("ANYO").agg(crashes=("UID", "size"), fatal_crashes=("fatal30", "sum"), deaths_30d=("TOTAL_MU30DF", "sum"),
                              deaths_24h=("TOTAL_MU24H", "sum"), serious_injuries_30d=("TOTAL_HG30DF", "sum"),
                              minor_injuries_30d=("TOTAL_HL30DF", "sum"), casualties_30d=("TOTAL_VICTIMAS_30DF", "sum"),
                              vulnerable_deaths=("vulnerable_deaths", "sum"))
kpi["% fatal crashes"] = 100 * kpi.fatal_crashes / kpi.crashes
kpi["case fatality (deaths/casualties, %)"] = 100 * kpi.deaths_30d / kpi.casualties_30d
kpi["24 h -> 30 d uplift (%)"] = 100 * (kpi.deaths_30d / kpi.deaths_24h - 1)
kpi["% vulnerable-user deaths"] = 100 * kpi.vulnerable_deaths / kpi.deaths_30d
display(kpi.T)
TABLES["kpi_annual"] = kpi

kz = acc.groupby(["ANYO", "area"]).agg(crashes=("UID", "size"), fatal_crashes=("fatal30", "sum"),
                                       deaths_30d=("TOTAL_MU30DF", "sum"), deaths_24h=("TOTAL_MU24H", "sum"))
kz["% fatal"] = 100 * kz.fatal_crashes / kz.crashes
kz["uplift 24 h -> 30 d (%)"] = 100 * (kz.deaths_30d / kz.deaths_24h - 1)
display(kz)
TABLES["kpi_annual_by_area"] = kz

# Figure 1: 30-day deaths by year and area type (counts are annotated so the plot can be read without the table).
pz = acc.pivot_table(index="ANYO", columns="area", values="TOTAL_MU30DF", aggfunc="sum")
fig, ax = plt.subplots(figsize=(8, 4)); w = 0.38; x = np.arange(len(pz))
for i, c in enumerate(pz.columns):
    b = ax.bar(x + (i - (len(pz.columns) - 1) / 2) * (w + 0.03), pz[c], width=w, color=SERIES[i], label=c)
    ax.bar_label(b, fmt="{:,.0f}", padding=2, color=INK2, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(pz.index); ax.set_title("30-day road deaths by year and area type"); ax.set_ylabel("Deaths within 30 days")
ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(.5, -.1))
show(fig, "01_deaths_by_year_area")

In [ ]:
# ---- Seasonality and hourly profile -------------------------------------------------------------------
# Monthly deaths expose the summer/holiday structure; the hourly case-fatality curve (with Wilson bands) shows that the *share* of
# casualty crashes that are fatal peaks at night even though the *number* of crashes peaks at commuting hours.
mm = acc.groupby(["ANYO", "MES"]).TOTAL_MU30DF.sum().unstack(0)
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, y in enumerate(mm.columns):
    ax.plot(mm.index, mm[y], color=SERIES[i % 8], marker="o", ms=6, mec=SURFACE, mew=1.5, label=str(y))
    ax.annotate(str(y), (mm.index[-1], mm[y].iloc[-1]), xytext=(7, 0), textcoords="offset points", color=INK2, va="center", fontsize=9)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTHS); ax.set_ylim(bottom=0); ax.set_ylabel("Deaths within 30 days")
ax.set_title("30-day deaths by month"); ax.legend(ncol=len(mm.columns), loc="upper center", bbox_to_anchor=(.5, -.1))
show(fig, "02_monthly_deaths")

h = acc.groupby(["area", "HORA"]).agg(n=("UID", "size"), x=("fatal30", "sum")).reset_index()
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, (z, g) in enumerate(h.groupby("area")):
    lo, hi = wilson(g.x, g.n)
    ax.fill_between(g.HORA, lo * 100, hi * 100, color=SERIES[i], alpha=.18, lw=0)
    ax.plot(g.HORA, g.x / g.n * 100, color=SERIES[i], label=z, marker="o", ms=5, mec=SURFACE, mew=1.2)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("Hour of the crash"); ax.set_ylabel("% of casualty crashes that are fatal")
ax.set_title("Case fatality by hour (band = 95% Wilson CI, all years pooled)"); ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(.5, -.18))
show(fig, "03_hourly_case_fatality")

hm = acc[acc.fatal30 == 1].pivot_table(index="DIA_SEMANA", columns="HORA", values="UID", aggfunc="count").reindex(index=range(1, 8), columns=range(24)).fillna(0)
fig, ax = plt.subplots(figsize=(10, 3.6)); im = ax.imshow(hm.values, cmap=SEQ_CMAP, aspect="auto")
ax.set_yticks(range(7)); ax.set_yticklabels(WEEKDAYS); ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2)); ax.grid(False)
ax.set_xlabel("Hour of the crash")
ax.set_title("Fatal crashes by weekday and hour (count, years pooled)"); fig.colorbar(im, ax=ax, pad=.01, label="fatal crashes")
show(fig, "04_heatmap_weekday_hour")

In [ ]:
# ---- How many more deaths does the 30-day definition add, and for whom? --------------------------------
# The answer depends on road-user type, which is why the follow-up definition must be fixed before any comparison across jurisdictions.
USER_TYPES = {"PEAT": "Pedestrian", "BICI": "Cyclist", "CICLO": "Moped rider", "MOTO": "Motorcyclist", "TUR": "Car occupant", "FURG": "Van occupant",
              "CAM_MENOS3500": "Truck ≤3.5 t", "CAM_MAS3500": "Truck >3.5 t", "BUS": "Bus occupant", "VMP": "E-scooter/PMV user", "OTRO": "Other", "SINESPECIF": "Unspecified"}
rows = []
for k, nm in USER_TYPES.items():
    a, b = f"TOT_{k}_MU24H", f"TOT_{k}_MU30DF"
    if a in acc and b in acc and acc[a].sum() > 0:
        rows.append(dict(road_user=nm, deaths_24h=acc[a].sum(), deaths_30d=acc[b].sum()))
inc = pd.DataFrame(rows).set_index("road_user"); inc["uplift 24 h -> 30 d (%)"] = 100 * (inc.deaths_30d / inc.deaths_24h - 1)
inc = inc.sort_values("uplift 24 h -> 30 d (%)")
display(inc); TABLES["uplift_24h_30d_by_user"] = inc
ic = inc[inc.deaths_24h >= 30]      # only users with >= 30 deaths at 24 h are plotted (with fewer, the percentage is noise); all appear in the table
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(ic) + 1.4)); b = ax.barh(ic.index, ic["uplift 24 h -> 30 d (%)"], color=SERIES[0], height=.55)
ax.bar_label(b, labels=[f"+{v:.0f}%  ({a:,}→{c:,})" for v, a, c in zip(ic["uplift 24 h -> 30 d (%)"], ic.deaths_24h, ic.deaths_30d)], padding=3, color=INK2, fontsize=8.5)
ax.grid(axis="y", visible=False); ax.grid(axis="x"); ax.set_xlim(right=ic["uplift 24 h -> 30 d (%)"].max() * 1.45)
ax.set_xlabel("Additional deaths between 24 h and 30 d (%)")
ax.set_title("Additional deaths between 24 h and 30 days, by road-user type (pooled)")
show(fig, "05_uplift_24h_30d")

## 3 · Comparing years properly

**Purpose.** Answer "has anything changed?" without being fooled. Three pitfalls recur in year-on-year road-safety comparisons and each is handled explicitly:

1. **Differences that are noise.** Rate ratios are reported with an **exact 95 % confidence interval** (conditional-binomial construction for two Poisson counts). The ratio is computed for *fatal crashes* (events that are closer to independent) and for *deaths* (which cluster within crashes and are therefore over-dispersed, so the Poisson interval for deaths is optimistic — see the quasi-Poisson model below).
2. **Calendar.** A year can contain more Fridays in summer months than another. Counts are re-expressed on a **reference calendar** (month × weekday, averaged over the years analysed). *Public holidays and long weekends are not adjusted.*
3. **Composition.** If the share of a more lethal road type grows, aggregate case fatality rises even if no road type became more dangerous. The **Kitagawa decomposition** splits any change in a proportion exactly into a *composition* effect (change in the mix) and a *rate* effect (change within strata).

Reading guide: when the rate-ratio interval contains 1 the data are compatible with no change; when the composition effect dominates the rate effect, the headline movement does not reflect a change in the *intrinsic* lethality of any stratum.

In [ ]:
def poisson_rr(x1, t1, x0, t0, alpha=0.05):
    """Rate ratio (x1/t1)/(x0/t0) of two Poisson counts with an exact interval.
    Conditional on the total n = x1 + x0, x1 ~ Binomial(n, t1/(t1+t0)) when the true rates are equal, and the Clopper-Pearson
    interval for the binomial proportion maps monotonically onto the rate ratio. t = person-time (here, days in the year)."""
    n = int(x1 + x0)
    if n == 0: return (np.nan,) * 4
    res = stats.binomtest(int(x1), n, t1 / (t1 + t0))
    lo, hi = res.proportion_ci(confidence_level=1 - alpha, method="exact")
    f = lambda p: (np.float64(p) / (1 - np.float64(p))) * (t0 / t1) if p < 1 else np.inf
    return f(x1 / n), f(lo), f(hi), res.pvalue

def rr_table(df, value_col, label, by=None):
    """Rate ratios of each year against the first year, optionally within the levels of a grouping variable."""
    out = []
    for key, g in ([("Total", df)] if by is None else df.groupby(by)):
        tot = g.groupby("ANYO")[value_col].sum(); yrs = sorted(tot.index); base = yrs[0]
        days = {y: 366 if calendar.isleap(y) else 365 for y in yrs}       # person-time = days, so leap years do not bias the ratio
        for y in yrs[1:]:
            rr, lo, hi, p = poisson_rr(tot[y], days[y], tot[base], days[base])
            out.append(dict(group=key, measure=label, year=y, vs=base, n=int(tot[y]), n_base=int(tot[base]), RR=rr, CI95_low=lo, CI95_high=hi, p=p))
    return pd.DataFrame(out)

if MULTI:
    rr = pd.concat([rr_table(acc, "fatal30", "fatal crashes"), rr_table(acc, "TOTAL_MU30DF", "30-day deaths"),
                    rr_table(acc, "fatal30", "fatal crashes", by="area")], ignore_index=True)
    display(rr); TABLES["rate_ratios_vs_base"] = rr
    d = rr[(rr.group == "Total") & (rr.measure == "fatal crashes")]
    fig, ax = plt.subplots(figsize=(6.5, 0.7 * len(d) + 1.5)); yy = np.arange(len(d))
    ax.hlines(yy, d.CI95_low, d.CI95_high, color=SERIES[0], lw=2); ax.scatter(d.RR, yy, s=60, color=SERIES[0], zorder=3, ec=SURFACE, lw=1.5)
    ax.axvline(1, color=INK2, lw=1); ax.set_yticks(yy); ax.set_yticklabels([f"{a} vs {b}" for a, b in zip(d["year"], d["vs"])]); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel("Rate ratio per day (reference = 1)")
    ax.set_title("Fatal crashes: daily rate ratio (95% exact CI)"); ax.invert_yaxis()
    show(fig, "06_forest_rate_ratios")
else:
    print("Only one year is loaded: Section 3 requires at least two years.")

In [ ]:
# ---- Calendar standardisation (month × weekday) ---------------------------------------------------------
# Direct standardisation: each (month, weekday) cell's count is rescaled to the number of such days in a reference calendar
# (the average calendar of the years analysed). The adjusted total answers: "what would this year's count have been under the reference calendar?"
def calendar_days(years):
    out = []
    for y in years:
        d = pd.date_range(f"{y}-01-01", f"{y}-12-31")
        out.append(pd.DataFrame({"ANYO": y, "MES": d.month, "DIA_SEMANA": d.dayofweek + 1}).groupby(["ANYO", "MES", "DIA_SEMANA"]).size().rename("days"))
    return pd.concat(out).reset_index()

cal = calendar_days(YL)
ref_cal = cal.groupby(["MES", "DIA_SEMANA"]).days.mean().rename("days_ref").reset_index()
cell = (acc.groupby(["ANYO", "MES", "DIA_SEMANA"]).agg(fatal_crashes=("fatal30", "sum"), deaths=("TOTAL_MU30DF", "sum")).reset_index()
        .merge(cal, on=["ANYO", "MES", "DIA_SEMANA"], how="right").fillna({"fatal_crashes": 0, "deaths": 0}).merge(ref_cal, on=["MES", "DIA_SEMANA"]))
for c in ("fatal_crashes", "deaths"):
    cell[c + "_std"] = cell[c] / cell.days * cell.days_ref
adj = cell.groupby("ANYO").agg(fatal_crashes=("fatal_crashes", "sum"), fatal_crashes_ref_calendar=("fatal_crashes_std", "sum"),
                               deaths=("deaths", "sum"), deaths_ref_calendar=("deaths_std", "sum"))
adj["deaths_diff"] = adj.deaths_ref_calendar - adj.deaths
display(adj); TABLES["calendar_adjustment"] = adj
print("Reading: 'deaths_diff' is how much the annual total would change if every year had the same mix of months and weekdays.")

In [ ]:
# ---- Kitagawa decomposition: composition change or rate change? ----------------------------------------
# For a proportion P = sum_k w_k p_k (w = stratum weight, p = stratum rate), the change between two periods splits exactly into
#   composition = sum_k (w_k^b - w_k^a) * (p_k^a + p_k^b)/2     and     rate = sum_k (p_k^b - p_k^a) * (w_k^a + w_k^b)/2.
# The symmetric (midpoint) weighting removes the order-dependence of the textbook version, so the two terms add up to the total with no residual.
def kitagawa(df, group, y, a, b):
    A, B = df[df.ANYO == a], df[df.ANYO == b]
    ga = A.groupby(group).agg(n=(y, "size"), p=(y, "mean")); gb = B.groupby(group).agg(n=(y, "size"), p=(y, "mean"))
    t = ga.join(gb, lsuffix="_a", rsuffix="_b", how="outer").fillna(0)
    wa, wb = t.n_a / t.n_a.sum(), t.n_b / t.n_b.sum()
    comp = (wb - wa) * (t.p_a + t.p_b) / 2
    rate = (t.p_b - t.p_a) * (wa + wb) / 2
    tab = pd.DataFrame({"weight_a_%": 100 * wa, "weight_b_%": 100 * wb, "rate_a_%": 100 * t.p_a, "rate_b_%": 100 * t.p_b,
                        "composition_effect_pp": 100 * comp, "rate_effect_pp": 100 * rate})
    tot = dict(total_change=(wb * t.p_b).sum() - (wa * t.p_a).sum(), composition=comp.sum(), rate=rate.sum())
    return tab, tot

if MULTI:
    a, b = YL[0], YL[-1]
    for grp in (["area", "road_group"], ["area", "crash_type"]):
        tab, tot = kitagawa(acc, grp, "fatal30", a, b)
        print(f"\n% fatal crashes {a}→{b} by {grp}: " + " · ".join(f"{k} = {100 * v:+.3f} pp" for k, v in tot.items()))
        display(tab.assign(abs_tot=tab["rate_effect_pp"].abs() + tab["composition_effect_pp"].abs()).sort_values("abs_tot", ascending=False).drop(columns="abs_tot").head(10).round(4))
        TABLES["kitagawa_" + "_".join(grp)] = tab

In [ ]:
# ---- Over-dispersion and trend: quasi-Poisson GLM on the monthly series (year + month effects) -------
# Deaths cluster in crashes (a multi-fatality crash adds several at once), so the variance exceeds the mean. The quasi-Poisson model keeps the
# Poisson mean structure but estimates a dispersion parameter phi from Pearson's chi-square, inflating the standard errors accordingly.
if MULTI:
    try:
        import statsmodels.api as sm, statsmodels.formula.api as smf
        m = acc.groupby(["ANYO", "MES"], as_index=False).agg(deaths=("TOTAL_MU30DF", "sum"))
        glm = smf.glm("deaths ~ C(ANYO) + C(MES)", data=m, family=sm.families.Poisson()).fit(scale="X2")
        ci = glm.conf_int()
        est = pd.DataFrame({"RR": np.exp(glm.params), "CI95_low": np.exp(ci[0]), "CI95_high": np.exp(ci[1]), "p": glm.pvalues})
        display(est.loc[est.index.str.startswith("C(ANYO)")])
        print(f"Dispersion parameter phi = {glm.scale:.2f}  (pure Poisson = 1; phi > 1 means plain Poisson intervals are too narrow — the intervals above are already corrected)")
        TABLES["quasi_poisson_year_effects"] = est
    except Exception as e:
        print("statsmodels is unavailable or failed:", type(e).__name__, e)

## 4 · Territory

**Purpose.** Quantify how much of the apparent variation among provinces is real. With roughly 1,650 fatal crashes a year spread over 52 provinces, **many apparent differences are sampling noise**: a small province with a handful of casualty crashes can display an extreme proportion by chance alone. Three complementary tools are used:

- **Empirical-Bayes shrinkage** (beta-binomial). Each province's proportion is pulled towards the national mean in proportion to how little information it carries; the amount of pull is estimated from the data (method of moments, Marshall 1991), not assumed. Intervals are posterior credible intervals.
- **Funnel plots** (Spiegelhalter 2005). Control limits widen as the number of crashes falls, so only provinces *outside* the funnel are candidates for a genuine difference. Using 95 % and 99.8 % limits guards against the multiple-comparison problem inherent in judging 52 units at once.
- **Between-year stability.** If a province's rank is not reproduced from one year to the next, there is no structural difference to explain. Spearman correlations and a reliability coefficient (share of between-province variance that is signal) make this explicit.

The outcome is **case fatality** = percentage of casualty crashes that are fatal within 30 days. It is *not* a risk: it depends on how many minor crashes are recorded, and a province that registers fewer minor crashes looks more lethal without being so. Section 6 introduces exposure to address this.

In [ ]:
def eb_beta(x, n):
    """Beta prior (a, b) for a set of proportions x/n by the method of moments (Marshall 1991).
    The between-unit variance tau2 is the observed weighted variance minus the expected binomial sampling variance;
    it is truncated at zero, in which case the prior collapses to a point mass at the pooled mean (maximal shrinkage).
    Returns a, b and tau2."""
    x, n = np.asarray(x, float), np.asarray(n, float)
    m = x.sum() / n.sum(); p = x / n
    s2 = np.sum(n * (p - m) ** 2) / n.sum()
    tau2 = s2 - m * (1 - m) * len(n) / n.sum()
    k = 1e6 if tau2 <= 1e-12 else max(m * (1 - m) / tau2 - 1, 2.0)      # k = a + b = prior "sample size"; floored at 2 for a proper prior
    return m * k, (1 - m) * k, max(tau2, 0.0)

def eb_table(df, keys):
    """Empirical-Bayes table for any grouping: raw proportion, posterior mean, 95 % credible interval and shrinkage factor
    (share of the raw estimate that is pulled to the prior: 0 = untouched, 1 = fully pooled)."""
    g = df.groupby(keys).agg(n=("UID", "size"), x=("fatal30", "sum"), deaths=("TOTAL_MU30DF", "sum")).reset_index()
    a, b, tau2 = eb_beta(g.x, g.n)
    g["raw"] = g.x / g.n; pa, pb = a + g.x, b + g.n - g.x
    g["eb"] = pa / (pa + pb); g["eb_lo"] = stats.beta.ppf(.025, pa, pb); g["eb_hi"] = stats.beta.ppf(.975, pa, pb)
    g["shrinkage"] = 1 - (g.n / (g.n + a + b))
    return g, a, b, tau2

def caterpillar(df, title, name):
    """Caterpillar plot of provincial EB estimates. Colour encodes whether the 95 % credible interval excludes the national mean."""
    g, a, b, tau2 = eb_table(df, ["COD_PROVINCIA", "province"])
    p0 = g.x.sum() / g.n.sum(); g = g.sort_values("eb").reset_index(drop=True)
    col = np.where(g.eb_lo > p0, UP, np.where(g.eb_hi < p0, DOWN, NEUTRAL))
    fig, ax = plt.subplots(figsize=(8, 0.22 * len(g) + 1.6)); yy = np.arange(len(g))
    ax.hlines(yy, g.eb_lo * 100, g.eb_hi * 100, color=col, lw=2.2)
    ax.scatter(g.eb * 100, yy, color=col, s=34, zorder=3, ec=SURFACE, lw=1.2)
    ax.scatter(g.raw * 100, yy, marker="|", color=INK2, s=45, zorder=2)
    ax.axvline(p0 * 100, color=INK2, lw=1, ls="--"); ax.set_yticks(yy); ax.set_yticklabels(g.province, fontsize=8); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel("% of casualty crashes that are fatal"); ax.set_title(title)
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], marker="o", color=UP, label="Credibly above the mean"), Line2D([], [], marker="o", color=DOWN, label="Credibly below"),
                       Line2D([], [], marker="o", color=NEUTRAL, label="Indistinguishable from the mean"), Line2D([], [], marker="|", color=INK2, ls="", label="Raw rate"),
                       Line2D([], [], color=INK2, ls="--", label="National mean")], loc="lower right", fontsize=8)
    show(fig, name)
    return g, tau2

def funnel(df, title, name, min_n=30):
    """Funnel plot: provincial proportion against the number of crashes, with exact-binomial-approximation control limits around the
    pooled proportion. Under the null of no between-province differences ~5 % of points should fall outside the 95 % limits and ~0.2 % outside 99.8 %."""
    g = df.groupby(["COD_PROVINCIA", "province"]).agg(n=("UID", "size"), x=("fatal30", "sum")).reset_index(); g = g[g.n >= min_n].copy()
    p0 = g.x.sum() / g.n.sum(); g["p"] = g.x / g.n; g["z"] = (g.p - p0) / np.sqrt(p0 * (1 - p0) / g.n)
    xs = np.geomspace(g.n.min() * 0.8, g.n.max() * 1.2, 200); se = np.sqrt(p0 * (1 - p0) / xs)
    fig, ax = plt.subplots(figsize=(9, 5))
    for z, ls in [(1.96, "--"), (3.09, "-")]:
        ax.plot(xs, np.clip(p0 + z * se, 0, 1) * 100, color=NEUTRAL, ls=ls, lw=1.2); ax.plot(xs, np.clip(p0 - z * se, 0, 1) * 100, color=NEUTRAL, ls=ls, lw=1.2)
    ax.axhline(p0 * 100, color=INK2, lw=1)
    col = np.where(g.z > 3.09, UP, np.where(g.z < -3.09, DOWN, NEUTRAL))
    ax.scatter(g.n, g.p * 100, c=col, s=42, ec=SURFACE, lw=1.2, zorder=3)
    for _, r in g[g.z.abs() > 3.09].reindex(g.z.abs().sort_values(ascending=False).index).dropna(subset=["z"]).head(12).iterrows():
        ax.annotate(r.province, (r.n, r.p * 100), xytext=(5, 4), textcoords="offset points", fontsize=8, color=INK2)
    ax.set_xscale("log"); ax.set_xlabel("Casualty crashes (log scale)"); ax.set_ylabel("% fatal"); ax.set_title(title)
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], color=NEUTRAL, ls="--", label="95% limit"), Line2D([], [], color=NEUTRAL, label="99.8% limit"),
                       Line2D([], [], marker="o", color=UP, ls="", label="Outside, above"), Line2D([], [], marker="o", color=DOWN, ls="", label="Outside, below")], fontsize=8, loc="upper right")
    show(fig, name)
    out95 = ((g.z.abs() > 1.96).mean() * 100); out998 = ((g.z.abs() > 3.09).mean() * 100)
    print(f"Provinces outside the limits: {out95:.0f}% (95% limits; ≈5% expected if there were no differences) · {out998:.0f}% (99.8% limits; ≈0.2% expected)")
    return g

prov_all, tau2_all = caterpillar(acc, "Case fatality by province · all roads, years pooled (EB + 95% credible interval)", "07_provinces_eb")
print("⚠️  With all road types pooled, strongly rural provinces appear 'more lethal' largely because a larger share of their crashes are interurban "
      "(far more lethal than urban ones). Compare within the same area type (next cells).")
display(prov_all.sort_values("eb", ascending=False).head(10)[["province", "n", "x", "raw", "eb", "eb_lo", "eb_hi", "shrinkage"]])
TABLES["provinces_eb"] = prov_all
prov_int, _ = caterpillar(acc[acc.ZONA_AGRUPADA == 1], "Case fatality by province · INTERURBAN roads only (EB + 95% credible interval)", "07b_provinces_eb_interurban")
TABLES["provinces_eb_interurban"] = prov_int

In [ ]:
f_all = funnel(acc, "Funnel plot · provincial case fatality (all roads)", "08_funnel_all")
f_int = funnel(acc[acc.ZONA_AGRUPADA == 1], "Funnel plot · provincial case fatality (interurban roads only)", "09_funnel_interurban")
f_urb = funnel(acc[acc.ZONA_AGRUPADA == 2], "Funnel plot · provincial case fatality (urban roads only)", "10_funnel_urban", min_n=100)
TABLES["funnel_all"] = f_all; TABLES["funnel_interurban"] = f_int; TABLES["funnel_urban"] = f_urb

In [ ]:
# ---- Are territorial differences stable across years? -----------------------------------------------
# Spearman rank correlation of provincial case fatality between pairs of years: values near 0 mean the ranking is not reproduced.
# Reliability = tau2 / (tau2 + typical sampling variance): the share of observed between-province variance that is signal for a single year.
if MULTI:
    for label, sub in [("all roads", acc), ("interurban", acc[acc.ZONA_AGRUPADA == 1]), ("urban", acc[acc.ZONA_AGRUPADA == 2])]:
        pv = sub.pivot_table(index="COD_PROVINCIA", columns="ANYO", values="fatal30", aggfunc="mean")
        print(f"\nSpearman correlation between years of provincial case fatality · {label}")
        display(pv.corr(method="spearman").round(2))
        TABLES["spearman_" + label.replace(" ", "_")] = pv.corr(method="spearman")
    rel = []
    for y in YL:
        s = acc[acc.ANYO == y]; g = s.groupby("COD_PROVINCIA").agg(n=("UID", "size"), x=("fatal30", "sum"))
        _, _, t2 = eb_beta(g.x, g.n); m0 = g.x.sum() / g.n.sum()
        rel.append(dict(year=y, between_province_sd_pp=100 * np.sqrt(t2), single_year_reliability=t2 / (t2 + m0 * (1 - m0) / g.n.median())))
    rel = pd.DataFrame(rel).set_index("year"); display(rel)
    print("Reliability ≈ share of the between-province variation that is 'signal'. Close to 0 → a single-year ranking is almost entirely noise.")

In [ ]:
# ---- Optional: per-capita rates from a legacy long CSV (Section 6 supersedes this with INE tables) ----
if POPULATION_CSV and Path(POPULATION_CSV).exists():
    pop = pd.read_csv(POPULATION_CSV)
    keys = ["COD_PROVINCIA"] + (["ANYO"] if "ANYO" in pop else [])
    d = acc.groupby(["COD_PROVINCIA"] + (["ANYO"] if "ANYO" in pop else [])).agg(deaths=("TOTAL_MU30DF", "sum"), fatal_crashes=("fatal30", "sum")).reset_index().merge(pop[keys + ["POBLACION"]], on=keys)
    d["deaths_per_million"] = d.deaths / d.POBLACION * 1e6
    display(d.sort_values("deaths_per_million", ascending=False).head(15)); TABLES["deaths_per_million_legacy"] = d
else:
    print("POPULATION_CSV = None → legacy per-capita table skipped (Section 6 computes per-capita rates from the INE table).")

# ---- Optional: choropleth map (needs geopandas and province polygons carrying the INE code) ----
if PROV_GEO and PROV_CODE_COL:
    try:
        import geopandas as gpd
        g = gpd.read_file(PROV_GEO); g["COD_PROVINCIA"] = pd.to_numeric(g[PROV_CODE_COL], errors="coerce").astype("Int64")
        m = g.merge(prov_all[["COD_PROVINCIA", "eb"]].assign(COD_PROVINCIA=lambda d: d.COD_PROVINCIA.astype("Int64")), on="COD_PROVINCIA", how="left")
        fig, ax = plt.subplots(figsize=(8, 7)); m.assign(eb=m.eb * 100).plot(column="eb", cmap=SEQ_CMAP, legend=True, ax=ax, edgecolor=SURFACE, linewidth=.4, missing_kwds={"color": GRID})
        ax.set_axis_off(); ax.set_title("Provincial case fatality (EB, %)"); show(fig, "11_province_map")
    except Exception as e:
        print("The map could not be drawn:", type(e).__name__, e)
else:
    print("PROV_GEO / PROV_CODE_COL = None → no map. Provide a province geojson/shapefile and the name of its INE-code column.")

In [ ]:
# ---- Municipalities >= 5,000 inhabitants (COD_MUNICIPIO > 0): all years pooled ----------------------
# Smaller municipalities are coded 0 by the DGT (statistical secrecy), so this table covers only the larger ones.
mun = acc[acc.COD_MUNICIPIO > 0]
mt, a_m, b_m, t2_m = eb_table(mun, ["COD_MUNICIPIO"])
prov_of = mun.groupby("COD_MUNICIPIO").province.first()
by_year = mun.pivot_table(index="COD_MUNICIPIO", columns="ANYO", values="fatal30", aggfunc="sum").add_prefix("fatal_crashes_")
mt = mt.set_index("COD_MUNICIPIO").join(prov_of).join(by_year)
print(f"{len(mt)} municipalities with a code · {int((acc.COD_MUNICIPIO == 0).mean() * 100)}% of crashes have COD_MUNICIPIO = 0 (municipality < 5,000 inhabitants or no data)")
display(mt.sort_values("x", ascending=False).head(20).drop(columns=["shrinkage"]))
TABLES["municipalities_top"] = mt.sort_values("x", ascending=False).head(200)
print("Codes are INE codes (2-digit province + 3-digit municipality). Join with the INE municipality code table to obtain names.")

## 5 · Roads and segments (linear referencing)

**Purpose.** On interurban roads, `CARRETERA` + `KM` place a crash along a road: the finest spatial resolution available in the microdata. Before speaking of "black spots" three cautions apply, and each is tested:

- **Location quality.** What fraction of crashes carry a kilometre point? How much is the kilometre rounded? A high share of exact integers signals coarser recording, which may depend on who records (e.g. regional vs. national police) and thus produces *administrative* rather than *physical* clustering.
- **Sparsity.** Within one year almost no segment records a repeat fatal crash. Years must be pooled, and the real question is whether the past **predicts** the future.
- **Regression to the mean.** A segment with two fatal crashes this year will tend to have fewer the next even if nothing is done; ranking by recent counts systematically overstates the danger of top-ranked sites. This is quantified, together with sensitivity to segment length (the Modifiable Areal Unit Problem, MAUP).

In [ ]:
inter = acc[acc.ZONA_AGRUPADA == 1]
noinv = inter.CARRETERA.astype(str).str.contains("INVENTARIADA")
q = inter.groupby("ANYO").agg(crashes=("UID", "size"), missing_km_pct=("KM", lambda s: s.isna().mean() * 100))
q["non_inventoried_pct"] = noinv.groupby(inter.ANYO).mean() * 100
q["fatal_crashes"] = inter.groupby("ANYO").fatal30.sum()
q["locatable_fatal_pct"] = inter[inter.fatal30 == 1].groupby("ANYO").segment.apply(lambda s: s.notna().mean() * 100)
display(q); TABLES["location_quality"] = q

kk = inter[inter.KM.notna()].assign(integer_km=lambda d: (d.KM % 1 == 0))
kq = kk.groupby(["ANYO", "ownership"]).integer_km.mean().unstack() * 100
print("\n% of records whose KM is exactly an integer, by year and road ownership (an indicator of recording precision):")
display(kq); TABLES["integer_km_by_ownership"] = kq
fig, ax = plt.subplots(figsize=(8, 4)); w = 0.8 / len(kq.columns); x = np.arange(len(kq))
for i, c in enumerate(kq.columns[:6]):
    ax.bar(x + (i - (len(kq.columns[:6]) - 1) / 2) * w, kq[c], width=w * .9, color=SERIES[i], label=c)
ax.set_xticks(x); ax.set_xticklabels(kq.index); ax.set_ylabel("% of records"); ax.set_title("% of records with an integer km · by road ownership"); ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.1))
show(fig, "12_integer_km")

In [ ]:
rd = inter[~noinv.reindex(inter.index, fill_value=False)]
top = rd.groupby("CARRETERA").agg(crashes=("UID", "size"), fatal_crashes=("fatal30", "sum"), deaths=("TOTAL_MU30DF", "sum")).sort_values("fatal_crashes", ascending=False).head(25)
top = top.join(rd.pivot_table(index="CARRETERA", columns="ANYO", values="fatal30", aggfunc="sum").add_prefix("fatal_").reindex(top.index))
display(top); TABLES["top_roads"] = top
print("Note: these are counts, not risks. A long, busy road accumulates more crashes simply by being long and busy. Rates need road length and AADT per segment (see Sections 6-7).")

n_segments = rd[rd.fatal30 == 1].groupby("segment").size()
print(f"\n{SEGMENT_KM}-km segments with >=1 fatal crash (all years): {len(n_segments)} · with >=2: {(n_segments >= 2).sum()} · with >=3: {(n_segments >= 3).sum()}")

In [ ]:
# ---- Does the past predict where fatal crashes will occur? Predictive validity of "black spots" ------
# A ranking of segments built on *training* years is scored on *test* years: the hit curve gives the share of future fatal crashes that fall
# in the top x % of segments. Random ranking yields the diagonal. Three scores are compared: previous fatal crashes, previous casualty crashes,
# and a severity-weighted count (10 per death, 3 per serious injury, 1 per minor injury; weights adjustable).
def segment_table(df, years, w):
    sub = df[df.ANYO.isin(years)]; t = make_segment(sub, w); sub = sub[t.notna()].assign(segment=t[t.notna()])
    sub = sub.assign(sev=10 * sub.TOTAL_MU30DF + 3 * sub.TOTAL_HG30DF + sub.TOTAL_HL30DF)
    return sub.groupby("segment").agg(crashes=("UID", "size"), fatal=("fatal30", "sum"), sev=("sev", "sum"))

def hit_curve(train_years, test_years, w, score, fracs, U):
    tr, te = segment_table(acc, train_years, w), segment_table(acc, test_years, w)
    order = (tr[score] + rng.uniform(0, 1e-6, len(tr))).sort_values(ascending=False).index          # random tie-breaking: many segments share the same score
    cum = te["fatal"].reindex(order).fillna(0).cumsum().to_numpy(); tot = te["fatal"].sum()
    return np.array([cum[max(1, min(int(round(f * U)), len(cum))) - 1] / tot for f in fracs])

if MULTI:
    U = len(segment_table(acc, YL, SEGMENT_KM))      # universe: segments with >=1 casualty crash in any year (an approximation of the road network used)
    SCHEMES = {f"{y}→{y + 1}": ([y], [y + 1]) for y in YL[:-1] if y + 1 in YL}
    if len(YL) >= 3: SCHEMES[f"{YL[0]}–{YL[-2]}→{YL[-1]}"] = (YL[:-1], [YL[-1]])
    if len(YL) == 4: SCHEMES[f"{YL[0]}–{YL[1]}→{YL[2]}–{YL[3]}"] = (YL[:2], YL[2:])
    FR = [0.01, 0.02, 0.05, 0.10]; rows = []
    for nm, (tr_y, te_y) in SCHEMES.items():
        for sc, lab in [("fatal", "previous fatal crashes"), ("crashes", "previous casualty crashes"), ("sev", "previous severity-weighted count")]:
            h = hit_curve(tr_y, te_y, SEGMENT_KM, sc, FR, U); rows.append(dict(scheme=nm, score=lab, **{f"top {int(f * 100)}%": v * 100 for f, v in zip(FR, h)}))
    hr = pd.DataFrame(rows); display(hr.round(1))
    print(f"Reading: 'top 5%' = % of future fatal crashes falling in the best-ranked 5% of segments (out of {U:,} with casualties) using previous data only. Chance = 5%.")
    TABLES["segment_predictive_validity"] = hr

    nm_last = f"{YL[0]}–{YL[-2]}→{YL[-1]}" if len(YL) >= 3 else list(SCHEMES)[0]; tr_y, te_y = SCHEMES[nm_last]
    fr = np.geomspace(0.002, 0.30, 40); fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for i, (sc, lab) in enumerate([("fatal", "Previous fatal crashes"), ("crashes", "Previous casualty crashes"), ("sev", "Previous severity-weighted count")]):
        ax.plot(fr * 100, hit_curve(tr_y, te_y, SEGMENT_KM, sc, fr, U) * 100, color=SERIES[i], label=lab)
    ax.plot(fr * 100, fr * 100, color=NEUTRAL, ls="--", label="Chance"); ax.set_xscale("log"); ax.set_xlabel("% of segments flagged as 'black spots' (log scale)"); ax.set_ylabel("% of future fatal crashes captured")
    ax.set_title(f"Gain curve · train {nm_last.split('→')[0]} → evaluate {YL[-1]}"); ax.legend(fontsize=8, loc="upper left"); ax.grid(axis="x")
    show(fig, "13_segment_gain_curve")

    tr, te = segment_table(acc, tr_y, SEGMENT_KM), segment_table(acc, te_y, SEGMENT_KM)
    j = tr.join(te, lsuffix="_tr", rsuffix="_te", how="outer").fillna(0)
    j["previous_fatal"] = j.fatal_tr.clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
    rtm = j.groupby("previous_fatal").agg(segments=("fatal_te", "size"), future_fatal_per_segment_year=("fatal_te", lambda s: s.mean() / len(te_y)))
    print("\nRegression to the mean: fatal crashes per segment-year in the test period by number of previous fatal crashes"); display(rtm); TABLES["regression_to_the_mean"] = rtm

    rows = []
    for w in (1, 2, 5, 10, 25):
        Uw = len(segment_table(acc, YL, w)); h = hit_curve(tr_y, te_y, w, "sev", [0.05], Uw)[0]; rows.append(dict(segment_km=w, segments=Uw, top5_capture_pct=h * 100))
    maup = pd.DataFrame(rows).set_index("segment_km"); print("\nSensitivity to segment length (MAUP), score = severity-weighted count"); display(maup.round(1)); TABLES["maup_segments"] = maup
    print("Caveats: the 'universe' network is approximated by segments that had >=1 casualty crash; roads are renumbered/renamed and kilometre origins can shift between years.")
else:
    print("Only one year is loaded: temporal validation of segments requires at least two years.")

## 6 · Exposure: from case fatality to **risk**

Case fatality (fatal crashes ÷ casualty crashes) **is not risk**. A province can have few but very severe crashes, or many but mild ones; and because the denominator is itself an outcome of the recording process, it cannot say how likely a resident or a vehicle is to be killed. Risk requires a measure of **exposure** in the denominator. Here the denominator is the resident population (distance-based exposure, i.e. vehicle-kilometres, is not available by province):

| Denominator | Source | What it measures | Limitation |
|---|---|---|---|
| Inhabitants | INE, population at 1 January (table 2852 or 67988) | Risk per resident | Tourism and through traffic inflate tourist or transit provinces |

**Person-years.** The population entering the denominator for year *y* is the mean of the 1-January figures of *y* and *y*+1, which approximates the person-time lived during the year. If the following year is unavailable the 1-January figure is used and the source is recorded in the table.

**Multiplicative decomposition.** The excess (or deficit) risk of a province relative to Spain is split exactly, on the log scale, into three factors: **how many casualty crashes** occur per inhabitant (frequency) × **how severe** they are (fatal crashes per casualty crash) × **how many deaths per fatal crash** (multiplicity). This is the quantity that tells a policy-maker *why* a province is dangerous. The three terms add up to the log risk ratio with no residual.

In [ ]:
# ---- Exposure readers (all tolerant: if a file is missing that block is skipped and a message is printed) ----
def _read_any(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(path, header=None, dtype=object)
    return pd.read_csv(path, header=None, sep=None, engine="python", dtype=object, encoding="utf-8-sig")

def _num(s):
    """Parse numbers written with Spanish formatting (dot as thousands separator, comma as decimal)."""
    s = s.astype(str).str.strip().str.replace(" ", "", regex=False)
    s = s.where(~s.str.fullmatch(r"-?\d{1,3}(\.\d{3})+(,\d+)?"), s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(s, errors="coerce")

def read_ine_pop(path):
    """Read an INE population export by province: table 2852 (official figures at 1 January) or table 67988 (annual census population;
    it interleaves autonomous-community rows, which are discarded). Several years in columns and Total/Men/Women blocks are supported
    (only 'Total' is kept), as is a long CSV with COD_PROVINCIA, ANYO, POBLACION. Returns COD_PROVINCIA, ANYO, POP_1JAN."""
    raw = _read_any(path).fillna("")
    first = raw.iloc[0].astype(str).str.upper().tolist()
    if "COD_PROVINCIA" in first and "POBLACION" in first:                    # long format
        d = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig"); d.columns = [c.upper() for c in d.columns]
        if "ANYO" not in d: raise ValueError("The long CSV needs an ANYO column")
        return d.rename(columns={"POBLACION": "POP_1JAN"})[["COD_PROVINCIA", "ANYO", "POP_1JAN"]].astype({"COD_PROVINCIA": int, "ANYO": int})
    is_year = lambda v: bool(re.fullmatch(r"(19|20)\d\d", str(v).strip().split(".")[0]))
    yr = next((i for i in range(min(len(raw), 40)) if sum(is_year(v) for v in raw.iloc[i, 1:]) >= 1), None)
    if yr is None: raise ValueError("Cannot find the row of years in the INE file.")
    years = raw.iloc[yr, 1:].map(lambda v: int(str(v).strip().split(".")[0]) if is_year(v) else np.nan)
    grp = raw.iloc[yr - 1, 1:].astype(str).str.strip().replace("", np.nan).ffill() if yr > 0 else pd.Series("Total", index=years.index)
    cols = [c for c in years.index if pd.notna(years[c]) and (pd.isna(grp.get(c)) or str(grp.get(c)).lower() in ("total", "ambos sexos", "nan"))]
    cand = []                                                     # (code, values by year) for every row starting with "NN name"
    for i in range(yr + 1, len(raw)):
        m = re.match(r"^\s*(\d{2})\s+\S", str(raw.iloc[i, 0]))
        if m: cand.append((int(m.group(1)), [raw.iloc[i, c] for c in cols]))
    if len({c for c, _ in cand}) < len(cand):                      # community rows are interleaved (their codes 01-19 clash with province codes, e.g. table 67988)
        v0 = [pd.to_numeric(pd.Series([r[0]]), errors="coerce").iloc[0] for _, r in cand]; keep, i = [], 0
        while i < len(cand):                                       # a community row is the row whose value equals the sum of the following 1-9 rows (its provinces)
            s, hit = 0.0, 0
            for j in range(i + 1, min(i + 10, len(cand))):
                s += v0[j]
                if abs(s - v0[i]) < 0.5: hit = j - i; break
                if s > v0[i] + 0.5: break
            if hit: keep += list(range(i + 1, i + hit + 1)); i += hit + 1
            else: keep.append(i); i += 1
        cand = [cand[k] for k in keep]
    out = [(code, int(years[c]), v) for code, vals in cand for c, v in zip(cols, vals)]
    d = pd.DataFrame(out, columns=["COD_PROVINCIA", "ANYO", "POP_1JAN"]); d["POP_1JAN"] = _num(d["POP_1JAN"])
    d = d.dropna().astype({"POP_1JAN": float}).drop_duplicates(["COD_PROVINCIA", "ANYO"])
    if d.COD_PROVINCIA.nunique() < 50: raise ValueError(f"Only {d.COD_PROVINCIA.nunique()} provinces were read; check the file format.")
    return d

def build_pop(p1, years, fixed=None):
    """Person-years of each analysis year = mean of the 1-January figures of that year and the next. If the next year is missing the 1-January figure
    is used; if the year itself is missing the nearest available year is used as a constant (flagged in POP_SOURCE). `fixed` forces one single year for all."""
    P = p1.pivot(index="COD_PROVINCIA", columns="ANYO", values="POP_1JAN"); av = sorted(P.columns); rows = []
    if fixed is not None and fixed not in P: raise ValueError(f"POPULATION_FIXED_YEAR={fixed} is not in the file (available years: {av})")
    for y in years:
        if fixed is not None: s, f = P[fixed], f"FIXED {fixed} (analyst's decision)"
        elif y in P and (y + 1) in P: s, f = (P[y] + P[y + 1]) / 2, "mean of 1 Jan y and y+1"
        elif y in P:                s, f = P[y], "1 Jan of the year"
        else:
            n = min(av, key=lambda a: abs(a - y)); s, f = P[n], f"CONSTANT {n} (approximation)"
        rows.append(pd.DataFrame({"COD_PROVINCIA": s.index, "ANYO": y, "POP": s.values, "POP_SOURCE": f}))
    return pd.concat(rows, ignore_index=True)

In [ ]:
# ---- Build the province x year exposure table ---------------------------------------------------------
YL = sorted(int(v) for v in acc.ANYO.unique())
_pop_path = POPULATION_XLSX or POPULATION_CSV
expo = pd.DataFrame([(p, y) for p in sorted(acc.COD_PROVINCIA.unique()) for y in YL], columns=["COD_PROVINCIA", "ANYO"])
HAS_POP = False
if _pop_path and Path(_pop_path).exists():
    pop1 = read_ine_pop(_pop_path)
    print(f"INE population read: {pop1.COD_PROVINCIA.nunique()} provinces · years {[int(a) for a in sorted(pop1.ANYO.unique())]}")
    pop = build_pop(pop1, YL, POPULATION_FIXED_YEAR); expo = expo.merge(pop, on=["COD_PROVINCIA", "ANYO"], how="left"); HAS_POP = expo.POP.notna().all()
    if POPULATION_FIXED_YEAR is not None:
        print(f"FIXED population of {POPULATION_FIXED_YEAR} used for every year (analyst's decision). Per-capita rates of later years are biased in provinces that grow or shrink (±1-2 %/year): "
              "state this in the paper and, if possible, cross-check against an annual source (e.g. the Continuous Population Statistics).")
    const = [int(a) for a in expo[expo.POP_SOURCE.astype(str).str.startswith("CONSTANT")].ANYO.unique()]
    if len(const):
        print(f"⚠️  No population is available in the file for years {const} → the nearest year is used as a constant.\n"
              "   Acceptable for exploration, but for the paper download INE table 2852 with the years 2022 to 2026 as columns: "
              "provincial population changes by ±1-2 %/year and biases the rates of provinces that grow or shrink.")
else:
    print("POPULATION_XLSX = None or missing → no per-capita rates.")
if not HAS_POP: print("No population available: the risk sections are skipped.")
TABLES["exposure"] = expo

In [ ]:
# ---- Risk rates: deaths per million inhabitant-years with exact Poisson CI -----
def poisson_ci(x, t, alpha=0.05):
    """Exact (Garwood) Poisson interval for a rate x/t, via the chi-square relation."""
    x = np.asarray(x, float); lo = np.where(x > 0, stats.chi2.ppf(alpha / 2, 2 * x) / 2, 0.0); hi = stats.chi2.ppf(1 - alpha / 2, 2 * x + 2) / 2
    return lo / t, hi / t

cnt = acc.groupby(["COD_PROVINCIA", "ANYO"]).agg(crashes=("UID", "size"), fatal=("fatal30", "sum"), deaths=("TOTAL_MU30DF", "sum")).reset_index()
py = cnt.merge(expo, on=["COD_PROVINCIA", "ANYO"], how="left").merge(acc[["COD_PROVINCIA", "province"]].drop_duplicates(), on="COD_PROVINCIA")
if HAS_POP:
    T = py.groupby(["COD_PROVINCIA", "province"]).agg(crashes=("crashes", "sum"), fatal=("fatal", "sum"), deaths=("deaths", "sum"), person_years=("POP", "sum")).reset_index()
    T["rate_pm"] = T.deaths / T.person_years * 1e6; lo, hi = poisson_ci(T.deaths, T.person_years); T["lo"], T["hi"] = lo * 1e6, hi * 1e6
    nat = T.deaths.sum() / T.person_years.sum() * 1e6
    print(f"30-day deaths per million inhabitant-years (national, {YL[0]}-{YL[-1]}): {nat:.1f}")
    s = T.sort_values("rate_pm").reset_index(drop=True); col = np.where(s.lo > nat, UP, np.where(s.hi < nat, DOWN, NEUTRAL))
    fig, ax = plt.subplots(figsize=(8, 0.22 * len(s) + 1.6)); yy = np.arange(len(s))
    ax.hlines(yy, s.lo, s.hi, color=col, lw=2.2); ax.scatter(s.rate_pm, yy, c=col, s=34, zorder=3, ec=SURFACE, lw=1.2); ax.axvline(nat, color=INK2, ls="--", lw=1)
    ax.set_yticks(yy); ax.set_yticklabels(s.province, fontsize=8); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel("30-day deaths per million inhabitant-years (exact 95% CI, unsmoothed)"); ax.set_title("Risk of dying in a road crash, by province")
    show(fig, "17_risk_per_inhabitant_province"); TABLES["risk_per_inhabitant_province"] = T
    # Is case fatality a proxy for risk? Reported as a rank correlation across provinces (an empirical question; in the real 2022-2025 data it is high, rho ≈ 0.86).
    T["case_fatality"] = T.fatal / T.crashes; rho_s = stats.spearmanr(T.case_fatality, T.rate_pm)
    print(f"Spearman correlation between case fatality and per-capita risk across provinces: rho = {rho_s.statistic:.2f} (p = {rho_s.pvalue:.3f})")

In [ ]:
# ---- Decomposition of excess mortality: frequency x severity x deaths per fatal crash ----------------
# log(risk_p / risk_Spain) = log(f1_p/f1) + log(f2_p/f2) + log(f3_p/f3), with
#   f1 = casualty crashes per person-year,  f2 = fatal crashes per casualty crash,  f3 = deaths per fatal crash.
# The identity is exact (asserted below), so each province's excess is attributed without residual.
if HAS_POP:
    N = T.assign(f1=lambda d: d.crashes / d.person_years, f2=lambda d: d.fatal / d.crashes, f3=lambda d: d.deaths / d.fatal)
    g1, g2, g3 = N.crashes.sum() / N.person_years.sum(), N.fatal.sum() / N.crashes.sum(), N.deaths.sum() / N.fatal.sum()
    N["c_freq"], N["c_sev"], N["c_mult"] = np.log(N.f1 / g1), np.log(N.f2 / g2), np.log(N.f3 / g3)
    N["c_tot"] = N.c_freq + N.c_sev + N.c_mult                       # = log(provincial risk / national risk), exact
    assert np.allclose(N.c_tot, np.log(N.rate_pm / nat), atol=1e-9)
    N["share_freq"] = N.c_freq / N.c_tot.where(N.c_tot.abs() > .05)  # share of the excess due to frequency (only when |excess| > 5 %)
    sel = pd.concat([N.nlargest(10, "c_tot"), N.nsmallest(10, "c_tot")]).sort_values("c_tot").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, 7)); yy = np.arange(len(sel)); pos = np.zeros(len(sel)); neg = np.zeros(len(sel))
    for c, lab, colr in [("c_freq", "More/fewer casualty crashes per inhabitant", SERIES[0]), ("c_sev", "More/fewer fatal crashes per casualty crash", SERIES[1]), ("c_mult", "More/fewer deaths per fatal crash", SERIES[2])]:
        v = sel[c].values; b = np.where(v >= 0, pos, neg); ax.barh(yy, v, left=b, color=colr, label=lab, height=.7, ec=SURFACE, lw=.6); pos += np.where(v >= 0, v, 0); neg += np.where(v < 0, v, 0)
    ax.scatter(sel.c_tot, yy, color=INK, s=22, zorder=4, label="Total (log risk ratio)"); ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(yy); ax.set_yticklabels(sel.province, fontsize=8); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel("log(province / Spain)  ·  0.10 ≈ +10 %"); ax.set_title("Why is a province more (or less) dangerous? · 10 from each extreme"); ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(.5, -.09), ncol=2); show(fig, "18_risk_decomposition")
    TABLES["risk_decomposition"] = N
    fig, ax = plt.subplots(figsize=(7, 6)); sz = 30 + 400 * N.deaths / N.deaths.max()
    ax.scatter(N.c_freq, N.c_sev, s=sz, c=np.where(N.c_tot > .1, UP, np.where(N.c_tot < -.1, DOWN, NEUTRAL)), alpha=.8, ec=SURFACE, lw=1)
    for _, r in pd.concat([N.nlargest(6, "c_tot"), N.nsmallest(6, "c_tot")]).iterrows(): ax.annotate(r.province, (r.c_freq, r.c_sev), xytext=(4, 3), textcoords="offset points", fontsize=8, color=INK2)
    ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); ax.grid(True)
    ax.set_xlabel("Frequency of casualty crashes per inhabitant (log ratio)"); ax.set_ylabel("Severity: fatal crashes per casualty crash (log ratio)")
    ax.set_title("Two faces of risk: frequency vs severity"); show(fig, "19_frequency_vs_severity")
    r = stats.spearmanr(N.c_freq, N.c_sev); print(f"Frequency–severity correlation across provinces: Spearman rho = {r.statistic:.2f} (p = {r.pvalue:.3f}). Caution: part of a negative correlation is mechanical (log of crashes enters with a + sign in frequency and a − sign in severity); the clean comparison is the Bayesian model of Section 8.")

## 7 · Geocoding (road + kilometre point → coordinates) and spatial analysis

The DGT publishes no coordinates, but **on interurban roads each record carries a road identifier and a kilometre point (KP)**. If a layer with the positions of the KPs is available (e.g. the IGN/CNIG transport network), the position of a crash is obtained by **linear interpolation** between the integer KPs that bracket `KM`.

**What can and cannot be done:**
- Only **interurban roads** are geocodable (≈ 35 % of crashes but ≈ 70 % of deaths). Urban crashes carry no street, only a municipality.
- Coverage is bounded by the KP layer (in testing, ≈ 88 % of interurban fatal crashes of 2024 with an old layer; municipal roads are almost absent).
- There is no ground-truth coordinate. Quality is therefore assessed with **three internal checks**: coherence with the declared province, pairwise coherence of crashes on the same road, and — if `PROV_GEO` is supplied — point-in-polygon agreement.
- **Ambiguous candidates** (same road name and KP at positions more than 2 km apart, typically homonymous roads in different regions) are *excluded* from the spatial analysis rather than guessed.

**Spatial analysis.** Cells of `GRID_KM` km; exposure = **kilometres of network** from the KP layer inside the cell (a proxy for road length, **not for traffic** — AADT would be the right denominator); the rate of fatal crashes per network-km per year is smoothed with a gamma-Poisson empirical-Bayes prior; global (Moran's I) and local (Getis-Ord Gi\*) autocorrelation are assessed by permutation with Benjamini-Hochberg control of the false-discovery rate; and, above all, **do hot spots persist?** — they are detected on earlier years and evaluated on later ones, which is the only test that matters for prevention.

In [ ]:
# ---- Kilometre-point layer and geocoder ---------------------------------------------------------------
from scipy.spatial import cKDTree
try:
    import pyproj
except ImportError:
    pyproj = None; print("⚠️ pyproj is missing (pip install pyproj): coordinates cannot be projected to UTM; Section 7 will be skipped.")

def hav_km(lon1, lat1, lon2, lat2):
    """Great-circle (haversine) distance in km between points given in decimal degrees."""
    p = np.pi / 180; a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def resolve_pk_layer(pk_csv):
    """Return the path of the kilometre-point layer, or None. 'auto' uses DATA_DIR/puntosKilometricosSpain.csv and downloads it from PK_URL if it is not there
    (about 21 MB, once). A path is used as given. The download goes to a temporary name first so that an interrupted transfer never leaves a corrupt file."""
    if not pk_csv: return None
    if str(pk_csv).lower() != "auto": return Path(pk_csv) if Path(pk_csv).exists() else None
    target = DATA_DIR / "puntosKilometricosSpain.csv"
    if target.exists(): return target
    try:
        import urllib.request
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Downloading the kilometre-point layer (~21 MB) from {PK_URL} ...")
        tmp = target.with_suffix(".part"); urllib.request.urlretrieve(PK_URL, tmp); tmp.replace(target)
        print("Layer saved to", target.resolve(), "· compiled in 2022 from a DGT source: roads created or renumbered since may be absent. Check its licence and prefer the current IGN/CNIG layer for publication.")
        return target
    except Exception as e:
        print("The layer could not be downloaded:", type(e).__name__, e, "\nDownload it manually, save it as DATA_DIR/puntosKilometricosSpain.csv or set PK_CSV to its path.")
        return None

PK_PATH = resolve_pk_layer(PK_CSV)
HAS_PK = PK_PATH is not None and pyproj is not None
if HAS_PK:
    pk = pd.read_csv(PK_PATH, encoding="utf-8-sig", usecols=list(PK_COLS.values())).rename(columns={v: k for k, v in PK_COLS.items()})
    pk["road"] = pk.road.astype(str).str.strip().str.upper(); pk["km"] = pd.to_numeric(pk.km, errors="coerce")
    pk = pk.dropna(subset=["x", "y", "km"]); pk["km"] = pk.km.round().astype(int)
    if not (pk.x.between(-20, 6).mean() > .99 and pk.y.between(26, 45).mean() > .99):          # coordinates are not lon/lat: reproject
        t = pyproj.Transformer.from_crs(PK_CRS, 4326, always_xy=True); pk["x"], pk["y"] = t.transform(pk.x.values, pk.y.values)
        print(f"KP coordinates reprojected from {PK_CRS} to WGS84.")
    PKIDX = {}                                            # road -> {integer km -> array of (lon, lat) candidates}; several candidates = homonymous roads
    for nom, num, x, y in zip(pk.road.values, pk.km.values, pk.x.values, pk.y.values): PKIDX.setdefault(nom, {}).setdefault(int(num), []).append((x, y))
    PKIDX = {nom: {k: np.array(v) for k, v in d.items()} for nom, d in PKIDX.items()}
    print(f"KP layer: {len(pk):,} points · {len(PKIDX):,} roads")
else:
    print("No kilometre-point layer (PK_CSV = None, download failed, or pyproj not installed) → geocoding is skipped. A layer needs columns X, Y, Nombre (road) and numero (km).")

def pk_candidates(road, km):
    """Candidate positions of (road, km). Interpolate between the two bracketing KPs when they are contiguous (< 3 km apart, i.e. one continuous
    stretch); otherwise fall back to the nearest KP and flag the method, so that the quality of every coordinate is traceable."""
    d = PKIDX.get(road)
    if d is None: return "road absent", None
    k0 = int(np.floor(km)); fr = km - k0; a, b = d.get(k0), d.get(k0 + 1)
    if a is not None and b is not None:
        c = [pa + (pb - pa) * fr for pa in a for pb in b if hav_km(pa[0], pa[1], pb[0], pb[1]) < 3]
        if c: return "interpolated", np.array(c)
        return "nearest KP (inconsistent neighbour)", np.array(a if fr < .5 else b)
    pts = a if a is not None else b
    if pts is None: return "km out of range", None
    return "nearest KP (no neighbour)", np.array(pts)

def geocode(df, max_prov_km=PK_MAX_DIST_PROV_KM):
    """Adds geo_lon, geo_lat, geo_method, geo_disp (km between surviving candidates) and geo_ok. Interurban records with road and km only.
    Two passes: (1) candidates by (road, km); (2) candidates farther than max_prov_km from the robust centre (median of the interpolated
    positions) of the record's province/island are discarded, which removes homonymous roads in other regions. A coordinate is flagged reliable
    (geo_ok) only if the surviving candidates lie within 2 km of each other."""
    ok = (df.ZONA_AGRUPADA == 1) & df.KM.notna() & df.CARRETERA.notna() & ~df.CARRETERA.astype(str).str.contains("INVENTARIADA")
    s = df.loc[ok, ["CARRETERA", "KM", "COD_PROVINCIA", "ISLA"]].copy(); s["ISLA"] = s.ISLA.fillna(-1)
    cache = {}
    for road, km in s[["CARRETERA", "KM"]].drop_duplicates().itertuples(index=False): cache[(road, km)] = pk_candidates(road, km)
    s["method"] = [cache[(r, k)][0] for r, k in zip(s.CARRETERA, s.KM)]
    s["m_lon"] = [np.nan if cache[(r, k)][1] is None else cache[(r, k)][1][:, 0].mean() for r, k in zip(s.CARRETERA, s.KM)]
    s["m_lat"] = [np.nan if cache[(r, k)][1] is None else cache[(r, k)][1][:, 1].mean() for r, k in zip(s.CARRETERA, s.KM)]
    ref = s[s.method == "interpolated"].groupby(["COD_PROVINCIA", "ISLA"])[["m_lon", "m_lat"]].median().rename(columns={"m_lon": "c_lon", "m_lat": "c_lat"})
    s = s.join(ref, on=["COD_PROVINCIA", "ISLA"])
    lon, lat, met, disp = [], [], [], []
    for (r, k, cl, ca, m) in zip(s.CARRETERA, s.KM, s.c_lon, s.c_lat, s.method):
        c = cache[(r, k)][1]
        if c is None: lon.append(np.nan); lat.append(np.nan); met.append(m); disp.append(np.nan); continue
        if np.isfinite(cl): c = c[hav_km(c[:, 0], c[:, 1], cl, ca) <= max_prov_km]
        if len(c) == 0: lon.append(np.nan); lat.append(np.nan); met.append("inconsistent with province"); disp.append(np.nan); continue
        sp = max(hav_km(c[i, 0], c[i, 1], c[j, 0], c[j, 1]) for i in range(len(c)) for j in range(i + 1, len(c))) if len(c) > 1 else 0.0
        lon.append(c[:, 0].mean()); lat.append(c[:, 1].mean()); met.append(m); disp.append(sp)
    out = pd.DataFrame({"geo_lon": lon, "geo_lat": lat, "geo_method": met, "geo_disp": disp}, index=s.index)
    res = df[[]].join(out); res["geo_ok"] = res.geo_lon.notna() & (res.geo_disp.fillna(99) <= 2.0)
    return res

if HAS_PK:
    import time; t0 = time.time()
    G = geocode(acc); acc = acc.drop(columns=[c for c in G.columns if c in acc.columns]).join(G); acc["geo_ok"] = acc.geo_ok.fillna(False).astype(bool)
    print(f"Geocoding: {time.time() - t0:.0f} s")
    el = acc[acc.geo_method.notna()]
    print("\nCoverage among interurban crashes with road and km (all years):")
    display(pd.concat([el.geo_method.value_counts().rename("n"), (el.geo_method.value_counts(normalize=True) * 100).rename("%")], axis=1).round(1))
    cov = lambda d, k: d.groupby(k).agg(n=("geo_ok", "size"), reliable_coords_pct=("geo_ok", "mean")).assign(reliable_coords_pct=lambda x: x.reliable_coords_pct * 100)
    tt = pd.concat({"all": cov(el, "ANYO"), "fatal": cov(el[el.fatal30 == 1], "ANYO")}, axis=1)
    display(tt.round(1)); TABLES["geocoding_coverage_by_year"] = tt
    ct_own = cov(el[el.fatal30 == 1], "ownership"); display(ct_own.round(1)); TABLES["geocoding_coverage_by_ownership_fatal"] = ct_own
    missing = el[(el.fatal30 == 1) & (el.geo_method == "road absent")].CARRETERA.value_counts().head(15)
    print("Roads with fatal crashes that are NOT in the KP layer (candidates to complete from regional sources):", missing.to_dict())

In [ ]:
# ---- Are the coordinates reliable? Internal checks (there is no ground truth) -------------------------
if HAS_PK:
    g = acc[acc.geo_ok].copy()
    # (1) Coherence with the province: distance to the robust centre (median) of the declared province/island
    med = g.assign(ISLA=g.ISLA.fillna(-1)).groupby(["COD_PROVINCIA", "ISLA"])[["geo_lon", "geo_lat"]].transform("median")
    g["d_prov"] = hav_km(g.geo_lon, g.geo_lat, med.geo_lon, med.geo_lat)
    print("(1) Distance to the robust centre of the declared province/island (km):", g.d_prov.describe(percentiles=[.5, .95, .99]).round(1)[["50%", "95%", "99%", "max"]].to_dict())
    # (2) Pairwise coherence: two crashes on the same road separated by |ΔKM| must lie at a straight-line distance <= |ΔKM| (+ tolerance)
    rg = np.random.default_rng(SEED); viol, ratios = [], []
    for _, d in g.assign(ISLA=g.ISLA.fillna(-1)).groupby(["CARRETERA", "COD_PROVINCIA", "ISLA"]):
        if len(d) < 2: continue
        n = min(len(d) * 3, 60); i = rg.integers(0, len(d), n); j = rg.integers(0, len(d), n); m = i != j
        dk = np.abs(d.KM.values[i[m]] - d.KM.values[j[m]]); dg = hav_km(d.geo_lon.values[i[m]], d.geo_lat.values[i[m]], d.geo_lon.values[j[m]], d.geo_lat.values[j[m]])
        sel = (dk > 0.5) & (dk <= 40); viol += list(dg[sel] > dk[sel] * 1.05 + 2.0); ratios += list(dg[sel & (dk >= 5)] / dk[sel & (dk >= 5)])
    if viol:
        print(f"(2) Pairs on the same road: {np.mean(viol):.1%} violate 'straight line <= distance along the road + 2 km' (n={len(viol):,}); "
              f"straight-line / ΔKM ratio (ΔKM>=5): median {np.median(ratios):.2f}, p10 {np.percentile(ratios, 10):.2f}, p90 {np.percentile(ratios, 90):.2f}  (≈1 = straight; <1 = winding)")
    # (3) Point-in-polygon, if province polygons are supplied
    if PROV_GEO and PROV_CODE_COL:
        try:
            import geopandas as gpd
            pol = gpd.read_file(PROV_GEO).to_crs(4326); pol["COD"] = pd.to_numeric(pol[PROV_CODE_COL], errors="coerce")
            pts = gpd.GeoDataFrame(g[["COD_PROVINCIA"]], geometry=gpd.points_from_xy(g.geo_lon, g.geo_lat), crs=4326)
            j = gpd.sjoin(pts, pol[["COD", "geometry"]], how="left", predicate="within")
            conc = (j.COD == j.COD_PROVINCIA).mean(); print(f"(3) Point inside its declared province: {conc:.1%} (failures are usually within 5 km of the border)")
        except Exception as e:
            print("(3) Point-in-polygon check could not be run:", type(e).__name__, e)
    else:
        print("(3) Without PROV_GEO no point-in-polygon check is made (add it to quantify the province-level accuracy).")
    TABLES["geocoding_quality_summary"] = pd.DataFrame({"check": ["p99 distance to province centre (km)", "pairs violating consistency"], "value": [g.d_prov.quantile(.99), np.mean(viol) if viol else np.nan]})
    acc.loc[acc.geo_ok, ["UID", "ANYO", "geo_lon", "geo_lat", "geo_method", "fatal30", "TOTAL_MU30DF", "CARRETERA", "KM"]].to_csv(OUT_DIR / "geocoded_crashes.csv", index=False)

In [ ]:
# ---- Cells, exposure (network km) and smoothed rates --------------------------------------------------
# Crashes are assigned to GRID_KM x GRID_KM cells in ETRS89 / UTM 30N (EPSG:25830, metric; the Canary Islands are excluded because they lie in another zone).
# Exposure of a cell = number of distinct integer KP points of the layer inside it ≈ km of road. Cells below RMIN_KM of network are not analysed.
# Every cell with at least RMIN_KM of network is analysed, also the ones where no crash was located (their crash count is 0): leaving them out would bias
# the national rate, the gamma prior, Moran's I and the neighbourhoods of Gi* upwards.
CELL_M = GRID_KM * 1000.0
if HAS_PK:
    CANARY_PROVINCES = [35, 38]
    tr25830 = pyproj.Transformer.from_crs(4326, 25830, always_xy=True)
    net = pk[pk.x > -12].drop_duplicates(["road", "km"]).copy(); net["X"], net["Y"] = tr25830.transform(net.x.values, net.y.values)
    net["cx"], net["cy"] = np.floor(net.X / CELL_M).astype(int), np.floor(net.Y / CELL_M).astype(int)
    road_km = net.drop_duplicates(["road", "km", "cx", "cy"]).groupby(["cx", "cy"]).size().rename("road_km")
    gg = acc[acc.geo_ok & ~acc.COD_PROVINCIA.isin(CANARY_PROVINCES)].copy(); gg["X"], gg["Y"] = tr25830.transform(gg.geo_lon.values, gg.geo_lat.values)
    gg["cx"], gg["cy"] = np.floor(gg.X / CELL_M).astype(int), np.floor(gg.Y / CELL_M).astype(int)
    cell_y = gg.groupby(["cx", "cy", "ANYO"]).agg(crashes=("UID", "size"), fatal=("fatal30", "sum")).reset_index()
    CE = cell_y.merge(road_km.reset_index(), on=["cx", "cy"], how="left"); no_net = CE[CE.road_km.isna()].fatal.sum()
    print(f"Geocoded fatal crashes (excl. Canary Islands): {int(gg.fatal30.sum()):,} · in cells with no network km in the layer: {int(no_net)} (dropped)")
    CE = CE.dropna(subset=["road_km"]); NY = len(YL)

    def eb_gamma(F, E):
        """Gamma(a, b) prior for Poisson rates with exposure E by the method of moments (Clayton & Kaldor 1987):
        total variance of the observed rates minus the expected Poisson variance gives the between-cell variance."""
        m = F.sum() / E.sum(); lam = F / E; s2 = np.sum(E * (lam - m) ** 2) / E.sum(); t2 = max(s2 - m / E.mean(), 1e-12); return m * m / t2, m / t2

    def cell_table(years, rmin=RMIN_KM):
        """Cell-level counts, exposure (network-km x years), raw rate and empirical-Bayes (gamma-Poisson) smoothed rate for the given years.
        The table starts from ALL cells that have network in the layer (road_km) and fills with 0 the cells where no crash was located."""
        c = CE[CE.ANYO.isin(years)].groupby(["cx", "cy"]).agg(crashes=("crashes", "sum"), fatal=("fatal", "sum")).reset_index()
        c = road_km.reset_index().merge(c, on=["cx", "cy"], how="left").fillna({"crashes": 0, "fatal": 0})
        c = c[c.road_km >= rmin].copy(); c["E"] = c.road_km * len(years); a, b = eb_gamma(c.fatal.values, c.E.values)
        c["rate"] = c.fatal / c.E; c["eb"] = (c.fatal + a) / (c.E + b); return c, a, b
    ct, a_g, b_g = cell_table(YL)
    print(f"{GRID_KM}x{GRID_KM} km cells with >= {RMIN_KM} km of network (incl. {int((ct.crashes == 0).sum()):,} with no located crash): {len(ct):,} · fatal crashes per network-km per year (national): {ct.fatal.sum() / ct.E.sum():.4f} · gamma prior a={a_g:.2f}, b={b_g:.1f}")

In [ ]:
# ---- Spatial autocorrelation (Moran's I) and hot spots (Getis-Ord Gi*), with permutations and FDR ----
# Moran's I (row-standardised queen contiguity) tests whether similar smoothed rates cluster in space; the reference distribution is built by permuting values over cells.
# Gi* is the local statistic: the share of the total contained in a cell's neighbourhood (including itself). Its z-score is standardised against the permutation
# distribution, and the Benjamini-Hochberg procedure controls the false-discovery rate across the thousands of cells tested simultaneously.
import scipy.sparse as sp
from statsmodels.stats.multitest import multipletests

def queen_W(c, include_self=False):
    ix = {(x, y): i for i, (x, y) in enumerate(zip(c.cx, c.cy))}; r, k = [], []
    for (x, y), i in ix.items():
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                if (dx or dy or include_self) and (x + dx, y + dy) in ix: r.append(i); k.append(ix[(x + dx, y + dy)])
    return sp.csr_matrix((np.ones(len(r)), (r, k)), shape=(len(c), len(c)))

def moran_I(x, W, nperm=999, seed=SEED):
    rg = np.random.default_rng(seed); z = x - x.mean(); Wr = sp.diags(1 / np.maximum(W.sum(1).A1, 1)) @ W
    I = lambda zz: (len(zz) / Wr.sum()) * (zz @ (Wr @ zz)) / (zz @ zz)
    obs = I(z); sim = np.array([I(rg.permutation(z)) for _ in range(nperm)]); return obs, (1 + np.sum(sim >= obs)) / (nperm + 1), sim.mean()

def getis_ord(x, W, nperm=999, seed=SEED, q=0.05):
    rg = np.random.default_rng(seed); Ws = (W + sp.eye(W.shape[0])).tocsr(); tot = x.sum()
    obs = (Ws @ x) / tot; P = np.column_stack([rg.permutation(x) for _ in range(nperm)]); sim = (Ws @ P) / tot
    z = (obs - sim.mean(1)) / np.maximum(sim.std(1), 1e-12)          # pseudo-exact z: mean and SD under the permutation distribution
    p = 2 * stats.norm.sf(np.abs(z))                                    # with 999 permutations the smallest empirical p (0.002) makes FDR control impossible across thousands of cells, so the normal tail of the permutation z is used
    fdr = multipletests(p, alpha=q, method="fdr_bh")[1]; return z, p, fdr

if HAS_PK:
    Wc = queen_W(ct); Ic, pI, EI = moran_I(ct.eb.values, Wc)
    print(f"Global Moran's I (smoothed fatal crashes per network-km): I = {Ic:.3f} (expected under randomness ≈ {EI:.3f}) · permutation p = {pI:.3f}")
    ct["z"], ct["p"], ct["p_fdr"] = getis_ord(ct.eb.values, Wc)
    ct["class"] = np.where((ct.p_fdr < .05) & (ct.z > 0), "hot", np.where((ct.p_fdr < .05) & (ct.z < 0), "cold", "n.s."))
    print("Cells by class (Gi*, FDR 5 %):", ct["class"].value_counts().to_dict())
    TABLES["gi_cells"] = ct
    # Map: KP network in the background + significant cells + fatal crashes
    from matplotlib.collections import PatchCollection
    from matplotlib.patches import Rectangle
    fig, ax = plt.subplots(figsize=(9.5, 8)); ax.scatter(net.X / 1000, net.Y / 1000, s=.15, color=GRID, alpha=.9, rasterized=True, lw=0)
    for cl, colr in [("cold", DOWN), ("hot", UP)]:
        s = ct[ct["class"] == cl]
        if len(s): ax.add_collection(PatchCollection([Rectangle((x * GRID_KM, y * GRID_KM), GRID_KM, GRID_KM) for x, y in zip(s.cx, s.cy)], facecolor=colr, edgecolor="none", alpha=.55, label=f"{cl} (n={len(s)})"))
    gf = gg[gg.fatal30 == 1]; ax.scatter(gf.X / 1000, gf.Y / 1000, s=2.5, color=INK, alpha=.35, lw=0, label="fatal crash", rasterized=True)
    ax.set_aspect("equal"); ax.set_xlim(net.X.min() / 1000 - 20, net.X.max() / 1000 + 20); ax.set_ylim(net.Y.min() / 1000 - 20, net.Y.max() / 1000 + 20)
    ax.set_xlabel("X, UTM 30N (km)"); ax.set_ylabel("Y (km)"); ax.grid(False); ax.legend(fontsize=8, loc="lower left"); ax.set_title(f"Fatal crashes per network-km · Gi* ({GRID_KM} km cells, FDR 5 %) · excl. Canary Islands"); show(fig, "20_gi_map")

In [ ]:
# ---- Do hot spots persist? Detect on earlier years, evaluate on the next -------------------------------
# For each forward split, Gi* hot cells are identified using only the training years; the rate ratio of fatal crashes per network-km in hot vs. all other cells
# is then computed on the held-out year (exact Poisson interval). "Share in top 5 %" reports how many future fatal crashes fall in the 5 % of cells with the
# highest smoothed rate, against the share of road network those cells occupy ("chance").
if HAS_PK and MULTI:
    rows = []
    for k in range(1, len(YL)):
        tr_y, te_y = YL[:k], [YL[k]]; c, _, _ = cell_table(tr_y); W = queen_W(c); c["z"], c["p"], c["pf"] = getis_ord(c.eb.values, W, nperm=499)
        hot = c[(c.pf < .05) & (c.z > 0)]; te = CE[CE.ANYO.isin(te_y)].groupby(["cx", "cy"]).agg(fatal=("fatal", "sum")).reset_index()
        te = c[["cx", "cy", "road_km"]].merge(te, on=["cx", "cy"], how="left").fillna({"fatal": 0}); te["hot"] = te.set_index(["cx", "cy"]).index.isin(hot.set_index(["cx", "cy"]).index)
        x1, t1 = te[te.hot].fatal.sum(), te[te.hot].road_km.sum() * len(te_y); x0, t0 = te[~te.hot].fatal.sum(), te[~te.hot].road_km.sum() * len(te_y)
        rr, lo, hi, pv = poisson_rr(x1, t1, x0, t0) if len(hot) else (np.nan,) * 4
        top = c.nlargest(max(1, int(.05 * len(c))), "eb"); capt = te.merge(top[["cx", "cy"]], on=["cx", "cy"]).fatal.sum() / max(te.fatal.sum(), 1)
        rows.append(dict(training=f"{YL[0]}–{YL[k - 1]}" if k > 1 else str(YL[0]), test=YL[k], hot_cells=len(hot), RR_hot_vs_rest=rr, CI95_low=lo, CI95_high=hi,
                         pct_fatal_in_top5pct_cells=capt * 100, chance_pct=100 * top.road_km.sum() / c.road_km.sum()))
    pers = pd.DataFrame(rows); display(pers.round(2)); TABLES["cell_persistence"] = pers
    print("Reading: RR > 1 with a CI above 1 = the hot cells of the past still have more fatal crashes per network-km. 'Chance' = % of network km occupied by the top-5 % cells.")

In [ ]:
# ---- Optional: distance to the nearest hospital and fatality (Euclidean approximation) -----------------
if HAS_PK and HOSPITALS_CSV and Path(HOSPITALS_CSV).exists():
    import statsmodels.formula.api as smf
    h = pd.read_csv(HOSPITALS_CSV).rename(columns=lambda c: c.strip().lower()); hx, hy = tr25830.transform(h["lon"].values, h["lat"].values)
    dist, _ = cKDTree(np.c_[hx, hy]).query(np.c_[gg.X.values, gg.Y.values]); gg["d_hosp_km"] = dist / 1000
    m = smf.logit("fatal30 ~ I(d_hosp_km/10) + C(COD_PROVINCIA) + C(ANYO) + C(road_group)", data=gg).fit(disp=0)
    b, ci = m.params["I(d_hosp_km / 10)"], m.conf_int().loc["I(d_hosp_km / 10)"]
    print(f"OR of a fatal crash per +10 km to the nearest hospital (adjusted for province, year and road group): {np.exp(b):.3f} (95% CI {np.exp(ci[0]):.3f}–{np.exp(ci[1]):.3f})")
    print("Caution: Euclidean distance ≠ arrival time; '30-day death' mixes on-scene and in-hospital deaths; use network isochrones for the article.")
else:
    print("HOSPITALS_CSV = None → no hospital-distance analysis (CSV with lon, lat columns; e.g. from the National Hospital Catalogue).")

## 8 · Hierarchical Bayesian models with spatial smoothing (BYM2)

Section 4 used empirical-Bayes shrinkage province by province, treating each unit as exchangeable. That is a reasonable first step but ignores geography: neighbouring provinces share roads, climate, traffic mix, enforcement regimes and recording practices, so their risks are correlated. Here a **full hierarchical model** is fitted in which each province borrows strength from its **neighbours** (the spatially structured BYM2 component of Riebler et al. 2016) and from the whole country (the unstructured component), and in which **all uncertainty is propagated** to every reported quantity.

**Three models, one scaffold** (province *p*, year *t*):

| Model | Response | Likelihood | Linear predictor |
|---|---|---|---|
| **L · Severity** | fatal crashes out of casualty crashes, by province × area type (urban / interurban) × year | Binomial, logit link | α + β·interurban + γ<sub>t</sub> + **b<sub>p</sub>** + v<sub>p,area</sub> + δ<sub>p,t</sub> |
| **F · Frequency** | casualty crashes | Poisson-lognormal, offset = exposure | log E<sub>p,t</sub> + α + γ<sub>t</sub> + **b<sub>p</sub>** + δ<sub>p,t</sub> |
| **R · Risk** | 30-day deaths | Poisson-lognormal, offset = exposure | log E<sub>p,t</sub> + α + γ<sub>t</sub> + **b<sub>p</sub>** + δ<sub>p,t</sub> |

- **b<sub>p</sub> = σ·(√ρ·u\*<sub>p</sub> + √(1−ρ)·v<sub>p</sub>)**. Here *u\** is an intrinsic conditional autoregressive (ICAR) field **scaled** so that its geometric-mean marginal variance equals 1 — this is what makes σ interpretable and the prior on ρ meaningful — over the provincial land-contiguity graph; *v* is an i.i.d. standard-normal component; and ρ ∈ (0,1) is the **fraction of the province-level variation that is spatially structured**.
- **Priors.** σ ~ Exponential with P(σ > 1) = 0.01 (a *penalised-complexity* prior, Simpson et al. 2017, which shrinks towards "no province effect" unless the data demand otherwise); ρ ~ Beta(1, 1); regression coefficients weakly-informative Normal. The intercept is centred on the national rate. A prior-predictive check is shown before fitting.
- **δ<sub>p,t</sub>**, an observation-level random effect, absorbs over-dispersion (an alternative to the negative-binomial) and makes the Poisson models robust to clustering of deaths in multi-fatality crashes. The year effects γ<sub>t</sub> are constrained to sum to zero.
- **Islands.** Baleares, Ceuta and Melilla have no land neighbours, so they receive only the unstructured component; the Canary Islands form their own two-province component.
- **Estimands with a direct interpretation:** the **risk / odds ratio of each province against the weighted national mean**, the exceedance probability P(RR > 1), the **posterior rank** with its interval (ranks are far less certain than point estimates suggest), annual changes, and the frequency–severity correlation.

In [ ]:
# ---- Dependencies and provincial contiguity graph -----------------------------------------------------
try:
    import pymc as pm, pytensor.tensor as pt, arviz as az
    HAS_PYMC = True; print("PyMC", pm.__version__, "· ArviZ", az.__version__)
except ImportError:
    HAS_PYMC = False; print("PyMC/ArviZ missing → pip install pymc arviz. Sections 8 and 9 are skipped.")
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

# Land contiguity between provinces (INE codes). The list is symmetrised and validated below; it can be replaced by queen contiguity computed from PROV_GEO.
_ADJ = {1: [20, 48, 9, 26, 31], 2: [13, 16, 46, 3, 30, 23], 3: [46, 2, 30], 4: [18, 30], 5: [37, 47, 40, 28, 45, 10], 6: [10, 45, 13, 14, 41, 21],
        8: [17, 25, 43], 9: [1, 48, 39, 34, 47, 40, 42, 26], 10: [6, 45, 5, 37], 11: [21, 41, 29], 12: [46, 44, 43], 13: [45, 16, 2, 23, 14, 6],
        14: [41, 29, 18, 23, 13, 6], 15: [27, 36], 16: [19, 44, 46, 2, 13, 45, 28], 17: [8, 25], 18: [4, 30, 23, 14, 29], 19: [28, 40, 42, 50, 44, 16],
        20: [31, 1, 48], 21: [6, 41, 11], 22: [50, 25, 31], 23: [13, 2, 18, 14], 24: [33, 34, 47, 49, 32, 27, 39], 25: [22, 43, 8, 17, 50],
        26: [31, 1, 9, 42, 50], 27: [15, 36, 32, 24, 33], 28: [19, 40, 5, 45, 16], 29: [11, 41, 14, 18], 30: [3, 2, 18, 4], 31: [20, 1, 26, 50, 22],
        32: [36, 27, 24, 49], 33: [27, 24, 39], 34: [39, 9, 47, 24], 35: [38], 36: [15, 27, 32], 37: [49, 47, 5, 10], 39: [33, 24, 34, 9, 48],
        40: [47, 9, 42, 19, 28, 5], 41: [21, 6, 14, 29, 11], 42: [9, 26, 50, 19, 40], 43: [25, 8, 50, 44, 12], 44: [50, 19, 16, 46, 12, 43],
        45: [28, 5, 10, 6, 13, 16], 46: [12, 44, 16, 2, 3], 47: [34, 9, 40, 5, 37, 49, 24], 48: [39, 9, 1, 20], 49: [24, 47, 37, 32],
        50: [22, 31, 26, 42, 19, 44, 43, 25]}

def build_graph(provs, adj=_ADJ):
    """Symmetric adjacency matrix, connected components, and for each component (>= 2 nodes) its edge list and ICAR scaling constant:
    the geometric mean of the marginal variances of the generalised inverse of the structure matrix Q = D - W (Riebler et al. 2016)."""
    idx = {p: i for i, p in enumerate(provs)}; N = len(provs)
    E = sorted({(min(idx[a], idx[b]), max(idx[a], idx[b])) for a, nb in adj.items() for b in nb if a in idx and b in idx and a != b})
    W = np.zeros((N, N))
    for i, j in E: W[i, j] = W[j, i] = 1
    ncomp, lab = connected_components(sp.csr_matrix(W), directed=False); comps = []
    for c in range(ncomp):
        nodes = np.where(lab == c)[0]
        if len(nodes) < 2: continue
        Wc = W[np.ix_(nodes, nodes)]; Q = np.diag(Wc.sum(1)) - Wc; m = len(nodes); J = np.ones((m, m)) / m
        S = np.linalg.inv(Q + J) - J; scale = float(np.exp(np.mean(np.log(np.diag(S)))))                 # geometric mean of marginal variances
        loc = {n: k for k, n in enumerate(nodes)}; ed = np.array([(loc[i], loc[j]) for i, j in E if i in loc and j in loc])
        comps.append(dict(nodes=nodes, edges=ed, scale=scale))
    return dict(N=N, W=W, edges=np.array(E), comps=comps, lab=lab, provs=list(provs))

PROVS = sorted(int(p) for p in acc.COD_PROVINCIA.unique()); PNAME = acc.drop_duplicates("COD_PROVINCIA").set_index("COD_PROVINCIA").province.to_dict()
GR = build_graph(PROVS); deg = GR["W"].sum(1)
print(f"Graph: {GR['N']} provinces · {len(GR['edges'])} edges · components with >=2 nodes: {[len(c['nodes']) for c in GR['comps']]} · without neighbours: {[PNAME[PROVS[i]] for i in np.where(deg == 0)[0]]}")
# Validation of the graph against the data: contiguous provinces must have nearby centres (detects typos in the neighbour list)
if "geo_lon" in acc and acc.geo_ok.any():
    cen = acc[acc.geo_ok].groupby("COD_PROVINCIA")[["geo_lon", "geo_lat"]].median()
    dd = [(PNAME[PROVS[i]], PNAME[PROVS[j]], hav_km(*cen.loc[PROVS[i]], *cen.loc[PROVS[j]])) for i, j in GR["edges"] if PROVS[i] in cen.index and PROVS[j] in cen.index]
    mx = sorted(dd, key=lambda t: -t[2])[:3]; print("Longest edges between provincial centres (km):", [(a, b, round(d)) for a, b, d in mx], "· if any exceeds ~350 km, review _ADJ")

In [ ]:
# ---- Model building blocks (scaled BYM2) and summary utilities ----------------------------------------
def bym2(name, G, U=1.0, spatial=True, rho_prior=(1, 1)):
    """Province effect b = sigma * (sqrt(rho) * u* + sqrt(1 - rho) * v). Returns (b, sigma, rho).
    - sigma ~ Exponential(rate = -log(0.01)/U): PC prior with P(sigma > U) = 0.01.
    - u: ICAR with exact sum-to-zero per connected component (ZeroSumNormal) and density -0.5 * sum over edges of (u_i - u_j)^2, rescaled by the scaling constant.
    - rho only acts where a neighbourhood exists (islands: b = sigma * v)."""
    N = G["N"]; sigma = pm.Exponential(f"{name}_sigma", -np.log(0.01) / U); v = pm.Normal(f"{name}_v", 0, 1, shape=N)
    if not spatial or not G["comps"]:
        return pm.Deterministic(name, sigma * v, dims="prov"), sigma, None
    rho = pm.Beta(f"{name}_rho", *rho_prior); u_full = pt.zeros(N); mask = np.zeros(N)
    for c, comp in enumerate(G["comps"]):
        nodes, ed = comp["nodes"], comp["edges"]; u = pm.ZeroSumNormal(f"{name}_u{c}", sigma=5, shape=len(nodes))    # exact sum-to-zero per component
        pm.Potential(f"{name}_icar{c}", -0.5 * pt.sum((u[ed[:, 0]] - u[ed[:, 1]]) ** 2))                 # ICAR: density on neighbour differences
        u_full = pt.set_subtensor(u_full[nodes], u / np.sqrt(comp["scale"])); mask[nodes] = 1
    w_u = mask * pt.sqrt(rho); w_v = mask * pt.sqrt(1 - rho) + (1 - mask)                              # rho only acts where there is a neighbourhood
    return pm.Deterministic(name, sigma * (w_u * u_full + w_v * v), dims="prov"), sigma, rho

def draws(idata, name):
    """Posterior draws of a variable as an array (chains * draws, ...)."""
    a = idata.posterior[name].values; return a.reshape(-1, *a.shape[2:])

def diag_table(idata, name):
    """MCMC diagnostics: worst R-hat, smallest bulk/tail effective sample size (and which parameter), and number of divergent transitions."""
    summ = az.summary(idata, kind="diagnostics"); div = int(idata.sample_stats["diverging"].sum()); worst = summ["ess_bulk"].idxmin()
    return dict(model=name, rhat_max=float(summ["r_hat"].max()), ess_bulk_min=float(summ["ess_bulk"].min()), parameter_min_ess=worst, ess_tail_min=float(summ["ess_tail"].min()), divergences=div, parameters=len(summ))

def prov_effects(idata, name, weights, ref="weighted mean"):
    """Rate (or odds) ratio of each province against the weighted national mean, with exceedance probability P(RR > 1) and posterior rank.
    The reference is the weighted mean of the log effects (weights = cases or exposure), so the RR answers 'how does this province compare with a typical case in Spain?'."""
    b = draws(idata, name); w = np.asarray(weights, float) / np.sum(weights); rel = b - (b * w).sum(1, keepdims=True)
    rk = (-rel).argsort(1).argsort(1) + 1                                                                   # rank 1 = largest effect
    return pd.DataFrame({"COD_PROVINCIA": PROVS, "province": [PNAME[p] for p in PROVS], "RR": np.exp(np.median(rel, 0)), "lo": np.exp(np.percentile(rel, 2.5, 0)), "hi": np.exp(np.percentile(rel, 97.5, 0)),
                         "P_RR>1": (rel > 0).mean(0), "rank_median": np.median(rk, 0), "rank_lo": np.percentile(rk, 2.5, 0), "rank_hi": np.percentile(rk, 97.5, 0), "P_top10": (rk <= 10).mean(0)})

def caterpillar_post(t, title, xlabel, name):
    s = t.sort_values("RR").reset_index(drop=True); col = np.where(s.lo > 1, UP, np.where(s.hi < 1, DOWN, NEUTRAL)); yy = np.arange(len(s))
    fig, ax = plt.subplots(figsize=(8, 0.22 * len(s) + 1.6)); ax.hlines(yy, s.lo, s.hi, color=col, lw=2.2); ax.scatter(s.RR, yy, c=col, s=34, zorder=3, ec=SURFACE, lw=1.2)
    ax.axvline(1, color=INK2, ls="--", lw=1); ax.set_xscale("log"); ax.set_yticks(yy); ax.set_yticklabels(s.province, fontsize=8); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel(xlabel); ax.set_title(title); show(fig, name)

In [ ]:
# ---- Data and model specification ---------------------------------------------------------------------
try:
    import nutpie; SAMPLER = "nutpie" if BAYES_SAMPLER in ("auto", "nutpie") else "pymc"
except ImportError:
    SAMPLER = "pymc"
print("Sampler:", SAMPLER, "" if SAMPLER == "nutpie" else "(install nutpie for a 5-10x speed-up)")
BAY = dict(draws=BAYES_DRAWS, tune=BAYES_TUNE, chains=BAYES_CHAINS, target_accept=BAYES_TARGET_ACCEPT, random_seed=SEED, progressbar=False, cores=min(BAYES_CHAINS, os.cpu_count() or 1), nuts_sampler=SAMPLER)
Y = len(YL); pidx = {p: i for i, p in enumerate(PROVS)}; yidx = {y: i for i, y in enumerate(YL)}
COORDS = {"prov": [PNAME[p] for p in PROVS], "year": YL}

def data_L(outcome="fatal30", areas=(1, 2)):
    """Severity data: casualty crashes (n) and fatal crashes (x) by province x year x area type (z = 1 for interurban)."""
    d = acc[acc.ZONA_AGRUPADA.isin(areas)].groupby(["COD_PROVINCIA", "ANYO", "ZONA_AGRUPADA"]).agg(n=("UID", "size"), x=(outcome, "sum")).reset_index()
    return d.assign(p=d.COD_PROVINCIA.map(pidx), t=d.ANYO.map(yidx), z=(d.ZONA_AGRUPADA == 1).astype(int))

def build_L(d, G=GR, U=1.0, spatial=True, rho_prior=(1, 1), w=None):
    """Model L (severity). `w` are optional 0/1 observation weights used by the province-grouped cross-validation to hold provinces out."""
    with pm.Model(coords={**COORDS, "cell": np.arange(len(d))}) as m:
        a = pm.Normal("alpha", -3.0, 1.5); bz = pm.Normal("beta_interurban", 0, 2.5)
        g = pm.ZeroSumNormal("gamma", sigma=0.5, dims="year") if Y > 1 else 0.0
        b, sb, rho = bym2("b", G, U, spatial, rho_prior)
        sv = pm.Exponential("sigma_pa", -np.log(0.01) / U); vpz = pm.Normal("v_pa", 0, 1, shape=(G["N"], 2))     # province x area-type interaction
        sd = pm.Exponential("sigma_py", -np.log(0.01) / U); dpy = pm.Normal("d_py", 0, 1, shape=(G["N"], Y))     # province x year (observation-level) effect
        eta = a + bz * d.z.values + (g[d.t.values] if Y > 1 else 0.0) + b[d.p.values] + sv * vpz[d.p.values, d.z.values] + sd * dpy[d.p.values, d.t.values]
        pm.Deterministic("eta", eta, dims="cell")
        if w is None: pm.Binomial("x", n=d.n.values, logit_p=eta, observed=d.x.values, dims="cell")
        else: pm.Potential("ll", pt.sum(w * pm.logp(pm.Binomial.dist(n=d.n.values, logit_p=eta), d.x.values)))
    return m

def data_P(resp):
    """Rate data for models F ('crashes') and R ('deaths'): the response, plus exposure E in millions of inhabitant-years."""
    d = py.dropna(subset=["POP"]).copy()
    d["E"] = d.POP / 1e6
    return d.assign(p=d.COD_PROVINCIA.map(pidx), t=d.ANYO.map(yidx), y=d[resp])

def build_P(d, G=GR, U=1.0, spatial=True, rho_prior=(1, 1)):
    """Models F and R: Poisson-lognormal with exposure offset."""
    with pm.Model(coords={**COORDS, "cell": np.arange(len(d))}) as m:
        a = pm.Normal("alpha", float(np.log(d.y.sum() / d.E.sum())), 1.0)
        g = pm.ZeroSumNormal("gamma", sigma=0.5, dims="year") if Y > 1 else 0.0
        b, sb, rho = bym2("b", G, U, spatial, rho_prior)
        sd = pm.Exponential("sigma_py", -np.log(0.01) / U); dpy = pm.Normal("d_py", 0, 1, shape=(G["N"], Y))
        eta = np.log(d.E.values) + a + (g[d.t.values] if Y > 1 else 0.0) + b[d.p.values] + sd * dpy[d.p.values, d.t.values]
        pm.Poisson("x", mu=pt.exp(eta), observed=d.y.values, dims="cell")
    return m

def fit(m, name):
    import time; t0 = time.time()
    with m: idata = pm.sample(**BAY, idata_kwargs={"log_likelihood": True})
    print(f"  {name}: {time.time() - t0:.0f} s")
    return idata

In [ ]:
# ---- Prior-predictive check (do the priors imply absurd values?) and fit of the severity model (L) ----
RES = {}
if HAS_PYMC and RUN_BAYES:
    dL = data_L(); mL = build_L(dL)
    with mL: pri = pm.sample_prior_predictive(300, random_seed=SEED)
    pl = (pri.prior_predictive["x"].values.reshape(-1, len(dL)) / dL.n.values); wmean = (pl * dL.n.values).sum(1) / dL.n.sum()
    print(f"Prior predictive · implied national case fatality: median {np.median(wmean):.1%}, 90% [{np.percentile(wmean, 5):.1%}, {np.percentile(wmean, 95):.1%}]  (observed: {dL.x.sum() / dL.n.sum():.1%}) "
          "→ the prior is wide but not absurd if the observed value falls inside and the range does not reach ~100 %.")
    idL = fit(mL, "L · severity (BYM2)"); RES["L"] = idL; DIAG = [diag_table(idL, "L · severity (BYM2)")]

In [ ]:
# ---- Frequency (F) and risk (R) per inhabitant --------------------------------------------
EXP_LAB = "million inhabitant-years"
if HAS_PYMC and RUN_BAYES:
    if HAS_POP:
        dF, dR = data_P("crashes"), data_P("deaths"); mF, mR = build_P(dF), build_P(dR)
        RES["F"] = fit(mF, "F · frequency (BYM2)"); DIAG.append(diag_table(RES["F"], "F · frequency (BYM2)"))
        RES["R"] = fit(mR, "R · risk (BYM2)"); DIAG.append(diag_table(RES["R"], "R · risk (BYM2)"))
    else:
        print("Population is not available for every province and year → only the severity model L is fitted.")
    dg = pd.DataFrame(DIAG); display(dg.round(3)); TABLES["bayes_diagnostics"] = dg
    if (dg.rhat_max > 1.01).any() or (dg.divergences > 0).any() or (dg.ess_bulk_min < 200).any():
        print("⚠️ Doubtful convergence (R-hat > 1.01, ESS < 200 or divergences): see 'parameter_min_ess' (almost always rho, which is weakly identified: R-hat of 1.01-1.02 on rho is tolerable if the province effects and sigma are well behaved). Otherwise increase BAYES_DRAWS/BAYES_TUNE or BAYES_TARGET_ACCEPT.")
    else:
        print("✓ Convergence adequate in all models (R-hat <= 1.01, ESS >= 200 with 4 chains, no divergences). The parameter with the lowest ESS is usually rho (weakly identified: this is information, not a failure).")

In [ ]:
# ---- Results: the effect of each province with full uncertainty ----------------------------------------
if HAS_PYMC and RUN_BAYES:
    tabs = {}
    wL = dL.groupby("p").n.sum().reindex(range(len(PROVS))).fillna(0).values
    tabs["L"] = prov_effects(RES["L"], "b", wL); caterpillar_post(tabs["L"], "Severity · odds of a fatal crash by province vs. national average (BYM2)", "Odds ratio vs. the average (95% credible interval, log scale)", "21_posterior_severity")
    for k, lab in [("F", "frequency of casualty crashes"), ("R", "risk of dying")]:
        if k in RES:
            dd_ = dF if k == "F" else dR; w = dd_.groupby("p").E.sum().reindex(range(len(PROVS))).fillna(0).values; tabs[k] = prov_effects(RES[k], "b", w)
            caterpillar_post(tabs[k], f"{lab.capitalize()} per {EXP_LAB} · rate ratio by province (BYM2)", "Rate ratio vs. the average (95% credible interval, log scale)", f"22_posterior_{lab.split()[0]}")
    for k, t in tabs.items(): TABLES[f"bayes_province_{k}"] = t
    show_cols = ["province", "RR", "lo", "hi", "P_RR>1", "rank_median", "rank_lo", "rank_hi", "P_top10"]
    for k in tabs: print(f"\nModel {k}: 8 provinces with the largest effect"); display(tabs[k].sort_values("RR", ascending=False).head(8)[show_cols].round(3))
    # Hyper-parameters: how much of the variation is spatial?
    hp = az.summary(RES["L"], var_names=["alpha", "beta_interurban", "b_sigma", "b_rho", "sigma_pa", "sigma_py"] + (["gamma"] if Y > 1 else []), ci_prob=.95)[["mean", "sd", "eti95_lb", "eti95_ub", "r_hat"]]
    display(hp.round(3)); TABLES["bayes_hyperparameters_L"] = hp
    if "R" in RES:
        hpR = az.summary(RES["R"], var_names=["alpha", "b_sigma", "b_rho", "sigma_py"] + (["gamma"] if Y > 1 else []), ci_prob=.95)[["mean", "sd", "eti95_lb", "eti95_ub", "r_hat"]]; display(hpR.round(3)); TABLES["bayes_hyperparameters_R"] = hpR
        print(f"Spatially structured fraction of the province effect (rho): severity {hp.loc['b_rho', 'mean']:.2f} [{hp.loc['b_rho', 'eti95_lb']:.2f}, {hp.loc['b_rho', 'eti95_ub']:.2f}] · risk {hpR.loc['b_rho', 'mean']:.2f} [{hpR.loc['b_rho', 'eti95_lb']:.2f}, {hpR.loc['b_rho', 'eti95_ub']:.2f}]. "
              "rho close to 0 = idiosyncratic differences; close to 1 = continuous geographic pattern.")

In [ ]:
# ---- Annual change and the central question: dangerous because of many crashes, or because they are severe? ----
if HAS_PYMC and RUN_BAYES:
    if Y > 1:
        rows = []
        for k, lab in [("L", "odds of a fatal crash"), ("F", "casualty crashes (rate)"), ("R", "deaths (rate)")]:
            if k not in RES: continue
            g = draws(RES[k], "gamma")
            for t in range(1, Y): rows.append(dict(measure=lab, year=YL[t], vs=YL[0], RR=np.exp(np.median(g[:, t] - g[:, 0])), lo=np.exp(np.percentile(g[:, t] - g[:, 0], 2.5)), hi=np.exp(np.percentile(g[:, t] - g[:, 0], 97.5)),
                                                    P_lower_than_base=(g[:, t] < g[:, 0]).mean(), P_lower_than_previous=(g[:, t] < g[:, t - 1]).mean()))
        yt = pd.DataFrame(rows); display(yt.round(3)); TABLES["bayes_year_effects"] = yt
        print("Annual effect adjusted for province (and area type in L). With only four years a trend must not be extrapolated: these are 'changes between years', not a slope.")
    if "F" in RES and "L" in RES:
        bF, bL = draws(RES["F"], "b"), draws(RES["L"], "b"); S = min(len(bF), len(bL)); rg = np.random.default_rng(SEED)
        cors = np.array([np.corrcoef(bF[i], bL[j])[0, 1] for i, j in zip(rg.integers(0, len(bF), 2000), rg.integers(0, len(bL), 2000))])
        print(f"Posterior correlation between the province effect of frequency and that of severity: median {np.median(cors):.2f} [95% CI {np.percentile(cors, 2.5):.2f}, {np.percentile(cors, 97.5):.2f}] "
              "(the models are independent: the correlation is computed by pairing random draws, which integrates the uncertainty of both).")
        mf, ml = tabs["F"].set_index("COD_PROVINCIA"), tabs["L"].set_index("COD_PROVINCIA"); fig, ax = plt.subplots(figsize=(7.2, 6))
        ax.errorbar(mf.RR, ml.RR.reindex(mf.index), xerr=[mf.RR - mf.lo, mf.hi - mf.RR], yerr=[ml.RR.reindex(mf.index) - ml.lo.reindex(mf.index), ml.hi.reindex(mf.index) - ml.RR.reindex(mf.index)], fmt="none", ecolor=GRID, lw=1, zorder=1)
        ax.scatter(mf.RR, ml.RR.reindex(mf.index), s=40, color=SERIES[0], ec=SURFACE, lw=1, zorder=2)
        for i, r in pd.concat([mf.nlargest(5, "RR"), mf.nsmallest(5, "RR")]).iterrows(): ax.annotate(r.province, (r.RR, ml.RR.reindex(mf.index)[i]), xytext=(4, 3), textcoords="offset points", fontsize=8, color=INK2)
        ax.axhline(1, color=INK2, lw=1); ax.axvline(1, color=INK2, lw=1); ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(True)
        ax.set_xlabel("Crash frequency: rate ratio (BYM2)"); ax.set_ylabel("Severity: odds ratio of a fatal crash (BYM2)"); ax.set_title("Frequency vs. severity by province, with posterior uncertainty"); show(fig, "23_posterior_frequency_vs_severity")

In [ ]:
# ---- Model checking: mixed predictive check and comparison with the non-spatial model ------------------
if HAS_PYMC and RUN_BAYES:
    def ppc_mixed(idata, d, name, kind):
        """*Mixed* predictive check (Marshall & Spiegelhalter 2007): alpha, gamma, b and the sigmas of each posterior draw are retained, but the cell-level effects
        (v_pa, delta_py) are re-simulated from their hyper-prior. With an observation-level random effect the in-sample posterior predictive is trivially good;
        this check tests the hierarchical level instead. Reports interval coverage and a Bayesian p-value for a dispersion statistic."""
        rgp = np.random.default_rng(SEED); S = 1500; s = rgp.integers(0, draws(idata, "alpha").shape[0], S)
        al, ga, b = draws(idata, "alpha")[s], (draws(idata, "gamma")[s] if Y > 1 else np.zeros((S, 1))), draws(idata, "b")[s]; sd_ = draws(idata, "sigma_py")[s]
        eta = al[:, None] + (ga[:, d.t.values] if Y > 1 else 0.0) + b[:, d.p.values] + sd_[:, None] * rgp.normal(size=(S, len(d)))
        if kind == "L":
            eta = eta + draws(idata, "beta_interurban")[s][:, None] * d.z.values + draws(idata, "sigma_pa")[s][:, None] * rgp.normal(size=(S, len(d)))
            yr = rgp.binomial(d.n.values, 1 / (1 + np.exp(-eta))); y = d.x.values
        else:
            yr = rgp.poisson(np.exp(eta + np.log(d.E.values))); y = d.y.values
        mu, sv = yr.mean(0), yr.std(0) + 1e-9; T_obs = (((y - mu) / sv) ** 2).sum(); T_rep = (((yr - mu) / sv) ** 2).sum(1)
        lo50, hi50 = np.percentile(yr, [25, 75], 0); lo90, hi90 = np.percentile(yr, [5, 95], 0)
        return dict(model=name, coverage_50=np.mean((y >= lo50) & (y <= hi50)), coverage_90=np.mean((y >= lo90) & (y <= hi90)), bayes_p_dispersion=(T_rep >= T_obs).mean())
    pc = [ppc_mixed(RES["L"], dL, "L", "L")]
    if "R" in RES: pc += [ppc_mixed(RES["F"], dF, "F", "P"), ppc_mixed(RES["R"], dR, "R", "P")]
    pc = pd.DataFrame(pc); display(pc.round(3)); TABLES["bayes_ppc"] = pc
    print("Reading (mixed predictive): nominal coverage is 0.50 and 0.90; with discrete data it can be somewhat higher. A dispersion p between 0.05 and 0.95 means the model reproduces the variability among cells; "
          "≈ 0 = there are more extreme data than the model generates (missing structure); ≈ 1 = over-dispersed model.")
    # Severity model without spatial structure (b = sigma * v) and LOO comparison
    mL0 = build_L(dL, spatial=False); idL0 = fit(mL0, "L · no spatial structure"); RES["L0"] = idL0
    try:
        cmp = az.compare({"BYM2": RES["L"], "iid (no neighbourhood)": idL0}); display(cmp.round(3)); TABLES["bayes_loo_L"] = cmp
        pk_ = az.loo(RES["L"], pointwise=True).pareto_k.values; print(f"LOO · Pareto-k values > 0.7 in BYM2: {(pk_ > .7).sum()} of {len(pk_)} cells (if there are many, cell-wise LOO is unreliable; use the province-grouped validation below).")
    except Exception as e:
        print("LOO could not be computed:", type(e).__name__, e)
    print("Note: cell-wise LOO leaves out one observation but keeps the rest of the province, so it does NOT measure the usefulness of borrowing strength from neighbours. For that, use the province-grouped validation (RUN_KFOLD).")

In [ ]:
# ---- Spatial validation: predicting UNSEEN provinces (province-grouped K-fold). Costs minutes: RUN_KFOLD = True ----
# Whole provinces are removed from the likelihood (weights = 0) and their cell predictions are scored by the log predictive density. The BYM2 model can
# predict a held-out province from its neighbours; the iid model can only predict it from the national mean. A positive difference favours borrowing from neighbours.
if HAS_PYMC and RUN_BAYES and RUN_KFOLD and SAMPLER != "nutpie":
    print("RUN_KFOLD with PyMC's own sampler would take hours: install nutpie (pip install nutpie) or reduce KFOLD_K.")
elif HAS_PYMC and RUN_BAYES and RUN_KFOLD:
    from scipy.special import logsumexp
    from scipy.stats import binom
    rgk = np.random.default_rng(SEED); perm = rgk.permutation(len(PROVS)); folds = np.array_split(perm, KFOLD_K); contrib = {"BYM2": [], "iid": []}
    for f, held in enumerate(folds):
        mask = ~dL.p.isin(held).values; w = mask.astype(float)
        for nm, sp_ in [("BYM2", True), ("iid", False)]:
            mk = build_L(dL, spatial=sp_, w=w)
            with mk: idk = pm.sample(**{**BAY, "draws": max(BAYES_DRAWS // 2, 300), "tune": max(BAYES_TUNE // 2, 500)}, idata_kwargs={"log_likelihood": False})
            eta = draws(idk, "eta")[:, ~mask]; ll = binom.logpmf(dL.x.values[~mask][None, :], dL.n.values[~mask][None, :], 1 / (1 + np.exp(-eta)))
            elpd_i = logsumexp(ll, axis=0) - np.log(ll.shape[0]); contrib[nm].append(elpd_i)
        print(f"  fold {f + 1}/{KFOLD_K}: {len(held)} provinces held out")
    e1, e0 = np.concatenate(contrib["BYM2"]), np.concatenate(contrib["iid"]); dlt = e1 - e0
    print(f"Out-of-sample elpd (unseen provinces) · BYM2: {e1.sum():.1f} · iid: {e0.sum():.1f} · difference: {dlt.sum():.1f} ± {np.sqrt(len(dlt)) * dlt.std():.1f} (> 0 favours borrowing information from neighbours)")
    TABLES["bayes_kfold_provinces"] = pd.DataFrame({"model": ["BYM2", "iid"], "elpd": [e1.sum(), e0.sum()]})
else:
    print("RUN_KFOLD = False (or no PyMC) → validation on unseen provinces is skipped.")

## 9 · Robustness: do the conclusions change if the analyst's decisions change?

A publishable result must survive reasonable alternative choices. The severity model is refitted varying (a) the scale of the prior on the standard deviations (U = 0.5 and U = 2), (b) the prior on ρ (Beta(0.5, 0.5), which pushes towards the extremes), and (c) the definition of the outcome (fatal within **24 h** instead of 30 days). The ranking and magnitude of the province effects are compared with the main model through the Spearman rank correlation and the maximum absolute difference in log odds ratio.

In [ ]:
if HAS_PYMC and RUN_BAYES and RUN_SENS:
    base = draws(RES["L"], "b").mean(0); rows = [dict(variant="main (U=1, rho~Beta(1,1), 30 d)", spearman_vs_main=1.0, max_abs_diff_logOR=0.0, sigma_b=float(draws(RES["L"], "b_sigma").mean()), rho=float(draws(RES["L"], "b_rho").mean()))]
    VARIANTS = [("stricter sigma prior (U=0.5)", dict(U=0.5), "fatal30"), ("looser sigma prior (U=2)", dict(U=2.0), "fatal30"),
                ("rho ~ Beta(0.5, 0.5)", dict(rho_prior=(.5, .5)), "fatal30"), ("outcome: fatal within 24 h", dict(), "fatal24")]
    for nm, kw, outc in VARIANTS:
        d_ = data_L(outc); idv = fit(build_L(d_, **kw), nm); bv = draws(idv, "b").mean(0)
        rows.append(dict(variant=nm, spearman_vs_main=stats.spearmanr(base, bv).statistic, max_abs_diff_logOR=float(np.abs(base - bv).max()), sigma_b=float(draws(idv, "b_sigma").mean()), rho=float(draws(idv, "b_rho").mean())))
    sens = pd.DataFrame(rows); display(sens.round(3)); TABLES["bayes_sensitivity_L"] = sens
    print("Reading: Spearman ≈ 1 and small maximum differences (≪ the range of effects) = conclusions are robust to these decisions.")
else:
    print("RUN_SENS = False (or no PyMC) → the sensitivity analysis is skipped.")

## 10 · Predictive model of fatal crashes

**Purpose.** Given a casualty crash, was it fatal within 30 days? The model is *not* intended for operational prediction; it serves to understand which recorded circumstances discriminate fatal from non-fatal crashes and as a **machine-learning baseline** against which the structured Bayesian models of Section 8 can be compared.

- **No information leakage.** Every `TOTAL_*` / `TOT_*` casualty counter is excluded because it describes the outcome. `TOTAL_VEHICULOS` (vehicles involved) is a legitimate pre-outcome descriptor.
- **Honest validation.** (a) *Forward-in-time*: train on earlier years, evaluate on the next. (b) *Spatial*: whole provinces are held out (`GroupKFold`), and the province identifier itself is dropped, so the model must generalise to unseen territory.
- **Metrics.** AUC and average precision (AP; the base rate is only ~1.6 %, so AP relative to the base rate is more informative than AUC), the Brier skill score (BSS) against a constant-rate forecast, and **calibration** by deciles of predicted probability.
- With ~1,650 fatal crashes a year, tabular gradient boosting is more than adequate; deep networks are not justified by these data. Permutation importance describes what the model *uses*, not what *causes* fatalities.

In [ ]:
if RUN_ML:
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
    from sklearn.model_selection import GroupKFold
    from sklearn.inspection import permutation_importance

    NUM = [c for c in ["MES", "DIA_SEMANA", "HORA", "KM", "TOTAL_VEHICULOS"] if c in acc]
    CAT = [c for c in ["COD_PROVINCIA", "ZONA", "TITULARIDAD_VIA", "TIPO_VIA", "TIPO_ACCIDENTE", "NUDO", "NUDO_INFO", "SENTIDO_1F", "CONDICION_NIVEL_CIRCULA",
                       "CONDICION_FIRME", "CONDICION_ILUMINACION", "CONDICION_METEO", "CONDICION_NIEBLA", "CONDICION_VIENTO", "VISIB_RESTRINGIDA_POR",
                       "ACERA", "TRAZADO_PLANTA", "ISLA"] + [c for c in acc.columns if c.startswith("PRIORI_")] if c in acc]
    X = acc[NUM + CAT].copy()
    for c in CAT: X[c] = acc[c].astype("category")      # numeric category codes; NaN is handled natively as a missing value
    y = acc["fatal30"].to_numpy()
    print(f"{X.shape[1]} predictors ({len(NUM)} numeric, {len(CAT)} categorical) · base rate of fatal crashes = {y.mean() * 100:.2f}%")

    def make_clf():
        """Histogram-based gradient boosting with strong regularisation (small leaves, L2, early stopping on an internal 10 % validation split)."""
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, max_leaf_nodes=24, min_samples_leaf=80, l2_regularization=1.0,
                                              categorical_features="from_dtype", early_stopping=True, validation_fraction=0.1, n_iter_no_change=25, random_state=SEED)
    def metrics(yt, p, base_train):
        """Discrimination (AUC, AP, AP relative to chance) and probabilistic accuracy (Brier, Brier skill score vs. the training base rate)."""
        b = brier_score_loss(yt, p); b0 = brier_score_loss(yt, np.full(len(yt), base_train))
        return dict(n=len(yt), fatal_crashes=int(yt.sum()), AUC=roc_auc_score(yt, p), AP=average_precision_score(yt, p), AP_vs_chance=average_precision_score(yt, p) / yt.mean(), Brier=b, BSS=1 - b / b0)

    # (a) Forward-in-time validation: expanding training window, next year as the test set
    fw = pd.Series(np.nan, index=acc.index); models = {}; rows = []
    if MULTI:
        for t in YL[1:]:
            tr, te = (acc.ANYO < t).to_numpy(), (acc.ANYO == t).to_numpy()
            clf = make_clf().fit(X[tr], y[tr]); p = clf.predict_proba(X[te])[:, 1]; fw[te] = p; models[t] = clf
            rows.append(dict(train=f"{YL[0]}–{t - 1}", test_year=t, **metrics(y[te], p, y[tr].mean())))
        temporal = pd.DataFrame(rows).set_index("test_year"); display(temporal.round(3)); TABLES["model_temporal"] = temporal

    # (b) Spatial validation: whole provinces held out of training (province identifier not used as a predictor)
    Xs = X.drop(columns=["COD_PROVINCIA"]); oof = np.zeros(len(acc)); rows = []
    for i, (tr, te) in enumerate(GroupKFold(n_splits=4).split(Xs, y, groups=acc.COD_PROVINCIA)):
        clf = make_clf().fit(Xs.iloc[tr], y[tr]); oof[te] = clf.predict_proba(Xs.iloc[te])[:, 1]
        rows.append(dict(fold=i + 1, test_provinces=acc.COD_PROVINCIA.iloc[te].nunique(), **metrics(y[te], oof[te], y[tr].mean())))
    spatial = pd.DataFrame(rows).set_index("fold"); display(spatial.round(3)); TABLES["model_spatial"] = spatial
    print(f"Overall spatial AUC (out-of-fold): {roc_auc_score(y, oof):.3f} · AP: {average_precision_score(y, oof):.3f}")

In [ ]:
if RUN_ML:
    # ---- Calibration (forward-in-time predictions) and permutation importance ----------------------------
    # Calibration compares mean predicted probability and observed frequency within deciles of the predicted risk; Wilson bands show sampling uncertainty.
    if MULTI:
        m = fw.notna().to_numpy(); d = pd.DataFrame({"p": fw[m].to_numpy(), "y": y[m]}); d["bin"] = pd.qcut(d.p, 10, duplicates="drop")
        cal_t = d.groupby("bin", observed=True).agg(predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size"), x=("y", "sum")).reset_index(drop=True)
        cal_t["lo"], cal_t["hi"] = wilson(cal_t.x, cal_t.n); display(cal_t); TABLES["calibration"] = cal_t
        fig, ax = plt.subplots(figsize=(5.2, 5)); lim = [cal_t.predicted.min() * .6, max(cal_t.hi.max(), cal_t.predicted.max()) * 1.3]
        ax.plot(lim, lim, color=NEUTRAL, ls="--", label="Perfect calibration")
        ax.errorbar(cal_t.predicted, cal_t.observed, yerr=[cal_t.observed - cal_t.lo, cal_t.hi - cal_t.observed], fmt="o", color=SERIES[0], ms=7, mec=SURFACE, label="Deciles (95% CI)")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed frequency"); ax.set_title("Calibration (forward in time)"); ax.legend(fontsize=8); ax.grid(axis="x")
        show(fig, "14_calibration")

        # Permutation importance on the last year (model trained on earlier years only): drop in AUC when a predictor is shuffled
        t = YL[-1]; te_idx = np.where(acc.ANYO.to_numpy() == t)[0]; idx = rng.choice(te_idx, size=min(40000, len(te_idx)), replace=False)
        pi = permutation_importance(models[t], X.iloc[idx], y[idx], scoring="roc_auc", n_repeats=3, random_state=SEED, n_jobs=1)
        imp = pd.DataFrame({"variable": X.columns, "AUC_drop": pi.importances_mean, "sd": pi.importances_std}).sort_values("AUC_drop", ascending=False)
        display(imp.head(15)); TABLES["permutation_importance"] = imp
        top = imp.head(15).iloc[::-1]; fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(top["variable"], top.AUC_drop, xerr=top.sd, color=SERIES[0], height=.6, ecolor=INK2); ax.grid(axis="y", visible=False); ax.grid(axis="x")
        ax.set_xlabel("Drop in AUC when the predictor is permuted")
        ax.set_title(f"Permutation importance (AUC drop in {t})"); show(fig, "15_importance")
        print("Importances describe what the model uses, not what causes fatalities. Causal effects require a design (see the manuscript plan).")
    else:
        print("With a single year only the spatial validation could be run.")

## 11 · Typologies of fatal crashes

**Purpose.** Describe the *kinds* of fatal crash and test whether their mix drifts between years. Fatal crashes (30 days) are clustered by K-means on one-hot-encoded categorical descriptors (crash type, road-user type, area, road group, lighting, weather, alignment, junction, number of vehicles, time band, weekday/weekend). For each cluster, the features with the highest **lift** (prevalence within the cluster ÷ prevalence overall) are listed. Finally a **χ² test of homogeneity** asks whether the distribution of clusters differs across years.

K-means on binary indicators yields a modest silhouette: treat the groups as a **useful description**, not as natural classes. The number of clusters K is chosen by silhouette unless fixed in `K_TYPOLOGIES`. Category labels derived here are English summaries of the DGT codes.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

fat = acc[acc.fatal30 == 1].copy()
ucols = {k: f"TOT_{k}_MU30DF" for k in USER_TYPES if f"TOT_{k}_MU30DF" in fat}
um = fat[list(ucols.values())].gt(0); um.columns = [USER_TYPES[k] for k in ucols]
fat["road_user"] = np.where(um.sum(axis=1) > 1, "Several road-user types", um.idxmax(axis=1))
CRASH_TYPE_GROUP = {1: "Head-on", 2: "Side/oblique", 3: "Side/oblique", 4: "Rear-end/multiple", 5: "Rear-end/multiple", 6: "Collision with obstacle",
                    7: "Pedestrian struck", 8: "Animal struck", 9: "Overturn", 10: "Fall", 11: "Run-off, single vehicle", 12: "Run-off with collision", 16: "Run-off with collision",
                    13: "Run-off with drop", 17: "Run-off with drop", 14: "Run-off with overturn", 18: "Run-off with overturn", 15: "Run-off, other", 19: "Run-off, other", 20: "Other type"}
fat["crash_type_grp"] = fat.TIPO_ACCIDENTE.map(CRASH_TYPE_GROUP).fillna("Other type")
fat["lighting_grp"] = fat.CONDICION_ILUMINACION.map({1: "Daylight", 2: "Twilight", 3: "Twilight", 4: "Night, lit road", 5: "Night, lighting off", 6: "Night, unlit"}).fillna("Lighting n/a")
fat["weather_grp"] = fat.CONDICION_METEO.map({1: "Clear", 2: "Overcast", 3: "Rain", 4: "Rain", 5: "Hail/snow", 6: "Hail/snow"}).fillna("Weather n/a")
fat["alignment_grp"] = fat.TRAZADO_PLANTA.map({1: "Straight", 2: "Signed curve", 3: "Unsigned curve"}).fillna("Alignment n/a")
fat["junction_grp"] = fat.NUDO.map({1: "At junction", 2: "Away from junction"}).fillna("Junction n/a")
fat["vehicles_grp"] = pd.cut(fat.TOTAL_VEHICULOS, [-1, 1, 2, 999], labels=["1 vehicle", "2 vehicles", "3+ vehicles"]).astype(str)
fat["daytype_grp"] = np.where(fat.weekend, "Weekend", "Weekday")
FEATS = ["crash_type_grp", "road_user", "area", "road_group", "lighting_grp", "weather_grp", "alignment_grp", "junction_grp", "vehicles_grp", "time_band", "daytype_grp"]
D = pd.get_dummies(fat[FEATS], prefix_sep="=", dtype=float); D = D.loc[:, D.mean() >= 0.02]       # indicators with < 2 % support are dropped (unstable)
print(f"{len(fat):,} fatal crashes · {D.shape[1]} indicators (>= 2% support)")

sil = {}
for k in range(4, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(D); sil[k] = silhouette_score(D, lab, sample_size=min(4000, len(D)), random_state=SEED)
sil = pd.Series(sil, name="silhouette"); display(sil.to_frame().T)
K_TYPOLOGIES = None            # set a number (4-8) to force it; if None the number with the highest silhouette is used
K = int(K_TYPOLOGIES or sil.idxmax())
fat["cluster"] = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(D)
share_all = D.mean(); rows = []
for c in range(K):
    s = D[fat.cluster.to_numpy() == c].mean(); lift = (s / share_all)[s >= 0.30].sort_values(ascending=False)
    rows.append(dict(cluster=c, n=int((fat.cluster == c).sum()), pct=100 * (fat.cluster == c).mean(), mean_deaths=fat.loc[fat.cluster == c, "TOTAL_MU30DF"].mean(),
                     label=" · ".join(i.split("=")[1] for i in lift.index[:3]), features=", ".join(f"{i.split('=')[1]} (×{v:.1f})" for i, v in lift.head(6).items())))
typ = pd.DataFrame(rows).set_index("cluster").sort_values("n", ascending=False); display(typ); TABLES["typologies"] = typ

In [ ]:
# ---- Drift of the typology mix across years (chi-square test of homogeneity) ---------------------------
if MULTI:
    ct = pd.crosstab(fat.cluster, fat.ANYO); chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f"Homogeneity of the typology mix across years: chi2 = {chi2:.1f}, df = {dof}, p = {p:.3f}"); display(ct)
    sh = ct / ct.sum() * 100; TABLES["typologies_by_year"] = sh
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    for i, c in enumerate(typ.index):
        ax.plot(sh.columns, sh.loc[c], color=SERIES[i % 8], marker="o", ms=6, mec=SURFACE, mew=1.5, label=f"{c}: {typ.loc[c, 'label']}")
    ax.set_xticks(list(sh.columns)); ax.set_ylim(0, sh.values.max() * 1.2); ax.set_ylabel("% of the year's fatal crashes"); ax.set_title("Mix of fatal-crash typologies by year")
    ax.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1)); show(fig, "16_typologies_by_year")
else:
    print("Year-on-year drift requires at least two years.")

## 12 · Reviewer-proofing analyses

Sections 1–11 answer the paper's questions; this block closes the gaps a referee is most likely to point at. Every analysis reuses the objects built above
(`acc`, `YL`, `CE`, `cell_table`, `py`, `GR`, …) and writes its tables to `TABLES`, so Section 12 exports them. Nothing here changes a number reported earlier.

| Block | Question | Needs |
|---|---|---|
| 12.1 | Does an **empirical-Bayes** score for road segments beat the raw count in forward validation (the claim behind H9)? Is the result robust to rounded kilometre points? | crashes only |
| 12.2 | What does **Gi\*** add over trivial rankings of cells (past fatal count, casualty-crash density, network length…)? | geocoding (Section 7) |
| 12.3 | Does **spatial smoothing** help for **frequency (F)** and **risk (R)**, as it was tested for severity (L)? | PyMC + nutpie (Section 8) |
| 12.4 | Do geocoded points fall **inside their declared province**? | province polygons (downloaded) |
| 12.5 | Does the risk ranking survive **other denominators** (network length, vehicle fleet)? | polygons; fleet file optional |
| 12.6 | Hypotheses that the data **can** test: H5 seasonality by road user, H2c tourism in hot cells, H1c remoteness and late deaths, H4 (lighting only) | crashes + Section 7 |

**Note on H4.** The crash files contain **no** alcohol, drug or speed variables (see the dictionary workbook): only lighting. 12.6 tests what lighting can say, and the manuscript text that says otherwise must be corrected.

In [ ]:
# ---- 12.1 Segments: empirical-Bayes score versus raw counts, forward in time ---------------------------------
# Paper claim (H9): "a list of black spots ordered by raw counts overstates the risk of its first places, and an estimate smoothed towards the mean is a better guide".
# Section 5 showed the regression to the mean but never ranked segments by a *shrunken* score. Here every score is built from the training years only and scored on the
# held-out years, with the same universe, schemes and tie-breaking as Section 5. Paired bootstrap over segments gives an interval for the difference to the raw count.
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)     # progress line with clock time
def seg_frame(df, years, w=SEGMENT_KM):
    """Per-segment totals over `years`: casualty crashes, fatal crashes, severity-weighted count (10/3/1) and the road ownership of the segment."""
    sub = df[df.ANYO.isin(years)]; t = make_segment(sub, w); sub = sub[t.notna()].assign(segment=t[t.notna()])
    sub = sub.assign(sev=10 * sub.TOTAL_MU30DF + 3 * sub.TOTAL_HG30DF + sub.TOTAL_HL30DF)
    g = sub.groupby("segment")
    return g.agg(crashes=("UID", "size"), fatal=("fatal30", "sum"), sev=("sev", "sum"), own=("ownership", "first"))

def eb_gamma_mom(F, E):
    """Gamma(a, b) prior for Poisson rates F/E by the method of moments (Clayton & Kaldor 1987); same estimator as the cell analysis of Section 7."""
    F, E = np.asarray(F, float), np.asarray(E, float); m = F.sum() / E.sum(); lam = F / E
    s2 = np.sum(E * (lam - m) ** 2) / E.sum(); t2 = max(s2 - m / E.mean(), 1e-12); return m * m / t2, m / t2

def eb_rate(F, E, groups=None, min_group=300):
    """Posterior mean of the rate under a gamma prior. With `groups`, the prior MEAN is group-specific (e.g. road ownership) and the shape is estimated within the
    group when it has >= min_group segments (else the pooled prior is used): it is a simple empirical-Bayes model with a covariate."""
    F, E = np.asarray(F, float), np.asarray(E, float); a0, b0 = eb_gamma_mom(F, E)
    if groups is None: return (F + a0) / (E + b0)
    g = pd.Series(np.asarray(groups).astype(str)); out = (F + a0) / (E + b0)
    for k, ix in g.groupby(g).groups.items():
        ix = np.asarray(ix)
        if len(ix) >= min_group and F[ix].sum() > 0:
            a, b = eb_gamma_mom(F[ix], E[ix]); out[ix] = (F[ix] + a) / (E[ix] + b)
    return out

def eb_prop(x, n):
    """Posterior mean of a proportion under a beta prior estimated by moments (empirical-Bayes shrinkage of the case fatality of a segment)."""
    x, n = np.asarray(x, float), np.asarray(n, float); ok = n > 0; p = x[ok].sum() / n[ok].sum()
    s2 = np.sum(n[ok] * (x[ok] / n[ok] - p) ** 2) / n[ok].sum(); v = max(s2 - p * (1 - p) / n[ok].mean(), 1e-9); k = max(p * (1 - p) / v - 1, 0.5)
    return (x + p * k) / (n + k)

def spf_eb(tr, ny):
    """Highway-Safety-Manual style empirical Bayes. A safety performance function for the fatal crashes of a segment is fitted on the training segments
    (Poisson GLM; predictors: log casualty crashes, which stands in for traffic because no counts are available, and road ownership; the negative-binomial
    dispersion is estimated by the moment method of Cameron & Trivedi). The estimate is the gamma-Poisson posterior mean, i.e. a weighted mix of the SPF
    prediction and the observed count: posterior rate = (fatal + k) / (ny + k / m), with m the SPF rate and k = 1 / dispersion."""
    import statsmodels.api as sm
    X = pd.get_dummies(tr.own.astype(str), drop_first=True, dtype=float); X["log_crashes"] = np.log(tr.crashes.values); X = sm.add_constant(X, has_constant="add")
    y = tr.fatal.values.astype(float); mu = sm.GLM(y, X, family=sm.families.Poisson(), offset=np.full(len(y), np.log(ny))).fit().fittedvalues.to_numpy()
    alpha = max(np.sum((y - mu) ** 2 - y) / np.sum(mu ** 2), 1e-6); k = 1 / alpha; m = mu / ny
    return (y + k) / (ny + k / m)

def seg_scores(tr, ny):
    """Scores of every segment seen in the training years.
    Raw: previous fatal crashes, previous casualty crashes, severity-weighted count (10/3/1).
    Empirical Bayes (rates per year = expected fatal crashes per segment-year):
      'EB gamma' shrinks the fatal count towards the mean. With equal exposure it is a monotone function of the count, so it cannot change the ranking; it is kept for the calibration check only.
      'EB-SPF' is the HSM-style estimate that borrows strength from casualty crashes and ownership; 'EB two-stage' multiplies a shrunken casualty-crash frequency by a shrunken case fatality."""
    E = np.full(len(tr), float(ny)); s = pd.DataFrame(index=tr.index)
    s["raw: previous fatal crashes"] = tr.fatal.values; s["raw: previous casualty crashes"] = tr.crashes.values; s["raw: severity-weighted count"] = tr.sev.values
    s["EB gamma (calibration only)"] = eb_rate(tr.fatal.values, E)
    s["EB-SPF (HSM-style)"] = spf_eb(tr, ny)
    s["EB two-stage (frequency x severity)"] = eb_rate(tr.crashes.values, E, tr.own.values) * eb_prop(tr.fatal.values, tr.crashes.values)
    return s

def top_flags(score, universe, frac, rg):
    """Boolean flag over `universe` for the best-ranked `frac` of segments. Segments without training crashes score 0; ties are broken at random."""
    sc = score.reindex(universe).fillna(0.0).to_numpy(); k = max(1, int(round(frac * len(universe))))
    order = np.lexsort((rg.random(len(sc)), -sc)); f = np.zeros(len(sc), bool); f[order[:k]] = True; return f

def seg_forward(df, tr_y, te_y, w=SEGMENT_KM, fracs=(0.01, 0.02, 0.05, 0.10), n_boot=400, seed=SEED, boot=True, tag=""):
    """One forward scheme. Returns the capture table (share of future fatal crashes in the best-ranked x % of segments) and the calibration of the top 1 %."""
    rg = np.random.default_rng(seed); tick(f"  {tag}building the segment universe (all years)"); universe = seg_frame(df, sorted(df.ANYO.unique()), w).index
    tick(f"  {tag}{len(universe):,} segments · training and test tables"); tr, te = seg_frame(df, tr_y, w), seg_frame(df, te_y, w); tick(f"  {tag}scores (SPF regression, empirical Bayes)"); S = seg_scores(tr, len(tr_y))
    tick(f"  {tag}ranking and capture")
    y = te.fatal.reindex(universe).fillna(0).to_numpy(); tot = y.sum(); rows = []; flags = {}
    RANKED = [c for c in S.columns if "calibration only" not in c]
    for sc in RANKED:
        for f in fracs:
            fl = top_flags(S[sc], universe, f, rg); rows.append(dict(score=sc, frac=f, capture_pct=100 * y[fl].sum() / tot)); flags[(sc, f)] = fl
    cap = pd.DataFrame(rows)
    if boot:
        tick(f"  {tag}paired bootstrap ({n_boot} resamples)")                  # paired bootstrap over segments: difference to the raw fatal-crash count in the best 5 %
        # Only segments with future fatal crashes can change a capture share, so the resampling is done on those: the counts of this subset under a bootstrap of all
        # segments are multinomial with probability 1/U each (the rest of the segments is lumped into one cell). This is exact and ~1000 times faster than resampling every segment.
        pos = np.flatnonzero(y > 0); m = len(pos); U = len(y); yp = y[pos]; p = np.r_[np.full(m, 1 / U), 1 - m / U]
        Wb = rg.multinomial(U, p, size=n_boot)[:, :m].astype(float); den = np.maximum(Wb @ yp, 1)
        ref = flags[("raw: previous fatal crashes", 0.05)][pos]
        for sc in RANKED:
            fl = flags[(sc, 0.05)][pos]; d = 100 * (Wb @ (yp * fl) - Wb @ (yp * ref)) / den
            cap.loc[(cap.score == sc) & (cap.frac == 0.05), ["diff_vs_raw_fatal", "diff_lo", "diff_hi"]] = [d.mean(), *np.percentile(d, [2.5, 97.5])]
    cal = []                                                         # calibration: predicted vs observed future fatal crashes per segment-year in the top 1 %
    for sc in ("raw: previous fatal crashes", "EB gamma (calibration only)", "EB-SPF (HSM-style)", "EB two-stage (frequency x severity)"):
        fl = top_flags(S[sc], universe, 0.01, rg); pred = S[sc].reindex(universe).fillna(0).to_numpy()[fl]
        if sc.startswith("raw"): pred = pred / len(tr_y)
        cal.append(dict(score=sc, predicted_per_seg_year=pred.mean(), observed_per_seg_year=y[fl].mean() / len(te_y)))
    cal = pd.DataFrame(cal).assign(observed_over_predicted=lambda d: d.observed_per_seg_year / d.predicted_per_seg_year)
    return cap, cal, len(universe)

if MULTI and len(YL) >= 2:
    tick("12.1 start")
    schemes = {f"{y}→{y + 1}": ([y], [y + 1]) for y in YL[:-1]}
    if len(YL) >= 3: schemes[f"{YL[0]}–{YL[-2]}→{YL[-1]}"] = (YL[:-1], [YL[-1]])
    if len(YL) == 4: schemes[f"{YL[0]}–{YL[1]}→{YL[2]}–{YL[3]}"] = (YL[:2], YL[2:])
    caps, cals = [], []
    for i_, (nm, (tr_y, te_y)) in enumerate(schemes.items(), 1):
        tick(f"12.1 scheme {i_}/{len(schemes)}: train {tr_y} → test {te_y}")
        c_, k_, U_ = seg_forward(acc, tr_y, te_y, tag=f"[{nm}] "); caps.append(c_.assign(scheme=nm)); cals.append(k_.assign(scheme=nm))
    tick("12.1 tables and figure"); cap = pd.concat(caps); cal = pd.concat(cals)
    t = cap.assign(col=lambda d: "top " + (d.frac * 100).astype(int).astype(str) + "%").pivot_table(index=["scheme", "score"], columns="col", values="capture_pct", sort=False)
    t = t[["top 1%", "top 2%", "top 5%", "top 10%"]]; display(t.round(1)); TABLES["segment_eb_capture"] = t.reset_index()
    d5 = cap[(cap.frac == 0.05) & cap.score.str.startswith("EB")][["scheme", "score", "capture_pct", "diff_vs_raw_fatal", "diff_lo", "diff_hi"]]
    print("\nBest 5 %: gain of each EB score over the raw count of previous fatal crashes (percentage points of future fatal crashes captured; paired bootstrap 95% interval)")
    display(d5.round(2)); TABLES["segment_eb_vs_raw"] = d5
    print("\nCalibration of the best-ranked 1 % of segments: observed / predicted future fatal crashes per segment-year (1 = calibrated; < 1 = the score overstates the risk)")
    display(cal.round(3)); TABLES["segment_eb_calibration"] = cal
    # Figure: capture in the best 5 % by score, one panel per scheme
    last = list(schemes)[-2] if len(schemes) >= 2 else list(schemes)[0]
    fig, ax = plt.subplots(figsize=(8.2, 4.2)); order = list(t.index.get_level_values("score").unique()); xs = np.arange(len(schemes)); wd = 0.8 / len(order)
    for i, sc in enumerate(order):
        v = [t.loc[(nm, sc), "top 5%"] for nm in schemes]; ax.bar(xs + (i - (len(order) - 1) / 2) * wd, v, wd, color=(SERIES + [NEUTRAL])[i % 9], label=sc, hatch="" if sc.startswith("EB") else "//", ec=SURFACE)
    ax.axhline(5, color=INK2, ls="--", lw=1); ax.set_xticks(xs); ax.set_xticklabels(list(schemes), fontsize=8); ax.set_ylabel("% of future fatal crashes in the best-ranked 5 %")
    ax.set_title("Raw (hatched) versus empirical-Bayes scores of 2-km segments"); ax.legend(fontsize=7, ncol=2, loc="upper left"); show(fig, "30_segment_eb_vs_raw")
    # Robustness to rounded kilometre points (13-54 % of records report a whole kilometre)
    nm = list(schemes)[len(YL) - 1] if len(YL) >= 3 else list(schemes)[0]; tr_y, te_y = schemes[nm]; rg2 = np.random.default_rng(SEED + 1); rows = []
    isint = (acc.KM % 1 == 0)
    variants = {"as in the paper": acc, "drop whole-km records": acc[~isint]}
    jit = acc.copy(); jit.loc[isint, "KM"] = (jit.loc[isint, "KM"] + rg2.uniform(-0.5, 0.5, isint.sum())).clip(lower=0); variants["whole-km records jittered ±0.5 km"] = jit
    for vn, dfv in variants.items():
        tick(f"12.1 km-rounding check: '{vn}'")
        c_, _, U_ = seg_forward(dfv, tr_y, te_y, boot=False, tag=f"[{vn}] ")
        for sc in ("raw: severity-weighted count", "raw: previous casualty crashes", "EB-SPF (HSM-style)", "EB two-stage (frequency x severity)"):
            rows.append(dict(variant=vn, score=sc, top5_capture_pct=c_[(c_.score == sc) & (c_.frac == 0.05)].capture_pct.iat[0], segments=U_))
    tick("12.1 done"); kr = pd.DataFrame(rows); print(f"\nSensitivity to rounded kilometre points (scheme {nm}); {isint.mean():.0%} of the records have a whole-number KM"); display(kr.round(1)); TABLES["segment_km_rounding"] = kr
else:
    print("12.1 needs at least two years.")

In [ ]:
# ---- 12.2 Cells: what does Gi* add over trivial rankings? -------------------------------------------------------
# Section 7 shows that the hot cells of the past keep a higher rate, but it never asks whether a *simpler* list would have done as well. Hot cells are the large urban
# areas, which stay large. Here, for every forward split, the same number of cells (the best-ranked 5 %) is flagged with each of the rankings below, using the training
# years only, and the share of the held-out year's fatal crashes that fall in them is compared (with the share of network kilometres they occupy).
#   Gi* z-score ........ the statistic of Section 7 (ranks by the strength of the cluster, not by the cell's own rate)
#   EB smoothed rate ... the ranking used for the 'top 5 %' of Section 7
#   raw fatal count, raw rate, casualty crashes (a proxy for traffic), casualty-crash density, network km (size alone)
# Paired cluster bootstrap over cells: the same resample is used for every ranking, so differences between rankings are comparable.
if HAS_PK and MULTI:
    import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)     # progress line with clock time
    tick("12.2 start")
    rg14 = np.random.default_rng(SEED + 14); rows, boots = [], []; B = 500
    for k in range(1, len(YL)):
        tr_y, te_y = YL[:k], [YL[k]]; tick(f"12.2 split {k}/{len(YL) - 1}: train {tr_y} → test {te_y} · cell table"); c, _, _ = cell_table(tr_y); W = queen_W(c)
        tick(f"  {len(c):,} cells · Gi* with 499 permutations (the slow step: it can take a minute or more)"); c["z"], _, _ = getis_ord(c.eb.values, W, nperm=499)
        c["crash_density"] = c.crashes / c.E
        te = CE[CE.ANYO.isin(te_y)].groupby(["cx", "cy"]).fatal.sum().rename("fatal_te").reset_index()
        c = c.merge(te, on=["cx", "cy"], how="left").fillna({"fatal_te": 0}); y = c.fatal_te.to_numpy(); km = c.road_km.to_numpy(float); n = len(c); n5 = max(1, int(round(.05 * n)))
        RK = {"Gi* z-score": c.z, "EB smoothed rate": c.eb, "raw fatal count": c.fatal, "raw fatal rate": c.rate, "casualty crashes": c.crashes,
              "casualty-crash density": c.crash_density, "network km (size only)": c.road_km}
        flags = {}
        for nm, v in RK.items():
            o = np.lexsort((rg14.random(n), -np.asarray(v, float))); f = np.zeros(n, bool); f[o[:n5]] = True; flags[nm] = f
        tick(f"  baselines and paired bootstrap ({B} resamples)"); idx = [rg14.integers(0, n, n) for _ in range(B)]
        for nm, f in flags.items():
            cap = y[f].sum() / y.sum(); net = km[f].sum() / km.sum()
            bc = np.array([y[i][f[i]].sum() / max(y[i].sum(), 1) for i in idx]); bd = np.array([(y[i][f[i]].sum() - y[i][flags["Gi* z-score"][i]].sum()) / max(y[i].sum(), 1) for i in idx])
            rows.append(dict(training=f"{YL[0]}–{YL[k - 1]}" if k > 1 else str(YL[0]), test=YL[k], ranking=nm, capture_pct=100 * cap, capture_lo=100 * np.percentile(bc, 2.5), capture_hi=100 * np.percentile(bc, 97.5),
                             network_pct=100 * net, PAI=cap / net, diff_vs_Gi_pp=100 * bd.mean(), diff_lo=100 * np.percentile(bd, 2.5), diff_hi=100 * np.percentile(bd, 97.5)))
    tick("12.2 done"); cb = pd.DataFrame(rows); display(cb.round(2)); TABLES["cell_baselines"] = cb
    print("Reading: capture = % of the test year's fatal crashes inside the best-ranked 5 % of cells; PAI = capture / share of network km flagged (1 = chance); diff = capture minus that of Gi* (paired 95% bootstrap interval).")
    last = cb[cb.test == YL[-1]].set_index("ranking"); fig, ax = plt.subplots(figsize=(7.4, 3.9)); yy = np.arange(len(last))
    ax.barh(yy, last.capture_pct, color=[SERIES[0] if r == "Gi* z-score" else (SERIES[1] if r == "EB smoothed rate" else NEUTRAL) for r in last.index])
    ax.errorbar(last.capture_pct, yy, xerr=[last.capture_pct - last.capture_lo, last.capture_hi - last.capture_pct], fmt="none", ecolor=INK2, lw=1, capsize=2)
    ax.axvline(5, color=INK2, ls="--", lw=1); ax.set_yticks(yy); ax.set_yticklabels(last.index, fontsize=8); ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.grid(axis="x")
    ax.set_xlabel(f"% of the fatal crashes of {YL[-1]} in the best-ranked 5 % of cells (dashed = 5 %)"); ax.set_title(f"Which ranking of cells predicts {YL[-1]} best?"); show(fig, "31_cell_baselines")
else:
    print("12.2 needs the geocoded cells (Section 7) and at least two years.")

In [ ]:
# ---- 12.3 Does spatial smoothing help for frequency (F) and risk (R)? Province-grouped K-fold, as done for severity (L) ----
# Section 8 tested the BYM2 against independent province effects only for model L. The same test is repeated here for F and R: whole provinces are removed from the
# likelihood, and the cell-level log predictive density of the held-out provinces is compared between BYM2 and the independent model. The runtime is K x 2 fits per model.
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
def build_P_w(d, G=GR, U=1.0, spatial=True, rho_prior=(1, 1), w=None):
    """Same model as build_P (Poisson-lognormal with exposure offset) with two additions needed for cross-validation: 0/1 observation weights `w`
    (0 = province held out of the likelihood) and the linear predictor stored as 'eta', so that held-out cells can be predicted."""
    with pm.Model(coords={**COORDS, "cell": np.arange(len(d))}) as m:
        a = pm.Normal("alpha", float(np.log(d.y.sum() / d.E.sum())), 1.0)
        g = pm.ZeroSumNormal("gamma", sigma=0.5, dims="year") if Y > 1 else 0.0
        b, sb, rho = bym2("b", G, U, spatial, rho_prior)
        sd = pm.Exponential("sigma_py", -np.log(0.01) / U); dpy = pm.Normal("d_py", 0, 1, shape=(G["N"], Y))
        eta = np.log(d.E.values) + a + (g[d.t.values] if Y > 1 else 0.0) + b[d.p.values] + sd * dpy[d.p.values, d.t.values]
        pm.Deterministic("eta", eta, dims="cell")
        if w is None: pm.Poisson("x", mu=pt.exp(eta), observed=d.y.values, dims="cell")
        else: pm.Potential("ll", pt.sum(w * pm.logp(pm.Poisson.dist(mu=pt.exp(eta)), d.y.values)))
    return m

def kfold_P(resp, label, K=None, seed=SEED):
    """Province-grouped K-fold for F ('crashes') or R ('deaths'). Returns the out-of-sample elpd of BYM2 and iid and their paired difference with its standard error."""
    from scipy.special import logsumexp
    from scipy.stats import poisson
    d = data_P(resp); K = K or KFOLD_K; perm = np.random.default_rng(seed).permutation(len(PROVS)); folds = np.array_split(perm, K); contrib = {"BYM2": [], "iid": []}
    tick(f"12.3 {label}: {K} folds x 2 models = {2 * K} fits")
    for f, held in enumerate(folds):
        mask = ~d.p.isin(held).values; w = mask.astype(float)
        for nm, sp_ in [("BYM2", True), ("iid", False)]:
            tick(f"  {label}: fold {f + 1}/{K} · {nm} · sampling")
            with build_P_w(d, spatial=sp_, w=w): idk = pm.sample(**{**BAY, "draws": max(BAYES_DRAWS // 2, 300), "tune": max(BAYES_TUNE // 2, 500)}, idata_kwargs={"log_likelihood": False})
            eta = draws(idk, "eta")[:, ~mask]; mu = np.exp(eta); ll = poisson.logpmf(d.y.values[~mask][None, :], mu)
            contrib[nm].append(logsumexp(ll, axis=0) - np.log(ll.shape[0]))
        tick(f"  {label}: fold {f + 1}/{K} finished ({len(held)} provinces held out)")
    e1, e0 = np.concatenate(contrib["BYM2"]), np.concatenate(contrib["iid"]); dl = e1 - e0
    return dict(model=label, elpd_BYM2=e1.sum(), elpd_iid=e0.sum(), diff=dl.sum(), se_diff=np.sqrt(len(dl)) * dl.std(), cells=len(dl))

if HAS_PYMC and RUN_BAYES and RUN_KFOLD and HAS_POP and SAMPLER == "nutpie":
    kf = pd.DataFrame([kfold_P("crashes", "F · frequency"), kfold_P("deaths", "R · risk")])
    if "bayes_kfold_provinces" in TABLES:                                         # model L from Section 8, for comparison
        L_ = TABLES["bayes_kfold_provinces"].set_index("model").elpd; kf = pd.concat([pd.DataFrame([dict(model="L · severity (Section 8)", elpd_BYM2=L_["BYM2"], elpd_iid=L_["iid"], diff=L_["BYM2"] - L_["iid"])]), kf], ignore_index=True)
    display(kf.round(2)); TABLES["bayes_kfold_FR"] = kf
    print("Reading: diff > 0 favours borrowing information from neighbouring provinces. If |diff| is about one standard error, the spatial structure adds little. Held-out provinces with no land neighbours (Baleares, Ceuta, Melilla) cannot benefit from the BYM2.")
else:
    print("12.3 skipped (needs PyMC, nutpie, RUN_BAYES, RUN_KFOLD and population).")

In [ ]:
# ---- 12.4 Do the geocoded points fall inside their declared province? -------------------------------------------
# Section 7 checks the coordinates only internally. With province polygons the check becomes external: the crash record declares a province independently of the layer
# of kilometre markers, so a point outside it is an error of the geocoder (or of the layer). The default polygons are a simplified public layer (fine for a check; for the
# paper use the INE/CNIG province boundaries, which are more precise at the border, and say so). Distances are computed in EPSG:3035 (valid for the whole of Spain).
PROV_GEO_URL  = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/spain-provinces.geojson"   # public, 52 units, property 'cod_prov' = INE code
PROV_GEO      = PROV_GEO or (DATA_DIR / "spain-provinces.geojson"); PROV_CODE_COL = PROV_CODE_COL or "cod_prov"
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
HAS_POLY = False
try:
    import geopandas as gpd
    if not Path(PROV_GEO).exists():
        import urllib.request; DATA_DIR.mkdir(parents=True, exist_ok=True); urllib.request.urlretrieve(PROV_GEO_URL, Path(PROV_GEO)); print("Province polygons downloaded to", Path(PROV_GEO).resolve())
    POL = gpd.read_file(PROV_GEO).to_crs(4326); POL["COD"] = pd.to_numeric(POL[PROV_CODE_COL], errors="coerce").astype("Int64"); POL = POL.dropna(subset=["COD"])[["COD", "geometry"]]
    HAS_POLY = len(POL) >= 50; print(f"Province polygons: {len(POL)} units")
except Exception as e:
    print("12.4-12.5 need geopandas and province polygons:", type(e).__name__, e)

if HAS_POLY and HAS_PK:
    tick("12.4 start · point-in-polygon join")
    g = acc[acc.geo_ok].copy(); pts = gpd.GeoDataFrame(g[["COD_PROVINCIA", "ownership", "fatal30"]], geometry=gpd.points_from_xy(g.geo_lon, g.geo_lat), crs=4326)
    j = gpd.sjoin(pts, POL, how="left", predicate="within"); j = j[~j.index.duplicated()]; g["COD_POINT"] = j["COD"].reindex(g.index).astype("float")
    g["inside"] = g.COD_POINT == g.COD_PROVINCIA
    tick(f"12.4 {len(g):,} points joined · distances of the points outside")
    pol3035 = POL.to_crs(3035).set_index("COD").geometry; pt3035 = pts.to_crs(3035).geometry; bad = g.index[~g.inside]
    dist = pd.Series([pt3035.loc[i].distance(pol3035.loc[int(g.COD_PROVINCIA.loc[i])]) / 1000 if int(g.COD_PROVINCIA.loc[i]) in pol3035.index else np.nan for i in bad], index=bad)
    g["d_border_km"] = 0.0; g.loc[bad, "d_border_km"] = dist
    tick(f"12.4 {len(bad):,} points outside their province")
    print(f"Reliable coordinates inside the declared province: {g.inside.mean():.1%} (n = {len(g):,}); among fatal crashes: {g[g.fatal30 == 1].inside.mean():.1%}")
    if len(bad): print("Distance of the points that fall outside to the declared province (km): ", dist.describe(percentiles=[.5, .9, .99]).round(1)[["50%", "90%", "99%", "max"]].to_dict(),
                       f"· within 2 km of the border: {(dist <= 2).mean():.0%} · within 5 km: {(dist <= 5).mean():.0%}")
    byown = g.groupby("ownership").agg(n=("inside", "size"), inside_pct=("inside", "mean"), median_error_km=("d_border_km", lambda s: s[s > 0].median())).assign(inside_pct=lambda d: d.inside_pct * 100)
    display(byown.round(1)); TABLES["geocoding_point_in_province"] = byown
    acc["in_declared_province"] = g.inside.reindex(acc.index)
else:
    print("12.4 skipped (no polygons or no geocoding).")

In [ ]:
# ---- 12.5 Does the risk ranking survive other denominators? ------------------------------------------------------------
# Risk per inhabitant ignores visitors and through traffic (H1d). Two denominators that are available without traffic counts:
#   (a) kilometres of interurban network of the province (distinct road/km markers of the layer inside its polygon): interurban deaths per 1000 network-km and year;
#   (b) vehicle fleet of the province, if a file is supplied (FLEET_FILE; tidy table with COD_PROVINCIA or province name, optional ANYO, and VEHICLES; the DGT publishes
#       the 'parque de vehículos' by province in DGT en cifras). Deaths per 100,000 registered vehicles and year.
# These are proxies: network length says nothing about how busy the roads are, and the fleet counts vehicles registered in the province, not vehicles circulating there.
# If the sparse provinces stay on top under all denominators the conclusion is robust; if they fall, the ranking reflects the choice of denominator.
FLEET_FILE = None    # e.g. DATA_DIR / "parque_vehiculos_provincia.xlsx"
import unicodedata
def _norm(s): return re.sub(r"[^a-z]", "", unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower())
def read_fleet(path):
    """Tidy fleet table -> DataFrame[COD_PROVINCIA, ANYO (or NaN), VEHICLES]. Province may be given as INE code or as a name (matched ignoring accents/case; 'Araba/Álava' etc. use the dictionary names)."""
    t = pd.read_excel(path) if str(path).lower().endswith(("xlsx", "xls")) else pd.read_csv(path, sep=None, engine="python"); t.columns = [str(c).strip().upper() for c in t.columns]
    vc = next(c for c in t.columns if c.startswith(("VEHIC", "PARQUE", "FLEET")))
    if "COD_PROVINCIA" in t: cod = pd.to_numeric(t.COD_PROVINCIA, errors="coerce")
    else:
        pc = next(c for c in t.columns if c.startswith(("PROV", "NOMBRE")));  m = {_norm(v): k for k, v in PNAME.items()}
        m.update({_norm(v.split("/")[0]): k for k, v in PNAME.items()}); m.update({_norm(v.split("/")[-1]): k for k, v in PNAME.items()}); m.update({_norm(v.split(",")[0]): k for k, v in PNAME.items()})
        cod = t[pc].map(lambda s: m.get(_norm(s), np.nan))
    out = pd.DataFrame({"COD_PROVINCIA": cod, "ANYO": pd.to_numeric(t["ANYO"], errors="coerce") if "ANYO" in t else np.nan, "VEHICLES": pd.to_numeric(t[vc], errors="coerce")}).dropna(subset=["COD_PROVINCIA", "VEHICLES"])
    out["COD_PROVINCIA"] = out.COD_PROVINCIA.astype(int); return out

if HAS_POLY and HAS_PK and HAS_POP:
    tick("12.5 start · network km per province")
    kp = pk[["x", "y", "road", "km"]].drop_duplicates(["road", "km"]); kpg = gpd.GeoDataFrame(kp, geometry=gpd.points_from_xy(kp.x, kp.y), crs=4326)
    jj = gpd.sjoin(kpg, POL, how="left", predicate="within"); jj = jj[~jj.index.duplicated()]
    netkm = jj.dropna(subset=["COD"]).groupby("COD").size().rename("network_km"); netkm.index = netkm.index.astype(int)
    tick("12.5 rates and rankings")
    D = (py.groupby("COD_PROVINCIA").agg(person_years=("POP", "sum"), deaths=("deaths", "sum")).join(netkm, how="left")
         .join(acc[acc.ZONA_AGRUPADA == 1].groupby("COD_PROVINCIA").TOTAL_MU30DF.sum().rename("deaths_interurban")))
    D["province"] = [PNAME[i] for i in D.index]; ny = len(YL)
    D["risk_per_million_inh"] = D.deaths / D.person_years * 1e6
    D["interurban_deaths_per_1000_km_year"] = D.deaths_interurban / (D.network_km * ny) * 1e3
    D["interurban_deaths_per_million_inh"] = D.deaths_interurban / D.person_years * 1e6
    cols = ["risk_per_million_inh", "interurban_deaths_per_million_inh", "interurban_deaths_per_1000_km_year"]
    if FLEET_FILE and Path(FLEET_FILE).exists():
        fl = read_fleet(FLEET_FILE); fl = fl.groupby("COD_PROVINCIA").VEHICLES.mean() if fl.ANYO.isna().all() else fl[fl.ANYO.isin(YL)].groupby("COD_PROVINCIA").VEHICLES.mean()
        D = D.join(fl.rename("vehicles")); D["deaths_per_100k_vehicles_year"] = D.deaths / (D.vehicles * ny) * 1e5; cols.append("deaths_per_100k_vehicles_year")
        print(f"Fleet read for {D.vehicles.notna().sum()} provinces.")
    else:
        print("No FLEET_FILE: the comparison uses inhabitants and network kilometres only.")
    R = D[cols].rank(ascending=False, method="min").astype("Int64").add_suffix("_rank")
    top = D.sort_values("risk_per_million_inh", ascending=False).head(10); res = pd.concat([top[["province"] + cols], R.loc[top.index]], axis=1)
    display(res.round(2)); TABLES["denominators_top10"] = res; TABLES["denominators_all"] = D.reset_index()
    sp_ = D[cols].corr(method="spearman"); print("\nSpearman correlation between the rankings:"); display(sp_.round(2)); TABLES["denominators_spearman"] = sp_
    k10 = {c: set(D[c].nlargest(10).index) for c in cols}; ov = {c: len(k10[cols[0]] & k10[c]) for c in cols[1:]}
    print("Provinces shared with the top 10 by deaths per inhabitant:", ov)
    fig, ax = plt.subplots(figsize=(6.4, 5.2)); sc = ax.scatter(D.risk_per_million_inh, D.interurban_deaths_per_1000_km_year, s=40, c=np.log10(D.person_years / ny), cmap=SEQ_CMAP, ec=SURFACE, lw=.8)
    for i in D.sort_values("risk_per_million_inh").tail(6).index.union(D.sort_values("interurban_deaths_per_1000_km_year").tail(4).index): ax.annotate(D.province[i], (D.risk_per_million_inh[i], D.interurban_deaths_per_1000_km_year[i]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("Deaths per million inhabitant-years (all roads)"); ax.set_ylabel("Interurban deaths per 1000 network-km and year"); ax.grid(axis="x"); fig.colorbar(sc, label="log10 population"); ax.set_title("Two denominators for the same deaths"); show(fig, "32_denominators")
else:
    print("12.5 skipped (needs polygons, geocoding layer and population).")

In [ ]:
# ---- 12.6a H5 · Seasonality by type of road user ----------------------------------------------------------------
# Hypothesis (Section 6.4): the summer peak follows exposure to leisure travel, so motorcyclists, cyclists and pedestrians should respond to the season differently from car occupants.
# The files give 30-day deaths by user type for every crash, so the monthly profile of each group can be compared directly. Reported: share of the group's deaths in June-September
# against the share of days (33.4 %), with Wilson bands, and a chi-square test of homogeneity of the monthly distribution against car occupants.
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
tick("12.6a H5")
USER = {"Car occupants": ["TOT_TUR_MU30DF"], "Motorcyclists": ["TOT_MOTO_MU30DF"], "Pedestrians": ["TOT_PEAT_MU30DF"], "Cyclists": ["TOT_BICI_MU30DF"],
        "Moped riders": ["TOT_CICLO_MU30DF"], "Vans and trucks": ["TOT_FURG_MU30DF", "TOT_CAM_MENOS3500_MU30DF", "TOT_CAM_MAS3500_MU30DF"]}
USER = {k: [c for c in v if c in acc] for k, v in USER.items()}
mu = pd.DataFrame({k: acc.groupby("MES")[v].sum().sum(axis=1) for k, v in USER.items()}).reindex(range(1, 13)).fillna(0)
days = np.array([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); summer = [6, 7, 8, 9]; dshare = days[np.array(summer) - 1].sum() / days.sum()
rows = []
for k in mu.columns:
    x, n = mu.loc[summer, k].sum(), mu[k].sum(); lo, hi = wilson(x, n); ref = mu["Car occupants"]
    chi = stats.chi2_contingency(np.vstack([mu[k].values, ref.values]) + 0.0)[1] if k != "Car occupants" else np.nan
    rows.append(dict(user=k, deaths=int(n), share_Jun_Sep_pct=100 * x / n, lo=100 * lo, hi=100 * hi, ratio_to_days_share=(x / n) / dshare, p_vs_car_occupants=chi))
h5 = pd.DataFrame(rows); display(h5.round(3)); TABLES["H5_seasonality_by_user"] = h5
print(f"Share of days in June-September: {dshare:.1%}. A ratio above 1 = more deaths in summer than the calendar alone gives. p = chi-square test of the 12-month distribution of the group against car occupants (exploratory, unadjusted for multiplicity).")
fig, ax = plt.subplots(figsize=(8.6, 4.2))
for i, k in enumerate(mu.columns): ax.plot(range(1, 13), 100 * (mu[k] / days) / (mu[k] / days).sum(), color=SERIES[i % 8], marker="o", ms=4, label=f"{k} (n={int(mu[k].sum())})")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTHS); ax.set_ylabel("% of the group's deaths per day of the month"); ax.set_title("Monthly profile of 30-day deaths by road user (per day, normalised)"); ax.legend(fontsize=7, ncol=2); show(fig, "33_H5_seasonality_users")

In [ ]:
# ---- 12.6b H2c · Are the hot cells of the tourist coast more seasonal than those of the metropolitan areas? -----------------
# Hypothesis (6.2): where tourism explains the hot cells, their fatal crashes should concentrate in the summer more than in the metropolitan areas. The classification of
# provinces below is an assumption (editable): cells are assigned to the province declared by their crashes (the majority).
TOURIST_PROV = [17, 43, 12, 3, 30, 4, 29, 11, 21, 7]        # Girona, Tarragona, Castelló, Alicante, Murcia, Almería, Málaga, Cádiz, Huelva, Baleares
METRO_PROV   = [28, 8, 46, 41, 50, 48, 15, 33]              # Madrid, Barcelona, València, Sevilla, Zaragoza, Bizkaia, A Coruña, Asturias
if HAS_PK and "class" in ct:
    import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True); tick("12.6b H2c")
    cellprov = gg.groupby(["cx", "cy"]).COD_PROVINCIA.agg(lambda s: s.mode().iat[0]).rename("prov_cell").reset_index()
    cc = ct[["cx", "cy", "class"]].merge(cellprov, on=["cx", "cy"], how="left")
    m = gg.merge(cc, on=["cx", "cy"], how="inner"); m["group"] = np.where(m["class"] == "hot", np.where(m.prov_cell.isin(TOURIST_PROV), "hot · tourist coast", np.where(m.prov_cell.isin(METRO_PROV), "hot · metropolitan", "hot · other")),
                                                                         np.where(m.prov_cell.isin(TOURIST_PROV), "other cells · tourist provinces", "other cells · rest"))
    m["summer"] = m.MES.isin([6, 7, 8, 9]); rows = []
    for gname, d in m.groupby("group"):
        for lab, s in [("fatal crashes", d[d.fatal30 == 1]), ("casualty crashes", d)]:
            x, n = int(s.summer.sum()), len(s); lo, hi = wilson(x, n) if n else (np.nan, np.nan); rows.append(dict(group=gname, outcome=lab, n=n, summer_share_pct=100 * x / max(n, 1), lo=100 * lo, hi=100 * hi))
    h2 = pd.DataFrame(rows); display(h2.round(1)); TABLES["H2c_summer_share_by_group"] = h2
    for lab in ("fatal crashes", "casualty crashes"):
        a, b = h2[(h2.group == "hot · tourist coast") & (h2.outcome == lab)], h2[(h2.group == "hot · metropolitan") & (h2.outcome == lab)]
        if len(a) and len(b):
            xa, na, xb, nb = (a.summer_share_pct.iat[0] / 100 * a.n.iat[0]), a.n.iat[0], (b.summer_share_pct.iat[0] / 100 * b.n.iat[0]), b.n.iat[0]
            p = stats.fisher_exact([[round(xa), na - round(xa)], [round(xb), nb - round(xb)]])[1]
            print(f"{lab}: share in June-September, tourist-coast hot cells {100 * xa / na:.1f}% vs metropolitan hot cells {100 * xb / nb:.1f}% (Fisher p = {p:.3f}). Share of days: {dshare:.1%}.")
else:
    print("12.6b needs Section 7 (hot cells).")

In [ ]:
# ---- 12.6c H1c · Remoteness and the share of deaths that occur after the first 24 hours ----------------------------------
# Hypothesis (6.1): a crash far from a hospital leaves more time to die at the scene, so in remote provinces a SMALLER share of the 30-day deaths should occur after 24 hours.
# Test: for every crash with deaths, `late deaths` (between 24 h and 30 days) out of `deaths at 30 days`, binomial GLM with cluster-robust standard errors by province.
# Province population (mean over the years) is the remoteness proxy: an odds ratio above 1 per tenfold population supports H1c. The user mix is controlled in model 2 (age is not observed).
import statsmodels.api as sm, statsmodels.formula.api as smf
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
if HAS_POP:
    tick("12.6c H1c")
    dd = acc[acc.TOTAL_MU30DF > 0].copy(); pm_ = expo.groupby("COD_PROVINCIA").POP.mean(); dd["log10_pop"] = np.log10(dd.COD_PROVINCIA.map(pm_)); dd = dd.dropna(subset=["log10_pop"])
    dd["late"] = dd.TOTAL_MU30DF - dd.TOTAL_MU24H; dd["early"] = dd.TOTAL_MU24H
    for lab, cols in {"ped": ["TOT_PEAT_MU30DF"], "cyc": ["TOT_BICI_MU30DF"], "moto": ["TOT_MOTO_MU30DF", "TOT_CICLO_MU30DF"], "truck_bus": ["TOT_CAM_MENOS3500_MU30DF", "TOT_CAM_MAS3500_MU30DF", "TOT_BUS_MU30DF", "TOT_FURG_MU30DF"]}.items():
        dd[f"sh_{lab}"] = dd[[c for c in cols if c in dd]].sum(axis=1) / dd.TOTAL_MU30DF
    dd["interurban"] = (dd.ZONA_AGRUPADA == 1).astype(int); dd["year_"] = dd.ANYO.astype(str); dd["late_share"] = dd.late / dd.TOTAL_MU30DF; out = []
    MIX = "sh_ped + sh_cyc + sh_moto + sh_truck_bus"
    for lab, data, f in [("1 · all roads: area + year", dd, "interurban + C(year_)"), ("2 · + user mix", dd, f"interurban + C(year_) + {MIX}"),
                         ("3 · interurban only, + user mix", dd[dd.interurban == 1], f"C(year_) + {MIX}")]:
        tick(f"  model {lab}")
        mod = smf.glm(f"late_share ~ log10_pop + {f}", data=data, family=sm.families.Binomial(), var_weights=data.TOTAL_MU30DF.values).fit(cov_type="cluster", cov_kwds={"groups": data.COD_PROVINCIA.values})
        b, ci = mod.params["log10_pop"], mod.conf_int().loc["log10_pop"]
        out.append(dict(model=lab, crashes_with_deaths=len(data), OR_per_10x_population=np.exp(b), lo=np.exp(ci[0]), hi=np.exp(ci[1]), p=mod.pvalues["log10_pop"], late_share_pct=100 * data.late.sum() / data.TOTAL_MU30DF.sum()))
    h1 = pd.DataFrame(out); display(h1.round(3)); TABLES["H1c_late_deaths_vs_population"] = h1
    print("Reading: OR > 1 (interval above 1) = the more populated the province, the larger the share of deaths that occur after 24 h, i.e. remote provinces lose relatively more people early, as H1c predicts. "
          "It is an association between provinces: population is a proxy for distance to care, and the age of the victims is not observed.")
else:
    print("12.6c needs population.")

In [ ]:
# ---- 12.6d H4 · What can lighting say about the night-time excess of fatal crashes? ------------------------------------------
# NOTE: the DGT crash files do NOT contain alcohol, drug or speed variables; only the lighting condition. The test that is possible: among casualty crashes, is the odds of a
# fatal outcome higher at night than in daylight, and does it depend on whether the road was lit? Logistic regression adjusted for province, year, area type, road group and crash type,
# cluster-robust by province. A night excess that remains on lit roads points to something other than lack of light (traffic mix, speed, fatigue, alcohol: not observable here).
ILU = {"Luz del día natural, solar": "daylight", "Amanecer o atardecer, sin luz artificial": "dawn/dusk, no lamps", "Amanecer o atardecer, con luz artificial": "dawn/dusk, lamps",
       "Noche, vía iluminada": "night, lit road", "Noche, alumbrado apagado": "night, lamps off", "Noche, sin alumbrado": "night, no lighting"}
import time; tick = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True); tick("12.6d H4 · logistic models with province fixed effects (can take 1-3 min each)")
d4 = acc[acc.lighting.isin(ILU)].copy(); d4["light"] = d4.lighting.map(ILU); d4["year_"] = d4.ANYO.astype(str); out = []
for area, dsub in [("interurban", d4[d4.ZONA_AGRUPADA == 1]), ("urban", d4[d4.ZONA_AGRUPADA == 2])]:
    tick(f"  {area}: {len(dsub):,} crashes")
    f = "fatal30 ~ C(light, Treatment('daylight')) + C(COD_PROVINCIA) + year_ + C(road_group) + C(crash_type)"
    try:
        mod = smf.logit(f, data=dsub).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": dsub.COD_PROVINCIA.values})
        if not mod.mle_retvals.get("converged", True): print(f"  ({area}: the optimiser did not fully converge; check for sparse categories, e.g. merge rare crash types)")
    except Exception as e:
        print(area, "model failed:", type(e).__name__, e); continue
    for nm in sorted(set(ILU.values()) - {"daylight"}):
        key = f"C(light, Treatment('daylight'))[T.{nm}]"
        if key in mod.params: ci = mod.conf_int().loc[key]; out.append(dict(area=area, lighting=nm, n=int((dsub.light == nm).sum()), fatal_pct=100 * dsub[dsub.light == nm].fatal30.mean(), OR_vs_daylight=np.exp(mod.params[key]), lo=np.exp(ci[0]), hi=np.exp(ci[1]), p=mod.pvalues[key]))
h4 = pd.DataFrame(out); display(h4.round(3)); TABLES["H4_lighting_odds"] = h4
print("Reading: OR of a fatal outcome against daylight, adjusted for province, year, road group and crash type. Compare 'night, lit road' with 'night, no lighting' to see how much of the night excess is the absence of light.")
tick("12.6d done · section 12 finished")

## 13 · Export, reproducibility and manuscript plan

Every table collected in `TABLES` is written to a single Excel workbook (one sheet per table, names truncated to Excel's 31-character limit), the cleaned microdata are stored as a pickle, and posterior draws are saved (NetCDF when `h5netcdf` is available, pickle otherwise). A **reproducibility manifest** (`manifest.json`) records the date, Python and package versions, the random seed, the key configuration values and the SHA-256 fingerprint of every input file, so that a reviewer can verify that the reported numbers derive from exactly those inputs.

In [ ]:
import json, hashlib, platform, sys, importlib, importlib.util
xl_path = OUT_DIR / "dgt_tables.xlsx"
with pd.ExcelWriter(xl_path) as w:
    for name, t in TABLES.items():
        (t if isinstance(t, pd.DataFrame) else pd.DataFrame(t)).to_excel(w, sheet_name=name[:31])
acc.drop(columns=["UID"], errors="ignore").to_pickle(OUT_DIR / "acc_clean.pkl")
for k, idt in (RES.items() if "RES" in globals() else []):
    try: idt.to_netcdf(OUT_DIR / f"posterior_{k}.nc")                      # requires h5netcdf or netCDF4
    except Exception: pd.to_pickle(idt, OUT_DIR / f"posterior_{k}.pkl"); print(f"  posterior {k} saved as pickle (for .nc: pip install h5netcdf h5py)")

# ---- Reproducibility manifest: versions, seed, configuration and SHA-256 fingerprints of the inputs ----
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()
inputs = [p for p in sorted(Path(DATA_DIR).glob("*")) if p.is_file()] if Path(DATA_DIR).exists() else []
inputs += [Path(x) for x in (POPULATION_XLSX, POPULATION_CSV, globals().get("PK_PATH"), HOSPITALS_CSV, PROV_GEO) if x and Path(x).exists()]
manifest = dict(date=pd.Timestamp.now().isoformat(timespec="seconds"), python=sys.version.split()[0], platform=platform.platform(), seed=SEED,
                packages={m: getattr(importlib.import_module(m), "__version__", "?") for m in ["pandas", "numpy", "scipy", "sklearn", "statsmodels", "pymc", "arviz", "pyproj", "nutpie", "geopandas"] if importlib.util.find_spec(m)},
                configuration={k: str(globals().get(k)) for k in ["YEARS", "SEGMENT_KM", "GRID_KM", "RMIN_KM", "PK_MAX_DIST_PROV_KM", "PK_URL", "POPULATION_FIXED_YEAR", "BAYES_DRAWS", "BAYES_TUNE", "BAYES_CHAINS", "BAYES_TARGET_ACCEPT", "SAMPLER"]},
                exposure=dict(population=bool(globals().get("HAS_POP")), kilometre_points=bool(globals().get("HAS_PK"))),
                inputs_sha256={p.name: sha(p) for p in inputs})
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"Tables ({len(TABLES)}) → {xl_path} · clean dataset → {OUT_DIR / 'acc_clean.pkl'} · figures → {FIG_DIR} · manifest → {OUT_DIR / 'manifest.json'}")

### 14 · Manuscript plan (designed for an international journal in the field)

**Central question.** *To what extent does the risk of dying in a road crash in Spain vary across territories and years once chance, the composition of crashes and exposure are accounted for — and how much of that variation is spatially structured and persistent?*

**Pre-specified estimands** (fix them *before* inspecting results and register them, e.g. on OSF):
1. **E1 · Change between years.** The rate ratio of 30-day deaths (Section 3) and the adjusted year effect from the risk model R (Section 8), with the posterior probability that risk is lower than in the previous year.
2. **E2 · Territorial heterogeneity.** The standard deviation σ of the province effect and its spatially structured fraction ρ in the severity (L), frequency (F) and risk (R) models.
3. **E3 · Frequency versus severity.** The posterior correlation between the province effects of F and L: are dangerous provinces dangerous because crashes are *numerous* or because they are *severe*?
4. **E4 · Spatial persistence.** The rate ratio of fatal crashes per km of network in hot cells detected in earlier years and evaluated in the following year (Section 7).
5. **E5 · Robustness.** Spearman correlation and maximum difference of province effects under alternative priors and outcome definitions (Section 9).

**Defensible contribution (what the usual literature does not do).** Multi-year comparison with a composition/rate decomposition; **full propagation of uncertainty** (posterior ranges, not rankings of raw rates); **out-of-sample validation on unseen provinces**; geocoding **with explicit quality checks** and spatial-persistence analysis; everything reproducible through a SHA-256 manifest.

**Figure/table map.** Fig. 1 KPIs and rate-ratio forest plot (Sections 2-3) · Fig. 2 funnel plot and EB caterpillar (Section 4) · Fig. 3 risk per inhabitant and decomposition (Section 6) · Fig. 4 Gi\* map and persistence (Section 7) · Fig. 5 BYM2 effects (Section 8) · Table 1 MCMC diagnostics · Table 2 hyper-parameters · Table 3 sensitivity analysis. Supplement: geocoding coverage, posterior-predictive checks, LOO and K-fold results.

**Reporting guidelines.** STROBE (observational studies) and RECORD (routinely collected data); for the Bayesian part, BARG (Bayesian Analysis Reporting Guidelines, Kruschke 2021). Attach the code, `manifest.json` and `dgt_tables.xlsx`.

**Threats to validity to discuss (the code does not hide them).**
- *Ecological fallacy*: everything is at province or cell level.
- *Denominators*: resident population ≠ exposed population (tourism, through traffic), and no distance-based exposure (vehicle-km) is available by province; urban and interurban driving are not separated. State this explicitly and, if possible, add AADT-based exposure for the interurban network.
- *Differential reporting* of minor crashes across provinces (affects F and L in opposite directions).
- *Geocoding*: coverage below 100 %, biased towards state and regional roads; the kilometre-point layer may be years old; positional error is unquantified without ground truth.
- *Definition of a death* (24 h vs. 30 days) and possible changes in registration between years.
- *Only four years*: trends are not estimated, only changes between years; do not extrapolate.

**Candidate journals** (check scope and requirements): *Accident Analysis & Prevention*, *Journal of Safety Research*, *Traffic Injury Prevention*, *Injury Prevention*, *Spatial and Spatio-temporal Epidemiology*, *International Journal of Health Geographics*.

**Methodological references (verify bibliographic details before citing).** Besag, York & Mollié (1991), *Ann. Inst. Stat. Math.* · Riebler et al. (2016), *Stat. Methods Med. Res.* (BYM2) · Simpson et al. (2017), *Stat. Sci.* (PC priors) · Vehtari, Gelman & Gabry (2017), *Stat. Comput.* (PSIS-LOO) · Marshall & Spiegelhalter (2007), *Bayesian Analysis* (mixed predictive checks) · Spiegelhalter (2005), *Stat. Med.* (funnel plots) · Kitagawa (1955), *JASA* · Getis & Ord (1992), *Geogr. Anal.* · Hauer (1997), *Observational Before-After Studies in Road Safety* · Lord & Mannering (2010), *Transp. Res. Part A* (review of crash-frequency models).

### Next steps (from least to most effort)
1. **Exposure data**: INE population for 2022-2026 (the annual figure for 1 January 2026 allows a proper person-year denominator for 2025).
2. **A current kilometre-point layer** (IGN/CNIG) and the regional road networks that are missing (the Section 7 table lists the roads with fatal crashes that could not be placed). Check the licence.
3. **Province polygons** (`PROV_GEO`): enable choropleth maps of the BYM2 effects and the point-in-polygon check.
4. **AADT per segment** (Ministry of Transport, state network) as cell-level exposure instead of road length; network isochrones to hospitals.
5. **Effect of interventions** (speed limits, etc.): difference-in-differences or synthetic control; this needs pre-2022 years.
6. **The full DGT register** (victim age, sex, vehicle) through a formal request: severity models by road user and age standardisation.